# Automated *In Situ* GIWAXS Analysis, Broadening Deconvolution & CIF Phase Comparison

This notebook automatically discovers your calibration and sample scans, lets you compare against **any number of `.cif` reference structures (1, 3, 4, 5, or more)**, removes thin-film $\text{LaB}_6$ instrumental broadening, isolates the perovskite $\alpha$-$(100)$ peak from any overlapping $\delta$-phase / hexagonal polytype shoulder (with an optional side-by-side un-isolated comparison cell), detects the beam-damage cutoff, and compares crystallization kinetics and orientation across processing conditions.

### Quick Start
1. **Check `DATA_DIR` and `BEAM_ENERGY_KEV` in Cell 1** (and optionally choose your `.cif` files via auto-discovery, explicit list, or the pop-up file picker).
2. **Run cells top-to-bottom (`Shift + Enter`).** Every step explains what it is doing and displays a verification plot before saving outputs to `./outputs/`.


## Cell 1: Set Data Folder, Beam Energy & `.cif` Reference Options (Only Required Input)
**What this cell does:** Sets your data folder (`DATA_DIR`), X-ray beam energy (`BEAM_ENERGY_KEV = 12.7` keV), and `.cif` / peak-fitting options:
* **Workflow Order:**
  1. **Cell 4** fits **all visible $\text{LaB}_6$ standard peaks** across the full integrated $I(q)$ range (no peak isolation on the standard) to measure the thin-film instrumental broadening $(H_{G,\text{inst}}, H_{L,\text{inst}})$.
  2. **Cell 6** fits your sample's perovskite $(100)$ peak **corrected for $\text{LaB}_6$ instrumental broadening** (`ENABLE_SECONDARY_PEAK = False` by default).
  3. **Cell 6B** (optional) lets you **isolate the $\alpha$-$(100)$ peak** from any overlapping low-$q$ $\delta$-phase shoulder on the instrumentally corrected data and compare both side by side.


In [ ]:
from pathlib import Path
import fabio
import h5py
import numpy as np
from scipy.ndimage import median_filter, binary_dilation

# ==============================================================================
# 1. SET YOUR DATA FOLDER PATH & X-RAY BEAM ENERGY HERE
# ==============================================================================
DATA_DIR = Path(r"H:\17.2 beam stuff\2026_05_Stone")
if not DATA_DIR.exists():
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_DIR}. Check that drive H: is connected.")
OUTPUT_DIR = DATA_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PONI_FILE_OVERRIDE = None           # None = auto-find valid .poni (ignores macOS ._ sidecars);
                                    # or set explicitly, e.g. Path("LaB6_calibration2_scan001.poni")
BEAM_ENERGY_KEV = None              # None = read exact Wavelength from .poni (e.g., 9.7625e-11 m = 12.7 keV);
                                    # or set float in keV (e.g., 12.7) to override
PIXEL_SIZE_M = None                 # None = auto-read from .poni Detector_config / Eiger H5 (75e-6 m)

# Base instrument mask (.edf) + auto-generated saturation mask (.npy)
BASE_EDF_MASK = Path(r"H:\17.2 beam stuff\2026_05_Stone\mask.edf")
SAMPLE_H5_FOR_MASK = None           # Optional: set Path(...) to one sample .h5, or leave None to auto-pick
SATURATION_MASK_FILE = OUTPUT_DIR / "saturation_mask.npy"
MASK_FILE = OUTPUT_DIR / "mask_edf_plus_saturation.npy"
DEAD_MASK_FILE = OUTPUT_DIR / "dead_pixel_mask.npy"

# Extra masking options
EXTRA_MASK_FILE = None              # Optional: your own mask (.npy / .edf / .tif), non-zero = masked. It is added to the combined mask.
DEAD_PIXEL_SCAN = True              # True: find dead / hot / cold pixels that misbehave the same way in MANY scans (incl. the LaB6 scan)
DEAD_SCAN_FILES = 14                # how many scans to inspect (LaB6 scans are always included)
DEAD_SCAN_FRAMES = 20               # evenly spaced frames averaged per scan
DEAD_MIN_FRACTION = 0.6             # a pixel must be an outlier in at least this fraction of the scans to be masked
DEAD_HOT_REL = 0.5                  # hot  = more than 50 % above the average of its neighbours (and > 5 sigma)
DEAD_COLD_REL = 0.5                 # cold = more than 50 % below the average of its neighbours (and > 5 sigma)
RAW_IMAGE_SCALE = "log"             # Cell 4 raw detector image: "log" or "linear" (colour scale starts at 0, nothing hidden below the top 0.1 %)
MANUAL_BAD_PIXELS = []              # extra pixels to mask by hand as (row, col) = (detector Y, detector X), e.g. [(2030, 1040), (770, 1505)]
MANUAL_BAD_PIXEL_RADIUS = 1         # also mask this many pixels around each manual pixel (0 = just that pixel)
MASK_HINTS = [(900, 2000), (900, 400), (1900, 400), (1850, 750), (1550, 875), (1800, 1100), (1350, 1200)]
                                    # APPROXIMATE (x, y) positions of bad pixels / short bad lines, read off the detector plot.
                                    # The exact pixels are found automatically: every pixel near a hint that stands out from its
                                    # surroundings in the LaB6 image is masked. Leave [] if you do not need it.
MASK_HINT_RADIUS = 12               # search radius around each hint (pixels). Larger = more forgiving of imprecise positions
DEAD_HINT_FRAMES = 60               # frames averaged for the LaB6 image used to snap the hints
LAB6_MAX_RMS_PCT = None            # None = use every fitted LaB6 peak for the broadening extrapolation; e.g. 8 = drop peaks whose fit residual exceeds 8 % of peak height
REBUILD_MASK = False                # True: rebuild the combined mask now (it is also rebuilt automatically if a source mask file is newer)


def _scan_average(h5p, n_frames, det_shape):
    """Average of n evenly spaced frames of one scan (bad readout values ignored). None if unreadable."""
    import warnings
    try:
        with h5py.File(h5p, "r") as f:
            ds = f["/entry/data/data"] if "/entry/data/data" in f else None
            if ds is None or ds.ndim != 3 or tuple(ds.shape[1:]) != tuple(det_shape):
                return None
            idx = np.unique(np.linspace(0, ds.shape[0] - 1, min(ds.shape[0], n_frames)).astype(int))
            acc = np.zeros(det_shape, np.float64)
            cnt = np.zeros(det_shape, np.float64)
            for i in idx:
                fr = np.asarray(ds[int(i)], dtype=np.float64)
                ok = np.isfinite(fr) & (fr >= 0) & (fr < 65530)
                acc += np.where(ok, fr, 0.0)
                cnt += ok
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return np.where(cnt > 0, acc / np.maximum(cnt, 1), np.nan), len(idx)
    except Exception:
        return None


def _outlier_maps(avg, n_avg, exclude, hot_rel=0.5, cold_rel=0.5):
    """Per-scan outliers of the averaged image against the median of the surrounding pixels."""
    A = np.where(np.isfinite(avg), avg, 0.0)
    M = median_filter(A, size=5)
    sigma = np.sqrt(np.maximum(M, 0.5) / max(n_avg, 1)) + 1e-6          # shot noise of an average of n frames
    near_excl = binary_dilation(exclude, iterations=2)
    hot = ((A - M) > hot_rel * M) & ((A - M) > 5.0 * sigma) & (M > 0.5)
    cold = ((M - A) > cold_rel * M) & ((M - A) > 5.0 * sigma) & (M > 2.0)
    dead0 = (A == 0) & (M > 2.0)
    return (hot | cold | dead0) & ~near_excl, hot & ~near_excl, (cold | dead0) & ~near_excl


def find_persistent_bad_pixels(h5_files, det_shape, exclude, n_frames=20, min_fraction=0.6, hot_rel=0.5, cold_rel=0.5):
    """A real defect sits on the SAME pixel in every scan, a diffraction spot does not.
    -> (mask, info) where mask = pixels that are outliers in >= min_fraction of the usable scans."""
    votes = np.zeros(det_shape, np.int32)
    hot_votes = np.zeros(det_shape, np.int32)
    cold_votes = np.zeros(det_shape, np.int32)
    n_ok = 0
    for h5p in h5_files:
        r = _scan_average(h5p, n_frames, det_shape)
        if r is None:
            continue
        avg, n_avg = r
        o, h, c = _outlier_maps(avg, n_avg, exclude, hot_rel, cold_rel)
        votes += o; hot_votes += h; cold_votes += c
        n_ok += 1
    if n_ok < 3:      # with fewer than 3 scans a diffraction spot cannot be told apart from a defect
        return np.zeros(det_shape, bool), {"scans": n_ok, "hot": 0, "cold": 0, "too_few": True}
    need = max(2, int(np.ceil(min_fraction * n_ok)))
    mask = votes >= need
    return mask, {"scans": n_ok, "need": need, "hot": int((mask & (hot_votes >= cold_votes)).sum()),
                  "cold": int((mask & (cold_votes > hot_votes)).sum())}


def snap_hints_to_pixels(avg, hints, radius, exclude, n_avg=1, hot_rel=0.25, cold_rel=0.4, n_sigma=4.0):
    """For each approximate (x, y) hint, mask the pixels within `radius` that stand out from the median of their
    surroundings (7x7, so a 1-pixel-wide line is still an outlier). -> (mask, report list)."""
    A = np.where(np.isfinite(avg), avg, 0.0)
    M = median_filter(A, size=7)
    sigma = np.sqrt(np.maximum(M, 0.5) / max(n_avg, 1)) + 1e-6
    cand = ((((A - M) > hot_rel * M) & ((A - M) > n_sigma * sigma) & (M > 0.5)) |
            (((M - A) > cold_rel * M) & ((M - A) > n_sigma * sigma) & (M > 2.0)) |
            ((A == 0) & (M > 2.0))) & ~exclude
    out = np.zeros(A.shape, bool)
    report = []
    ny, nx = A.shape
    for (x, y) in hints:
        r0, r1 = max(int(y) - radius, 0), min(int(y) + radius + 1, ny)
        c0, c1 = max(int(x) - radius, 0), min(int(x) + radius + 1, nx)
        win = cand[r0:r1, c0:c1]
        ys, xs = np.nonzero(win)
        pix = [(int(r0 + a), int(c0 + b)) for a, b in zip(ys, xs)]
        out[r0:r1, c0:c1] |= win
        report.append(((x, y), pix))
    return out, report


def _load_extra_mask(p, shape):
    p = Path(p)
    raw = np.load(p) if p.suffix.lower() == ".npy" else fabio.open(str(p)).data
    m = np.asarray(raw) != 0
    assert m.shape == tuple(shape), f"EXTRA_MASK_FILE shape {m.shape} != detector shape {tuple(shape)}"
    return m


_src_masks = [p for p in (BASE_EDF_MASK, EXTRA_MASK_FILE) if p is not None and Path(p).exists()]
_mask_stale = MASK_FILE.exists() and any(Path(p).stat().st_mtime > MASK_FILE.stat().st_mtime for p in _src_masks)
if _mask_stale:
    print("A source mask file is newer than the combined mask -> rebuilding it.")

if REBUILD_MASK or _mask_stale or not MASK_FILE.exists():
    base_mask = fabio.open(str(BASE_EDF_MASK)).data != 0
    det_shape = base_mask.shape

    # Find one sample .h5 quickly (top-level first, then 1 subfolder deep — no slow deep rglob)
    if SAMPLE_H5_FOR_MASK is not None:
        h5_list = [Path(SAMPLE_H5_FOR_MASK)]
    else:
        h5_list = [p for p in sorted(DATA_DIR.glob("*.h5")) if not p.name.startswith("._")]
        if not h5_list:
            h5_list = [p for p in sorted(DATA_DIR.glob("*/*.h5")) if not p.name.startswith("._")]

    frame = None
    used_h5 = None
    for h5p in h5_list:
        if "lab6" in h5p.name.lower() and len(h5_list) > 1:
            continue
        try:
            with h5py.File(h5p, "r") as f:
                dset = f["/entry/data/data"] if "/entry/data/data" in f else None
                if dset is None or dset.ndim != 3 or dset.shape[1:] != det_shape:
                    continue
                # Read middle frame (1 frame only -> <0.5s over H:\)
                frame = np.asarray(dset[dset.shape[0] // 2], dtype=np.float32)
                used_h5 = h5p.name
                used_h5_path = h5p
                break
        except Exception:
            continue

    assert frame is not None, f"Could not read a (*, {det_shape[0]}, {det_shape[1]}) frame from {DATA_DIR}"

    # Build standalone saturation / dead-pixel mask
    sat_raw = ~np.isfinite(frame) | (frame < 0) | (frame >= 65530)
    clean_frame = np.where(base_mask | sat_raw, 0.0, frame)
    local_med = median_filter(clean_frame, size=5)
    hot_raw = (clean_frame > 10.0 * np.maximum(local_med, 1.0)) & (clean_frame > 100.0)
    sat_mask = binary_dilation(sat_raw | hot_raw, iterations=1)

    # Dead / hot / cold pixels: outliers on the SAME pixel in many scans (LaB6 included) are detector defects
    dead_mask = np.zeros(det_shape, dtype=bool)
    if DEAD_PIXEL_SCAN:
        _all_h5 = [p for p in sorted(DATA_DIR.glob("*.h5")) + sorted(DATA_DIR.glob("*/*.h5")) if not p.name.startswith("._")]
        _lab6 = [p for p in _all_h5 if "lab6" in p.name.lower()]
        _rest = [p for p in _all_h5 if p not in _lab6]
        _keep = max(DEAD_SCAN_FILES - len(_lab6), 1)
        if len(_rest) > _keep:
            _rest = [_rest[int(i)] for i in np.linspace(0, len(_rest) - 1, _keep).astype(int)]
        _files = _lab6 + _rest
        print(f"Dead-pixel scan: inspecting {len(_files)} scan(s) ({len(_lab6)} LaB6) x {DEAD_SCAN_FRAMES} frames ...")
        dead_mask, _dead_info = find_persistent_bad_pixels(
            _files, det_shape, exclude=base_mask | sat_mask, n_frames=DEAD_SCAN_FRAMES,
            min_fraction=DEAD_MIN_FRACTION, hot_rel=DEAD_HOT_REL, cold_rel=DEAD_COLD_REL)
        np.save(DEAD_MASK_FILE, dead_mask.astype(np.uint8))
        if _dead_info.get("too_few"):
            print(f"  only {_dead_info['scans']} usable scan(s): need >= 3 to separate defects from diffraction spots -> nothing auto-masked. "
                  "List the bad pixels in MANUAL_BAD_PIXELS instead.")
        print(f"  usable scans: {_dead_info['scans']} | pixel must be an outlier in >= {_dead_info.get('need', '-')} of them")
        print(f"  +{int(dead_mask.sum()):,} new masked pixels (hot {_dead_info['hot']:,}, cold/dead {_dead_info['cold']:,}) -> {DEAD_MASK_FILE.name}")
        _ys, _xs = np.nonzero(dead_mask)
        if len(_ys):
            import pandas as _pd
            _pd.DataFrame({"row_y": _ys, "col_x": _xs}).to_csv(OUTPUT_DIR / "dead_pixels_found.csv", index=False)
            print("  first pixels (row_y, col_x):", [(int(y), int(x)) for y, x in list(zip(_ys, _xs))[:12]], "... (full list: dead_pixels_found.csv)")
    for _r, _c in MANUAL_BAD_PIXELS:
        dead_mask[max(_r - MANUAL_BAD_PIXEL_RADIUS, 0): _r + MANUAL_BAD_PIXEL_RADIUS + 1,
                  max(_c - MANUAL_BAD_PIXEL_RADIUS, 0): _c + MANUAL_BAD_PIXEL_RADIUS + 1] = True
    if MANUAL_BAD_PIXELS:
        print(f"  + {len(MANUAL_BAD_PIXELS)} manual bad pixel(s) (radius {MANUAL_BAD_PIXEL_RADIUS})")
        np.save(DEAD_MASK_FILE, dead_mask.astype(np.uint8))
    if MASK_HINTS:
        _lab6_for_hints = [p for p in sorted(DATA_DIR.glob("*.h5")) + sorted(DATA_DIR.glob("*/*.h5"))
                           if "lab6" in p.name.lower() and not p.name.startswith("._")]
        _src = (_lab6_for_hints or h5_list)[0]
        _r = _scan_average(_src, DEAD_HINT_FRAMES, det_shape)
        if _r is None:
            print(f"  ! could not read {_src.name} for MASK_HINTS (skipped)")
        else:
            _avg_h, _n_h = _r
            _hint_mask, _rep = snap_hints_to_pixels(_avg_h, MASK_HINTS, MASK_HINT_RADIUS, exclude=base_mask | sat_mask, n_avg=_n_h)
            dead_mask |= _hint_mask
            np.save(DEAD_MASK_FILE, dead_mask.astype(np.uint8))
            print(f"MASK_HINTS (snapped on {_src.name}): +{int(_hint_mask.sum()):,} pixels")
            for (hx, hy), pix in _rep:
                print(f"  hint (x={hx}, y={hy}): {len(pix)} pixel(s)" + (f"  e.g. (row,col) {pix[:6]}" if pix else "  -> nothing stands out here; raise MASK_HINT_RADIUS or list the pixel in MANUAL_BAD_PIXELS"))
            try:
                import matplotlib.pyplot as _plt
                _n = len(MASK_HINTS); _cols = min(4, _n); _rows = int(np.ceil(_n / _cols))
                _fig, _axs = _plt.subplots(_rows, _cols, figsize=(3.4 * _cols, 3.4 * _rows), squeeze=False)
                _lg = np.log10(np.maximum(_avg_h, 1.0))
                for _k, ((hx, hy), pix) in enumerate(_rep):
                    _ax = _axs[_k // _cols][_k % _cols]
                    _r0, _r1 = max(int(hy) - 35, 0), min(int(hy) + 36, det_shape[0]); _c0, _c1 = max(int(hx) - 35, 0), min(int(hx) + 36, det_shape[1])
                    _ax.imshow(_lg[_r0:_r1, _c0:_c1], origin="lower", cmap="inferno", extent=(_c0, _c1, _r0, _r1))
                    if pix:
                        _ax.scatter([c for _, c in pix], [r for r, _ in pix], s=70, facecolors="none", edgecolors="cyan", linewidths=0.9)
                    _ax.set_title(f"hint ({hx}, {hy}): {len(pix)} px", fontsize=9)
                    _ax.tick_params(labelsize=7)
                for _k in range(_n, _rows * _cols):
                    _axs[_k // _cols][_k % _cols].axis("off")
                _fig.suptitle("Pixels masked near each hint (cyan circles) — check they are single bad pixels, not part of a ring", fontsize=10)
                _fig.tight_layout()
                _fig.savefig(OUTPUT_DIR / "mask_hint_check.png", dpi=150)
                _plt.show()
            except Exception as _e:
                print("  (hint check figure skipped:", _e, ")")
    extra_mask = _load_extra_mask(EXTRA_MASK_FILE, det_shape) if EXTRA_MASK_FILE is not None else np.zeros(det_shape, bool)
    if EXTRA_MASK_FILE is not None:
        print(f"EXTRA_MASK_FILE ({Path(EXTRA_MASK_FILE).name}): {int((extra_mask & ~base_mask).sum()):,} additional masked pixels")

    np.save(SATURATION_MASK_FILE, sat_mask.astype(np.uint8))
    np.save(MASK_FILE, (base_mask | sat_mask | dead_mask | extra_mask).astype(np.uint8))

    print(f"Probed 1 frame from    : {used_h5}")
    print(f"mask.edf masked pixels : {base_mask.sum():,}")
    print(f"New saturation pixels  : {(sat_mask & ~base_mask).sum():,} (saved -> {SATURATION_MASK_FILE.name})")
    print(f"Active MASK_FILE      -> {MASK_FILE}")
else:
    print(f"Using existing combined mask -> {MASK_FILE}")

# ==============================================================================
# 2. SELECT REFERENCE .CIF FILES (HANDLES 0, 1, 2, 3, 4, 5+ FILES)
# ==============================================================================
OPEN_CIF_PICKER = False         # True = open OS file dialog to multi-select 1-5+ .cif files
CIF_FILES = [r"H:\17.2 beam stuff\CF3NH3PbI3.cif", r"H:\17.2 beam stuff\FAPbI3.cif"]                  # Leave empty [] to auto-find all .cif files in DATA_DIR,
                                # OR list specific paths, e.g. [Path("FAPbI3.cif"), Path("PbI2.cif")]

# ==============================================================================
# 3. SAMPLE PEAK WINDOWS & OPTIONAL DELTA-PHASE ISOLATION SETTINGS
# ==============================================================================
Q_TARGET_100 = 0.990                # Sample alpha-(100) q (A^-1) where the LaB6 instrumental width is evaluated (S-series: a ~ 6.35 A -> q ~ 0.990)
Q_100_SEARCH_WINDOW = (0.96, 1.05)  # Search/fit bounds for sample perovskite alpha-(100) peak (A^-1)
ENABLE_SECONDARY_PEAK = False       # False (default) = fit instrumentally corrected (100) peak first;
                                    # True = also isolate alpha-(100) from overlapping delta-phase peak
                                    # (Cell 6B lets you run and compare both side by side!)
Q_DELTA_SEARCH_WINDOW = (0.82, 0.96)# Search/fit bounds for overlapping delta-phase peak in Cell 6B (A^-1)
Q_100_CORE_WINDOW = (0.975, 1.018)  # tight core of the alpha-(100) peak used to pick t_max (A^-1)
Q_PBI2_WINDOW = (0.875, 0.930)      # PbI2 (001) area window (A^-1)
Q_PLOT_RANGE = (0.40, 2.20)         # q-range (A^-1) for heatmaps and CIF stick comparisons

# ==============================================================================
# 4. INTEGRATION & CACHING OPTIONS
# ==============================================================================
NPT_RADIAL = 3000                   # Number of 1D radial q-bins (~0.0012 A^-1/bin, matched to 75 um Eiger pixel)
AZIMUTH_RANGE = None                # None = full cake; or e.g. (-115, -65) for out-of-plane wedge
DT_SEC_FALLBACK = 0.5               # Used only if frame timestamps aren't stored inside the H5
FORCE_RECOMPUTE = False             # True = re-integrate H5 files instead of using cached .npz

# ==============================================================================
# 5. SAMPLE METADATA: WHAT EACH SAMPLE ACTUALLY CHANGES  (edit here; used by Cells 6, 7, 7B)
#    Samples are grouped by the variable that changes, NOT by the letter prefix, so every plot
#    compares like with like. Replicates (A/B) share a level and are drawn in the same colour
#    (B dashed). A sample missing from SAMPLE_META falls back to its letter prefix.
# ==============================================================================
import re as _re
import numpy as _np

# group id -> title, short name, numeric?, axis label for "outcome vs. variable" plots,
#             check = which measured process quantity should equal the nominal level (setpoint check)
ONLY_TABLE_SAMPLES = True   # True: a sample must be on the sample table AND have in situ data to be processed

EXPERIMENT_GROUPS = {
    "Y_comp":        dict(title="Y · Composition (Cs content)",     short="Y comp.",    numeric=True,  xlabel="Cs content (%)",              check=None),
    "P_surfactant":  dict(title="P · Surfactant",                   short="P surf.",    numeric=False, xlabel="Surfactant",                  check=None),
    "F_silane":      dict(title="F · Functional group (silane)",    short="F silane",   numeric=False, xlabel="Functional group",            check=None),
    "S_speed":       dict(title="S · Coating speed",                short="S speed",    numeric=True,  xlabel="Coating speed (m/min)",       check="vel"),
    "S_gap":         dict(title="S · Confinement gap",              short="S gap",      numeric=True,  xlabel="Confinement gap (µm)",        check=None),
    "S_substrate":   dict(title="S · Substrate annealing temp.",    short="S substr. T", numeric=True, xlabel="Substrate annealing temp. (°C)", check="T_plateau"),
    "S_superstrate": dict(title="S · Superstrate temp.",            short="S superstr. T", numeric=True, xlabel="Superstrate temp. (°C)",     check=None),
    "S_time":        dict(title="S · Anneal time",                  short="S anneal t", numeric=True,  xlabel="Anneal time (min)",           check=None),
}
GROUP_ORDER = list(EXPERIMENT_GROUPS)

# base sample id -> (group, numeric level or None, label shown on plots)
SAMPLE_META = {}
def _add_meta(group, ids, level, label):
    for sid in ids:
        SAMPLE_META[sid.lower()] = (group, level, label)

# Y: composition (Cs fraction; FA/MA fractions shift with it)
_add_meta("Y_comp", ["Y1A", "Y1B"], 0.0,  "Cs 0 %")
_add_meta("Y_comp", ["Y2", "Y2A"],  5.0,  "Cs 5 %")
_add_meta("Y_comp", ["Y3A"],        15.0, "Cs 15 %")
# P: surfactant (categorical; level = order of appearance)
for _i, (_sid, _lab) in enumerate([("P1A", "Nha8Nhis"), ("P2A", "Nbrpe6Nae3Nhis"), ("P3A", "Nbrpe6Nce6"), ("P4A", "Nbrpe6Nae6")]):
    _add_meta("P_surfactant", [_sid], float(_i), _lab)
# F: silane functional group (categorical; halide series first)
for _i, (_ids, _lab) in enumerate([(["F1A"], "3-iodopropyl"), (["F2A"], "3-bromopropyl"), (["F3A"], "3-chloropropyl"),
                                   (["F4A", "F4B"], "3-aminopropyl"), (["F5A", "F5B"], "3-cyanopropyl"),
                                   (["F6A", "F6B"], "3-mercaptopropyl")]):
    _add_meta("F_silane", _ids, float(_i), _lab)
# S: one experiment per changing variable
_add_meta("S_speed", ["S1A", "S1B"], 6.0, "6 m/min"); _add_meta("S_speed", ["S2A", "S2B"], 4.0, "4 m/min"); _add_meta("S_speed", ["S3A", "S3B"], 2.0, "2 m/min")
_add_meta("S_gap", ["S4A", "S4B"], 25.4, "25.4 µm"); _add_meta("S_gap", ["S6A"], 50.0, "50 µm"); _add_meta("S_gap", ["S7A"], 100.0, "100 µm")
_add_meta("S_substrate", ["S8A", "S8B"], 150.0, "150 °C"); _add_meta("S_substrate", ["S9A", "S9B"], 125.0, "125 °C"); _add_meta("S_substrate", ["S10A", "S10B"], 100.0, "100 °C")
_add_meta("S_superstrate", ["S11A", "S11B"], 25.0, "RT"); _add_meta("S_superstrate", ["S12A", "S12B"], 75.0, "75 °C"); _add_meta("S_superstrate", ["S14A", "S14B"], 50.0, "50 °C")
_add_meta("S_time", ["S16A", "S16B"], 5.0, "5 min"); _add_meta("S_time", ["S17A", "S17B"], 10.0, "10 min"); _add_meta("S_time", ["S18A", "S18B"], 15.0, "15 min")
# S15A / S15B = glass background (not a perovskite): excluded from the perovskite figures via EXACT_EXCLUDE in Cells 6/7.


def sample_info(name):
    """-> dict(group, level, label, rep, numeric, known) for a sample name such as 'S2B' or 'S1A-try2'."""
    raw = str(name).replace("_reduced", "").strip()
    m = _re.match(r"^([A-Za-z]+\d+)([A-Za-z]?)", raw)
    base = (m.group(1) + m.group(2)).lower() if m else raw.lower()
    for key in (raw.lower(), base, (m.group(1).lower() if m else base)):
        if key in SAMPLE_META:
            g, lvl, lab = SAMPLE_META[key]
            return dict(group=g, level=lvl, label=lab, rep=(m.group(2).upper() or "A") if m else "A",
                        numeric=EXPERIMENT_GROUPS[g]["numeric"], known=True)
    letter = (_re.match(r"^[A-Za-z]", raw).group(0).upper() if _re.match(r"^[A-Za-z]", raw) else "Other")
    return dict(group=letter, level=_np.nan, label=raw, rep="A", numeric=False, known=False)



# --- General (overview) groups: one per letter series, shown BEFORE the per-experiment breakdown ---------
FAMILY_ORDER = ["Y", "P", "S", "F"]
FAMILY_TITLES = {"Y": "Y series · overview (all samples)", "P": "P series · overview (all samples)",
                 "S": "S series · overview (all samples)", "F": "F series · overview (all samples)"}
SKIP_IDENTICAL_OVERVIEWS = True     # True: skip an overview whose samples are exactly one experiment (Y, P, F are
                                    # single experiments, so only S gets a separate overview). False = always draw all four.


def build_overview_groups(recs, key="cond"):
    """recs: record dicts with 'name' and key ('cond' or 'series' = experiment id).  -> {family: [copies]}.
    Within an overview, COLOUR = experiment (so you can see which samples belong together), lighter shade =
    higher rank of the changed variable, replicate B dashed.  Labels stay 'level · sample'.  Originals untouched."""
    import matplotlib.cm as _cm
    out = {}
    for fam in FAMILY_ORDER:
        items = [r for r in recs if str(r["name"])[:1].upper() == fam and not r.get("baseline_copy")]
        if len(items) < 2:
            continue
        if SKIP_IDENTICAL_OVERVIEWS and len({r[key] for r in items}) <= 1:
            continue
        gi = {g: i for i, g in enumerate(GROUP_ORDER)}
        nat = lambda n: [int(t) if t.isdigit() else t.lower() for t in _re.split(r"(\d+)", str(n))]
        items.sort(key=lambda r: (gi.get(r[key], 99), r["level"] if _np.isfinite(r.get("level", _np.nan)) else 1e9, nat(r["name"])))
        exps = []
        for r in items:
            if r[key] not in exps:
                exps.append(r[key])
        copies = []
        for r in items:
            e = exps.index(r[key])
            lv = sorted({x["level"] for x in items if x[key] == r[key]})
            rank = lv.index(r["level"]) if r["level"] in lv else 0
            f = 0.0 if len(lv) < 2 else 0.55 * rank / (len(lv) - 1)          # blend toward white
            base = _np.array(_cm.tab10(e % 10)[:3])
            col = tuple(base * (1 - f) + _np.ones(3) * f) + (1.0,)
            copies.append(dict(r, cond=fam, series=fam, color=col))
        out[fam] = copies
    return out


# --- Shared baseline (reference) condition -------------------------------------------------------------------
# The reference film (6 m/min, 25.4 um gap, 150 C anneal, 50 C superstrate) is the S1 sample.  It is the baseline
# level of the gap, substrate-T and superstrate-T experiments too, so S1 is ALSO shown in those groups (as a copy
# flagged baseline_copy=True).  A copy is only added when the group has no real sample at that level, so nothing is
# invented: a level with no data (e.g. S4, S8) simply does not appear.
BASELINE_SPEED = 6.0
BASELINE_LEVELS = {"S_gap": (25.4, "25.4 µm"), "S_substrate": (150.0, "150 °C"), "S_superstrate": (50.0, "50 °C")}


def add_baseline_copies(recs, key="cond"):
    """recs: record dicts with 'name', 'level', and key ('cond' for Cells 7/7B/7C, 'series' for Cell 6).
    -> recs + baseline copies (originals untouched; copies carry baseline_copy=True)."""
    base = [r for r in recs if sample_info(r["name"])["group"] == "S_speed"
            and _np.isfinite(sample_info(r["name"])["level"]) and abs(sample_info(r["name"])["level"] - BASELINE_SPEED) < 1e-6]
    out = list(recs)
    for g, (lvl, lab) in BASELINE_LEVELS.items():
        have = any(r.get(key) == g and _np.isfinite(r.get("level", _np.nan)) and abs(r["level"] - lvl) < 1e-6 for r in recs)
        if have:
            continue
        for r in base:
            c = dict(r)
            c[key] = g
            if key == "cond":
                c["series"] = g
            c.update(level=lvl, level_label=lab, numeric=True, known=True, baseline_copy=True,
                     label=f"{lab} · {r['name']}", label2=f"{lab}\n{r['name']}")
            out.append(c)
    return out


def assign_group_styles(recs):
    """Sort one experiment's records by level and give each a colour (by level) and linestyle (B replicate dashed).
    Records need: name, known, level, rep, numeric.  Adds: color, ls."""
    import matplotlib.cm as _cm
    recs.sort(key=lambda r: (r["level"] if _np.isfinite(r["level"]) else 1e9,
                             [int(t) if t.isdigit() else t.lower() for t in _re.split(r"(\d+)", str(r["name"]))]))
    levels = []
    for r in recs:
        if r["known"] and r["level"] not in levels:
            levels.append(r["level"])
    n = len(levels)
    for i, r in enumerate(recs):
        if r["known"] and n >= 1:
            k = levels.index(r["level"])
            if r["numeric"] and n > 1:
                r["color"] = _cm.viridis(0.08 + 0.80 * k / (n - 1))
            else:
                r["color"] = _cm.tab10(k % 10)
        else:
            r["color"] = _cm.tab10(i % 10)
        r["ls"] = "--" if r["rep"] == "B" else "-"
    return recs

print(f"Data directory : {DATA_DIR.resolve()}")
print(f"Output folder  : {OUTPUT_DIR.resolve()}")
if BEAM_ENERGY_KEV is not None:
    print(f"Beam energy    : {BEAM_ENERGY_KEV:.4f} keV (lambda = {12.39841984 / BEAM_ENERGY_KEV:.5f} A)")

## Cell 2: Load Libraries, Broadening Deconvolution Engine & CIF Structure Calculator
**What this cell does:**
* Loads `hdf5plugin` (for Eiger `.h5` decompression), `h5py`, and `pyFAI`, including an automatic geometry resolver (`load_azimuthal_integrator`) that populates missing `.poni` pixel dimensions and X-ray wavelength (from `BEAM_ENERGY_KEV`, `.poni` text, or Eiger HDF5 metadata).
* Sets up **1-peak and 2-peak Thompson-Cox-Hastings (TCH) pseudo-Voigt** convolution models:
  * Because your NIST $\text{LaB}_6$ calibration standard is a **thin film measured at the exact same incidence angle ($\alpha_i$) and geometry** as your samples, geometric beam-footprint broadening and detector resolution are already captured directly in $(H_{G,\text{inst}}, H_{L,\text{inst}})$.
  * When `ENABLE_SECONDARY_PEAK = True`, the adaptive fitter tests whether adding a second deconvolved TCH peak in `Q_DELTA_SEARCH_WINDOW` significantly improves the Bayesian Information Criterion ($\Delta\text{BIC} > 8$), isolating the $\alpha\text{-MA}_{0.6}\text{FA}_{0.4}\text{PbI}_3$ $(100)$ peak from any overlapping $\delta$-phase or hexagonal polytype shoulder without overfitting pure single-phase frames.
* Loads a built-in **Crystallographic Information File (`.cif`) parser & structure-factor calculator** to compute exact theoretical Bragg positions ($q_{hkl}$) and relative intensities.


In [ ]:
import json
import os
import re

# Cap OpenMP/BLAS threads to ~50% of CPU cores BEFORE importing pyFAI/numpy
# so 1D integration stays fast without maxing out your CPU or freezing Windows.
_N_CPU = os.cpu_count() or 4
_TARGET_THREADS = str(max(2, _N_CPU // 2))
for _env_var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_env_var, _TARGET_THREADS)

import hdf5plugin  # Must be imported before h5py for Eiger LZ4/BSLZ4 decompression
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyFAI
from scipy.ndimage import uniform_filter1d
from scipy.optimize import curve_fit

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10
plt.rcParams["legend.frameon"] = False      # no boxes around legends anywhere in the notebook

HC_KEV_M = 1.2398419843320026e-09  # h*c in keV * meters


# ==============================================================================
# A. EIGER HDF5, DETECTOR GEOMETRY RESOLVER & MASKING HELPERS
# ==============================================================================
def find_eiger_data_path(h5_file):
    """Locate the primary 3D image stack inside an Eiger *_master.h5 file."""
    for path in ["/entry/data/data", "/entry/data/data_000001"]:
        if path in h5_file:
            return path
    if "/entry/data" in h5_file:
        for key in h5_file["/entry/data"].keys():
            obj = h5_file[f"/entry/data/{key}"]
            if isinstance(obj, h5py.Dataset) and obj.ndim == 3:
                return f"/entry/data/{key}"
    raise KeyError("Could not locate 3D detector dataset inside /entry/data/.")


def get_eiger_timestamps(h5_file, n_frames, dt_fallback=0.5):
    """Extract relative timestamps (s) from Eiger metadata, falling back to dt_fallback."""
    for path in [
        "/entry/instrument/detector/frame_time",
        "/entry/data/timestamp",
        "/entry/instrument/NDAttributes/NDArrayTimeStamp",
    ]:
        if path in h5_file:
            val = np.array(h5_file[path])
            if val.ndim == 0 and float(val) > 0:
                return np.arange(n_frames) * float(val)
            elif val.ndim == 1 and len(val) >= n_frames:
                t = val[:n_frames].astype(float)
                return t - t[0]
    return np.arange(n_frames) * float(dt_fallback)


def make_eiger_mask(sample_frame):
    """Mask Eiger module gaps and dead/hot pixels (< 0, 65535 uint16 max, or >= 1e8)."""
    bad = (~np.isfinite(sample_frame)) | (sample_frame < 0) | (sample_frame >= 1e8)
    if np.nanmax(sample_frame) == 65535:
        bad = bad | (sample_frame >= 65535)
    return bad.astype(np.uint8)


def load_detector_mask(mask_file, expected_shape=None):
    """
    Load an external detector mask (.edf, .tif, .npy, .npz, .h5) where non-zero != 0
    indicates masked pixels (standard pyFAI / pyFAI-drawmask / DIOPTAS convention).
    """
    if mask_file is None or mask_file is False:
        if expected_shape is not None:
            return np.zeros(tuple(expected_shape), dtype=np.uint8)
        return None

    mpath = Path(mask_file)
    if not mpath.exists():
        raise FileNotFoundError(f"Mask file not found: {mpath}")

    suffix = mpath.suffix.lower()
    if suffix == ".npy":
        raw_mask = np.load(mpath)
    elif suffix == ".npz":
        npz = np.load(mpath)
        raw_mask = npz[list(npz.keys())[0]]
    elif suffix in (".h5", ".hdf5"):
        with h5py.File(mpath, "r") as hf:
            for key in ("mask", "data", "/entry/data/data"):
                if key in hf:
                    raw_mask = hf[key][:]
                    break
            else:
                first_k = list(hf.keys())[0]
                raw_mask = hf[first_k][:]
    else:
        import fabio
        raw_mask = fabio.open(str(mpath)).data

    ext_mask = (np.asarray(raw_mask) != 0).astype(np.uint8)
    if expected_shape is not None and ext_mask.shape != tuple(expected_shape):
        raise ValueError(
            f"Mask shape {ext_mask.shape} in '{mpath.name}' does not match detector image shape {tuple(expected_shape)}."
        )
    return ext_mask


def build_combined_mask(sample_frame, mask_file=None, verbose=False):
    """Combine external mask (e.g. mask.edf) with automatic Eiger gap/hot-pixel mask."""
    eiger_mask = make_eiger_mask(sample_frame)
    ext_mask = load_detector_mask(mask_file, expected_shape=sample_frame.shape)
    combined = np.bitwise_or(eiger_mask, ext_mask).astype(np.uint8)
    if verbose:
        n_tot = combined.size
        n_ext = int(np.sum(ext_mask))
        n_comb = int(np.sum(combined))
        src_name = Path(mask_file).name if mask_file else "none"
        print(
            f"Detector mask ({src_name}) -> external={n_ext:,} px | "
            f"combined with Eiger gaps={n_comb:,}/{n_tot:,} px ({100.0 * n_comb / n_tot:.1f}%)"
        )
    return combined


def load_azimuthal_integrator(
    poni_file,
    reference_h5=None,
    img_shape=None,
    pixel_size_override=None,
    energy_kev_override=None,
):
    """
    Load a pyFAI AzimuthalIntegrator, verify it is not a macOS ._ sidecar file,
    and ensure pixel1/pixel2, detector shape, and X-ray wavelength (in meters) are populated.
    """
    poni_path = Path(poni_file)
    if poni_path.name.startswith("._"):
        raise ValueError(
            f"Refusing to load macOS AppleDouble sidecar file '{poni_path.name}'. "
            "Use the real .poni file without the '._' prefix."
        )

    poni_txt = poni_path.read_text(encoding="utf-8", errors="ignore")
    if "distance:" not in poni_txt.lower():
        raise ValueError(
            f"File '{poni_path.name}' does not contain a valid 'Distance:' entry. "
            "Check PONI_FILE_OVERRIDE in Cell 1."
        )

    ai = pyFAI.load(str(poni_path))

    # Sanity-check that pyFAI parsed Distance rather than falling back to 1.0 m default
    m_dist = re.search(r"^\s*Distance\s*:\s*([0-9.eE+-]+)", poni_txt, re.MULTILINE | re.IGNORECASE)
    if m_dist:
        expected_dist = float(m_dist.group(1))
        if not np.isclose(ai.dist, expected_dist, rtol=1e-4):
            raise RuntimeError(
                f"pyFAI loaded dist={ai.dist} m, which does not match Distance={expected_dist} m "
                f"in {poni_path.name}."
            )

    # 1. Resolve detector pixel dimensions (meters)
    p1 = getattr(ai, "pixel1", None)
    p2 = getattr(ai, "pixel2", None)
    if p1 is None or p2 is None:
        px_note = ""
        if pixel_size_override is not None:
            p1 = p2 = float(pixel_size_override)
            px_note = "PIXEL_SIZE_M override"
        else:
            m_cfg = re.search(r"^\s*Detector_config\s*:\s*(\{.+\})", poni_txt, re.MULTILINE)
            if m_cfg:
                try:
                    cfg_dict = json.loads(m_cfg.group(1))
                    if "pixel1" in cfg_dict and "pixel2" in cfg_dict:
                        p1 = float(cfg_dict["pixel1"])
                        p2 = float(cfg_dict["pixel2"])
                        px_note = f"{poni_path.name} Detector_config"
                except Exception:
                    pass
            if p1 is None or p2 is None:
                m1 = re.search(r'(?:PixelSize1|pixel1)["\s:=]+([0-9.eE+-]+)', poni_txt)
                m2 = re.search(r'(?:PixelSize2|pixel2)["\s:=]+([0-9.eE+-]+)', poni_txt)
                if m1 and m2:
                    p1, p2 = float(m1.group(1)), float(m2.group(1))
                    px_note = f"{poni_path.name} text"

        if (p1 is None or p2 is None) and reference_h5 is not None and Path(reference_h5).exists():
            with h5py.File(reference_h5, "r") as hf:
                if "/entry/instrument/detector/y_pixel_size" in hf and "/entry/instrument/detector/x_pixel_size" in hf:
                    p1 = float(np.array(hf["/entry/instrument/detector/y_pixel_size"]).flat[0])
                    p2 = float(np.array(hf["/entry/instrument/detector/x_pixel_size"]).flat[0])
                    px_note = f"{Path(reference_h5).name} metadata"

        if p1 is None or p2 is None:
            p1 = p2 = 7.5e-5
            px_note = "default Eiger 75 um fallback"

        if getattr(ai, "detector", None) is not None:
            ai.detector.pixel1 = p1
            ai.detector.pixel2 = p2
            if img_shape is not None and getattr(ai.detector, "shape", None) is None:
                ai.detector.shape = tuple(img_shape)
                ai.detector.max_shape = tuple(img_shape)
        ai.pixel1 = p1
        ai.pixel2 = p2
        print(f"Resolved pixel size -> pixel1={p1*1e6:.1f} um, pixel2={p2*1e6:.1f} um ({px_note})")

    # 2. Resolve X-ray wavelength (meters)
    wl = getattr(ai, "wavelength", None)
    if energy_kev_override is not None:
        wl = HC_KEV_M / float(energy_kev_override)
        ai.wavelength = wl
        print(f"Set X-ray wavelength -> {wl*1e10:.5f} A ({float(energy_kev_override):.4f} keV)")
    elif wl is None:
        wl_note = ""
        m_wl = re.search(r"^\s*Wavelength\s*:\s*([0-9.eE+-]+)", poni_txt, re.MULTILINE | re.IGNORECASE)
        if m_wl:
            wl = float(m_wl.group(1))
            wl_note = f"{poni_path.name} text"
        elif reference_h5 is not None and Path(reference_h5).exists():
            with h5py.File(reference_h5, "r") as hf:
                if "/entry/instrument/beam/incident_wavelength" in hf:
                    val = float(np.array(hf["/entry/instrument/beam/incident_wavelength"]).flat[0])
                    wl = val * 1e-10 if val > 1e-3 else val
                    wl_note = f"{Path(reference_h5).name} metadata"
                elif "/entry/instrument/beam/incident_energy" in hf:
                    en = float(np.array(hf["/entry/instrument/beam/incident_energy"]).flat[0])
                    en_kev = en / 1000.0 if en > 1000.0 else en
                    wl = HC_KEV_M / en_kev
                    wl_note = f"{Path(reference_h5).name} energy metadata"

        if wl is None:
            raise ValueError(
                "X-ray wavelength is missing from the .poni file. "
                "Please set BEAM_ENERGY_KEV (e.g., 12.7) in Cell 1."
            )
        ai.wavelength = wl
        print(f"Resolved X-ray wavelength -> {wl*1e10:.5f} A ({wl_note})")

    if hasattr(ai, "reset"):
        ai.reset()
    return ai


# ==============================================================================
# B. THOMPSON-COX-HASTINGS (TCH) PSEUDO-VOIGT & 1- OR 2-PEAK DECONVOLUTION
# ==============================================================================
def tch_fwhm_and_eta(H_G, H_L):
    """Convert pure Gaussian (H_G) and Lorentzian (H_L) FWHMs to total FWHM (H) and mixing (eta)."""
    H = (
        H_G**5
        + 2.69269 * (H_G**4) * H_L
        + 2.42843 * (H_G**3) * H_L**2
        + 4.47163 * (H_G**2) * H_L**3
        + 0.07842 * H_G * (H_L**4)
        + H_L**5
    ) ** 0.2
    ratio = H_L / H
    eta = 1.36603 * ratio - 0.47719 * (ratio**2) + 0.11116 * (ratio**3)
    return H, eta


def tch_pseudo_voigt(q, A, q0, H_G, H_L):
    """Area-normalized TCH pseudo-Voigt profile."""
    H, eta = tch_fwhm_and_eta(H_G, H_L)
    sigma_factor = 2.0 * np.sqrt(np.log(2.0)) / (H * np.sqrt(np.pi))
    G = sigma_factor * np.exp(-4.0 * np.log(2.0) * ((q - q0) / H) ** 2)
    L = (2.0 / (np.pi * H)) / (1.0 + 4.0 * ((q - q0) / H) ** 2)
    return A * (eta * L + (1.0 - eta) * G)


def free_peak_model(q, A, q0, H_G, H_L, slope, intercept):
    """Unconvolved TCH pseudo-Voigt + linear background (for thin-film LaB6 standard)."""
    return tch_pseudo_voigt(q, A, q0, H_G, H_L) + slope * (q - q0) + intercept


def make_deconvolved_peak_model(H_G_inst, H_L_inst, q_ref=1.00):
    """Single-peak sample profile convolved with thin-film LaB6 instrument broadening."""
    def deconvolved_model(q, A, q0, H_G_sample, H_L_sample, slope, intercept):
        H_G_tot = np.sqrt(H_G_inst**2 + H_G_sample**2)
        H_L_tot = H_L_inst + H_L_sample
        return tch_pseudo_voigt(q, A, q0, H_G_tot, H_L_tot) + slope * (q - q_ref) + intercept
    return deconvolved_model


def make_double_deconvolved_peak_model(H_G_inst, H_L_inst, q_ref=1.00):
    """Two-peak sample profile (alpha-(100) + overlapping delta/polytype) convolved with LaB6."""
    def double_deconvolved_model(
        q, A1, q0_1, HG_s1, HL_s1, A2, q0_2, HG_s2, HL_s2, slope, intercept
    ):
        HG_t1 = np.sqrt(H_G_inst**2 + HG_s1**2)
        HL_t1 = H_L_inst + HL_s1
        HG_t2 = np.sqrt(H_G_inst**2 + HG_s2**2)
        HL_t2 = H_L_inst + HL_s2
        peak1 = tch_pseudo_voigt(q, A1, q0_1, HG_t1, HL_t1)
        peak2 = tch_pseudo_voigt(q, A2, q0_2, HG_t2, HL_t2)
        return peak1 + peak2 + slope * (q - q_ref) + intercept
    return double_deconvolved_model


def fit_single_or_double_peak(
    q_sub,
    I_sub,
    single_model,
    double_model,
    q_alpha_bounds=(0.96, 1.05),
    q_delta_bounds=(0.82, 0.96),
    enable_second_peak=True,
    delta_bic_threshold=8.0,
    q_ref=0.996,
):
    """
    Fit primary alpha-(100) peak and, if enabled, test whether an overlapping secondary
    delta-phase peak improves BIC significantly without overfitting.
    """
    n_pts = len(q_sub)
    bg_edge = np.concatenate([I_sub[:6], I_sub[-6:]])
    bg_mean = float(np.mean(bg_edge))

    alpha_mask = (q_sub >= q_alpha_bounds[0]) & (q_sub <= q_alpha_bounds[1])
    if np.any(alpha_mask):
        q0_1_init = float(q_sub[alpha_mask][np.argmax(I_sub[alpha_mask])])
        prom_1 = max(float(np.max(I_sub[alpha_mask]) - bg_mean), 1e-3)
    else:
        q0_1_init = float(q_sub[np.argmax(I_sub)])
        prom_1 = max(float(np.max(I_sub) - bg_mean), 1e-3)

    H_init = 0.018
    p0_1 = [prom_1 * H_init, q0_1_init, H_init * 0.6, H_init * 0.6, 0.0, bg_mean]
    b_1 = (
        [0.0, q_alpha_bounds[0], 1e-5, 1e-5, -np.inf, -np.inf],
        [np.inf, q_alpha_bounds[1], 0.25, 0.25, np.inf, np.inf],
    )

    popt1, _ = curve_fit(single_model, q_sub, I_sub, p0=p0_1, bounds=b_1, maxfev=12000)
    fit1 = single_model(q_sub, *popt1)
    rss1 = float(np.sum((I_sub - fit1) ** 2))
    bic1 = n_pts * np.log(max(rss1 / n_pts, 1e-12)) + len(popt1) * np.log(n_pts)

    if not enable_second_peak:
        return {
            "n_peaks": 1,
            "main_params": popt1[:4],
            "delta_params": None,
            "bg_params": popt1[4:],
            "I_total_fit": fit1,
            "I_main_comp": fit1,
            "I_delta_comp": None,
        }

    # Seed secondary peak from the largest positive residual in q_delta_bounds
    res1 = I_sub - fit1
    delta_mask = (q_sub >= q_delta_bounds[0]) & (q_sub <= q_delta_bounds[1])
    if np.any(delta_mask) and np.max(res1[delta_mask]) > 0:
        idx_d = int(np.argmax(res1[delta_mask]))
        q0_2_init = float(q_sub[delta_mask][idx_d])
        prom_2 = max(float(res1[delta_mask][idx_d]), 0.03 * prom_1)
    else:
        q0_2_init = 0.5 * (q_delta_bounds[0] + q_delta_bounds[1])
        prom_2 = 0.08 * prom_1

    p0_2 = [
        popt1[0], popt1[1], popt1[2], popt1[3],
        prom_2 * 0.02, q0_2_init, 0.014, 0.014,
        popt1[4], popt1[5],
    ]
    b_2 = (
        [0.0, q_alpha_bounds[0], 1e-5, 1e-5, 0.0, q_delta_bounds[0], 1e-5, 1e-5, -np.inf, -np.inf],
        [np.inf, q_alpha_bounds[1], 0.25, 0.25, np.inf, q_delta_bounds[1], 0.25, 0.25, np.inf, np.inf],
    )

    try:
        popt2, _ = curve_fit(double_model, q_sub, I_sub, p0=p0_2, bounds=b_2, maxfev=20000)
        fit2 = double_model(q_sub, *popt2)
        rss2 = float(np.sum((I_sub - fit2) ** 2))
        bic2 = n_pts * np.log(max(rss2 / n_pts, 1e-12)) + len(popt2) * np.log(n_pts)

        H_samp2, _ = tch_fwhm_and_eta(popt2[6], popt2[7])
        peak_sep = abs(popt2[1] - popt2[5])
        rel_area = popt2[4] / max(popt2[0], 1e-9)

        # Accept 2-peak model only when BIC improves significantly and peak 2 is physically distinct
        if (
            (bic1 - bic2 > delta_bic_threshold)
            and (rel_area >= 0.015)
            and (peak_sep >= 0.015)
            and (H_samp2 <= 0.18)
        ):
            bg_line = popt2[8] * (q_sub - q_ref) + popt2[9]
            main_only = single_model(q_sub, popt2[0], popt2[1], popt2[2], popt2[3], popt2[8], popt2[9])
            delta_only = fit2 - main_only + bg_line
            return {
                "n_peaks": 2,
                "main_params": popt2[:4],
                "delta_params": popt2[4:8],
                "bg_params": popt2[8:],
                "I_total_fit": fit2,
                "I_main_comp": main_only,
                "I_delta_comp": delta_only,
            }
    except RuntimeError:
        pass

    return {
        "n_peaks": 1,
        "main_params": popt1[:4],
        "delta_params": None,
        "bg_params": popt1[4:],
        "I_total_fit": fit1,
        "I_main_comp": fit1,
        "I_delta_comp": None,
    }


# ==============================================================================
# C. PURE-PYTHON .CIF PARSER & BRAGG STRUCTURE FACTOR CALCULATOR
# ==============================================================================
ATOMIC_Z = {
    "H": 1, "He": 2, "Li": 3, "Be": 4, "B": 5, "C": 6, "N": 7, "O": 8, "F": 9, "Ne": 10,
    "Na": 11, "Mg": 12, "Al": 13, "Si": 14, "P": 15, "S": 16, "Cl": 17, "Ar": 18,
    "K": 19, "Ca": 20, "Sc": 21, "Ti": 22, "V": 23, "Cr": 24, "Mn": 25, "Fe": 26,
    "Co": 27, "Ni": 28, "Cu": 29, "Zn": 30, "Ga": 31, "Ge": 32, "As": 33, "Se": 34,
    "Br": 35, "Kr": 36, "Rb": 37, "Sr": 38, "Y": 39, "Zr": 40, "Nb": 41, "Mo": 42,
    "Ru": 44, "Rh": 45, "Pd": 46, "Ag": 47, "Cd": 48, "In": 49, "Sn": 50, "Sb": 51,
    "Te": 52, "I": 53, "Xe": 54, "Cs": 55, "Ba": 56, "La": 57, "Ce": 58, "Pr": 59,
    "Nd": 60, "Sm": 62, "Eu": 63, "Gd": 64, "Yb": 70, "W": 74, "Pt": 78, "Au": 79,
    "Hg": 80, "Tl": 81, "Pb": 82, "Bi": 83,
}


def _parse_cif_float(token):
    cleaned = re.sub(r"\([^)]*\)", "", str(token)).strip()
    return float(cleaned)


def _tokenize_cif_line(line):
    return [m.group(1) or m.group(2) or m.group(3) for m in re.finditer(r"'([^']*)'|\"([^\"]*)\"|(\S+)", line)]


def _eval_symop(op_str, x, y, z):
    parts = [p.strip().lower() for p in op_str.split(",")]
    if len(parts) != 3:
        return None
    ns = {"x": x, "y": y, "z": z}
    coords = []
    for expr in parts:
        expr_eval = re.sub(r"(\d+)\s*/\s*(\d+)", lambda m: str(float(m.group(1)) / float(m.group(2))), expr)
        coords.append(float(eval(expr_eval, {"__builtins__": {}}, ns)) % 1.0)
    return np.array(coords)


def parse_cif_file(cif_path, q_min=0.35, q_max=2.50, min_rel_intensity=1.0):
    """Parse a .cif file and compute theoretical Bragg reflections (q, I_rel, hkl)."""
    cif_path = Path(cif_path)
    raw_lines = cif_path.read_text(encoding="utf-8", errors="ignore").splitlines()

    lines = []
    in_semicolon_block = False
    for raw in raw_lines:
        s = raw.strip()
        if s.startswith(";"):
            in_semicolon_block = not in_semicolon_block
            continue
        if in_semicolon_block or not s or s.startswith("#"):
            continue
        lines.append(s)

    scalars = {}
    loops = []
    i = 0
    while i < len(lines):
        ln = lines[i]
        if ln.lower() == "loop_":
            i += 1
            headers = []
            while i < len(lines) and lines[i].startswith("_"):
                headers.append(lines[i].split()[0].lower())
                i += 1
            rows = []
            while (
                i < len(lines)
                and not lines[i].startswith("_")
                and lines[i].lower() != "loop_"
                and not lines[i].lower().startswith("data_")
            ):
                tokens = _tokenize_cif_line(lines[i])
                if len(tokens) == len(headers):
                    rows.append(tokens)
                i += 1
            loops.append((headers, rows))
            continue
        elif ln.startswith("_"):
            tokens = _tokenize_cif_line(ln)
            if len(tokens) >= 2:
                scalars[tokens[0].lower()] = tokens[1]
        i += 1

    a = _parse_cif_float(scalars["_cell_length_a"])
    b = _parse_cif_float(scalars["_cell_length_b"])
    c = _parse_cif_float(scalars["_cell_length_c"])
    alpha = np.deg2rad(_parse_cif_float(scalars.get("_cell_angle_alpha", 90.0)))
    beta = np.deg2rad(_parse_cif_float(scalars.get("_cell_angle_beta", 90.0)))
    gamma = np.deg2rad(_parse_cif_float(scalars.get("_cell_angle_gamma", 90.0)))

    label = cif_path.stem

    symops = ["x, y, z"]
    for headers, rows in loops:
        for sym_key in ("_space_group_symop_operation_xyz", "_symmetry_equiv_pos_as_xyz"):
            if sym_key in headers:
                idx_s = headers.index(sym_key)
                symops = [r[idx_s] for r in rows]

    base_atoms = []
    for headers, rows in loops:
        if {"_atom_site_fract_x", "_atom_site_fract_y", "_atom_site_fract_z"}.issubset(headers):
            ix = headers.index("_atom_site_fract_x")
            iy = headers.index("_atom_site_fract_y")
            iz = headers.index("_atom_site_fract_z")
            isym = (
                headers.index("_atom_site_type_symbol")
                if "_atom_site_type_symbol" in headers
                else headers.index("_atom_site_label")
            )
            iocc = headers.index("_atom_site_occupancy") if "_atom_site_occupancy" in headers else None
            for r in rows:
                m_el = re.match(r"([A-Za-z]+)", r[isym])
                if not m_el:
                    continue
                elem_raw = m_el.group(1)
                elem = elem_raw[0].upper() + elem_raw[1:].lower() if len(elem_raw) > 1 else elem_raw.upper()
                Z = ATOMIC_Z.get(elem, 10)
                x, y, z = _parse_cif_float(r[ix]), _parse_cif_float(r[iy]), _parse_cif_float(r[iz])
                occ = _parse_cif_float(r[iocc]) if iocc is not None else 1.0
                base_atoms.append((Z, occ, x, y, z))

    expanded_atoms = []
    for Z, occ, x, y, z in base_atoms:
        unique_coords = []
        for op in symops:
            try:
                c_new = _eval_symop(op, x, y, z)
            except Exception:
                continue
            if c_new is None:
                continue
            if not any(
                np.allclose(c_new, u, atol=1e-3) or np.allclose(np.abs(c_new - u), 1.0, atol=1e-3)
                for u in unique_coords
            ):
                unique_coords.append(c_new)
        if not unique_coords:
            unique_coords = [np.array([x % 1.0, y % 1.0, z % 1.0])]
        for u in unique_coords:
            expanded_atoms.append((Z, occ, u[0], u[1], u[2]))

    G = np.array([
        [a * a, a * b * np.cos(gamma), a * c * np.cos(beta)],
        [b * a * np.cos(gamma), b * b, b * c * np.cos(alpha)],
        [c * a * np.cos(beta), c * b * np.cos(alpha), c * c],
    ])
    G_star = np.linalg.inv(G)

    h_max = int(np.ceil(q_max * a / (2 * np.pi))) + 1
    k_max = int(np.ceil(q_max * b / (2 * np.pi))) + 1
    l_max = int(np.ceil(q_max * c / (2 * np.pi))) + 1

    hs, ks, ls = np.meshgrid(
        np.arange(-h_max, h_max + 1),
        np.arange(-k_max, k_max + 1),
        np.arange(-l_max, l_max + 1),
        indexing="ij",
    )
    hkl_all = np.column_stack([hs.ravel(), ks.ravel(), ls.ravel()])
    hkl_all = hkl_all[np.any(hkl_all != 0, axis=1)]

    inv_d2 = np.einsum("ni,ij,nj->n", hkl_all, G_star, hkl_all)
    valid = inv_d2 > 0
    hkl_all = hkl_all[valid]
    q_all = 2.0 * np.pi * np.sqrt(inv_d2[valid])

    in_range = (q_all >= q_min) & (q_all <= q_max)
    hkl_all = hkl_all[in_range]
    q_all = q_all[in_range]

    if expanded_atoms:
        Zs = np.array([at[0] * at[1] for at in expanded_atoms])
        coords = np.array([[at[2], at[3], at[4]] for at in expanded_atoms])
        f_q = Zs[None, :] * np.exp(-0.5 * (q_all[:, None] / (4 * np.pi)) ** 2)
        phase = 2.0 * np.pi * (hkl_all @ coords.T)
        F_hkl = np.sum(f_q * np.exp(1j * phase), axis=1)
        I_hkl = np.abs(F_hkl) ** 2
    else:
        I_hkl = np.ones_like(q_all)

    order = np.argsort(q_all)
    q_sorted, I_sorted, hkl_sorted = q_all[order], I_hkl[order], hkl_all[order]
    grouped = []
    for q_val, I_val, hkl_val in zip(q_sorted, I_sorted, hkl_sorted):
        if grouped and abs(q_val - grouped[-1]["q"]) < 1e-4:
            grouped[-1]["I"] += I_val
            if sum(hkl_val >= 0) > sum(grouped[-1]["hkl"] >= 0):
                grouped[-1]["hkl"] = hkl_val
        else:
            grouped.append({"q": float(q_val), "I": float(I_val), "hkl": hkl_val})

    max_I = max((g["I"] for g in grouped), default=1.0)
    peaks = []
    for g in grouped:
        rel_I = 100.0 * g["I"] / max_I if max_I > 0 else 0.0
        if rel_I >= min_rel_intensity:
            h, k, l = g["hkl"]
            peaks.append({"q": g["q"], "I_rel": rel_I, "hkl": f"({h}{k}{l})"})

    return {
        "label": label,
        "path": str(cif_path),
        "cell": (a, b, c),
        "peaks": peaks,
    }


print(f"Engine & CIF calculator ready (CPU mode capped at {_TARGET_THREADS}/{_N_CPU} threads; GPU idle).")


## Cell 2B: Shared settings & helpers (single source of truth)

Everything that Cells 5 – 7C used to copy-paste lives here once: q-windows, Scherrer K, which samples are processed (exclusions + sample table), beam-damage cut-offs, the **LaB₆ instrument broadening** (`H_G_inst`, `H_L_inst` from the calibration, no hard-coded fallback), the t_max rule, the peak-area definition, and the single α-(100) fit (TCH pseudo-Voigt convolved with the instrument profile) with its 95 % CIs. Run it once after Cell 2; the calibration is read when it is needed.


In [ ]:
# ==============================================================================
# SHARED ANALYSIS SETTINGS & HELPERS  — single source of truth for Cells 5 – 7C
# Run AFTER Cell 1 (settings + sample table) and Cell 2 (peak models). The LaB6 calibration (CALIB_DATA)
# is read lazily, so this cell can run before the calibration cell.
#
# Everything that used to be copy-pasted into several cells lives here once:
#   • q-windows, Scherrer K, resolution limit
#   • which samples are processed (exclusion list + "must be on the sample table")
#   • per-sample beam-damage cut-offs
#   • instrument broadening (H_G_inst / H_L_inst from the LaB6 calibration — NO hard-coded fallback)
#   • which frame is "t_max", how the (100) peak area is measured
#   • ONE alpha-(100) peak fit (TCH pseudo-Voigt convolved with the LaB6 profile) and its 95 % CIs
# ==============================================================================
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import norm as _norm

_trapz = getattr(np, "trapezoid", None) or getattr(np, "trapz")

# ---- windows / constants -----------------------------------------------------------------------
Q_100_SEARCH_WINDOW = globals().get("Q_100_SEARCH_WINDOW", (0.96, 1.05))     # fit window for alpha-(100)   (Cell 1)
Q_100_CORE_WINDOW = globals().get("Q_100_CORE_WINDOW", (0.975, 1.018))       # tight core used to pick t_max
Q_DELTA_SEARCH_WINDOW = globals().get("Q_DELTA_SEARCH_WINDOW", (0.82, 0.96))
Q_PBI2_WINDOW = globals().get("Q_PBI2_WINDOW", (0.875, 0.930))               # PbI2 (001) area window
SCHERRER_K = 0.9                       # Scherrer shape factor, used by EVERY cell
LC_RES_LIMIT_NM = 1500.0               # L_c above this = resolution-limited (treated as "no value")
LAB6_REL_ERR = 0.03                    # relative uncertainty on the LaB6 instrumental widths (for the L_c CIs)
MIN_FRAMES = 10                        # scans with fewer frames are skipped
KINETICS_MIN_POINTS = 10
KINETICS_MIN_PROM_FRAC = 0.04          # frames with (100) prominence < 4 % of the strongest frame = "no peak yet"

# ---- which samples are processed -----------------------------------------------------------------
EXACT_EXCLUDE = {"s1a", "test", "s15a", "s15b"}      # s1a replaced by S1A-try2; s15 = glass background
SUBSTR_EXCLUDE = ("burst_test", "lab6", "shear", "shearing", "coat", "coating")
DROPPED_NOT_IN_TABLE = set()


def is_excluded_sample(name_or_path):
    """True if a scan must not be processed: excluded by name, or (ONLY_TABLE_SAMPLES) not on the sample table."""
    from pathlib import Path as _P
    low = str(name_or_path).replace("_reduced", "").strip().lower()
    stem_low = _P(str(name_or_path)).stem.replace("_reduced", "").strip().lower()
    if stem_low in EXACT_EXCLUDE or low in EXACT_EXCLUDE:
        return True
    if any(sub in low for sub in SUBSTR_EXCLUDE):
        return True
    if globals().get("ONLY_TABLE_SAMPLES", False) and not sample_info(stem_low)["known"]:
        DROPPED_NOT_IN_TABLE.add(stem_low)
        return True
    return False


# ---- beam-damage cut-offs (seconds; analysis stops there) --------------------------------------------
SAMPLE_T_CUTOFFS = {
    "Y2A": 50.0,
    "S1A-try2": 32.0,
    "S12A": 26.0,
    "S9A": 26.0,
    "F6B": 31.0,
}
_CUTOFF_LOOKUP = {k.strip().lower(): float(v) for k, v in SAMPLE_T_CUTOFFS.items()}


def get_sample_cutoff(name):
    return _CUTOFF_LOOKUP.get(str(name).replace("_reduced", "").strip().lower(), None)


# ---- instrument broadening (LaB6) -----------------------------------------------------------------
def get_instrument(calib=None):
    """Instrument widths from the LaB6 calibration, evaluated at the sample (100) q.
    H_G / H_L = Gaussian / Lorentzian FWHM (A^-1), H_tot = TCH total FWHM, eta = TCH mixing.
    There is deliberately NO hard-coded fallback: a wrong silent default is worse than an error."""
    calib = calib if calib is not None else globals().get("CALIB_DATA")
    if not isinstance(calib, dict) or "H_G_inst" not in calib or "H_L_inst" not in calib:
        raise KeyError("CALIB_DATA with 'H_G_inst' and 'H_L_inst' is required - run the LaB6 calibration cell first.")
    HG, HL = float(calib["H_G_inst"]), float(calib["H_L_inst"])
    H, eta = tch_fwhm_and_eta(HG, HL)
    return {"H_G": HG, "H_L": HL, "H_tot": float(H), "eta": float(eta), "rel_err": LAB6_REL_ERR}


# ---- t_max and peak area ---------------------------------------------------------------------------
def find_alpha100_frame_max(q_arr, I_mat, t_arr=None, t_cutoff=None, q_core=None):
    """Frame with the strongest alpha-(100) peak = height above a LOCAL baseline in a tight core window,
    restricted to t <= the sample's beam-damage cutoff. Used by every cell."""
    q_core = q_core or Q_100_CORE_WINDOW
    q_arr = np.asarray(q_arr, float)
    m = (q_arr >= q_core[0]) & (q_arr <= q_core[1])
    if np.sum(m) < 8:
        m = (q_arr >= Q_100_SEARCH_WINDOW[0]) & (q_arr <= Q_100_SEARCH_WINDOW[1])
    sub = np.asarray(I_mat, float)[:, m]
    edge = max(2, min(4, sub.shape[1] // 6))
    bg_local = 0.5 * (np.mean(sub[:, :edge], axis=1, keepdims=True) + np.mean(sub[:, -edge:], axis=1, keepdims=True))
    prom = np.max(np.maximum(sub - bg_local, 0.0), axis=1)
    if t_arr is not None and t_cutoff is not None:
        valid_t = np.asarray(t_arr) <= float(t_cutoff)
        if np.any(valid_t):
            prom = np.where(valid_t, prom, -np.inf)
    return int(np.argmax(prom))


def baseline_subtracted_area(q_arr, I_mat, q_win):
    """Peak area above a straight-line baseline between the window edges (the one area definition used everywhere)."""
    q_arr = np.asarray(q_arr, float)
    m = (q_arr >= q_win[0]) & (q_arr <= q_win[1])
    qs = q_arr[m]
    sub = np.asarray(I_mat, float)[:, m]
    edge = max(2, min(4, qs.size // 6))
    y_left = np.mean(sub[:, :edge], axis=1, keepdims=True)
    y_right = np.mean(sub[:, -edge:], axis=1, keepdims=True)
    frac = ((qs - qs[0]) / max(qs[-1] - qs[0], 1e-9))[None, :]
    bg = y_left + frac * (y_right - y_left)
    return _trapz(np.maximum(sub - bg, 0.0), qs, axis=1)


# ---- the alpha-(100) peak fit ------------------------------------------------------------------------
_MODEL_CACHE = {}


def _deconvolved_models(inst, q_ref):
    key = (round(inst["H_G"], 9), round(inst["H_L"], 9), round(q_ref, 6))
    if key not in _MODEL_CACHE:
        _MODEL_CACHE[key] = (make_deconvolved_peak_model(inst["H_G"], inst["H_L"], q_ref=q_ref),
                             make_double_deconvolved_peak_model(inst["H_G"], inst["H_L"], q_ref=q_ref))
    return _MODEL_CACHE[key]


def _q_ref():
    return float(globals().get("Q_TARGET_100", 0.990))


def fit_alpha100(q, y, q_win=None, inst=None, scherrer_k=None, enable_second_peak=None):
    """Fit one alpha-(100) peak with a TCH pseudo-Voigt CONVOLVED with the LaB6 instrument profile, so the fitted
    widths are SAMPLE-ONLY (same engine as the Cell 5D kinetics fit). Returns a dict, or None if the fit fails.
    Keys: q_sub, y_sub, y_fit, amp, q0, a_ang, fwhm_obs (observed total), fwhm_corr (sample-only total), eta
    (sample-only TCH mixing, 0 = Gaussian, 1 = Lorentzian), H_G_s, H_L_s, r2, Lc_nm, n_peaks."""
    q_win = tuple(q_win) if q_win is not None else tuple(Q_100_SEARCH_WINDOW)
    inst = inst or get_instrument()
    k = SCHERRER_K if scherrer_k is None else float(scherrer_k)
    two = bool(globals().get("ENABLE_SECONDARY_PEAK", False)) if enable_second_peak is None else bool(enable_second_peak)
    q = np.asarray(q, float)
    y = np.asarray(y, float)
    q_lo = min(q_win[0], Q_DELTA_SEARCH_WINDOW[0]) if two else q_win[0]
    m = (q >= q_lo) & (q <= q_win[1]) & np.isfinite(y)
    if m.sum() < KINETICS_MIN_POINTS or np.nanmax(y[m]) <= 0:
        return None
    qs, ys = q[m], y[m]
    single, double = _deconvolved_models(inst, _q_ref())
    try:
        import warnings as _w
        with _w.catch_warnings():
            _w.simplefilter("ignore")
            res = fit_single_or_double_peak(qs, ys, single, double, q_alpha_bounds=q_win,
                                            q_delta_bounds=tuple(Q_DELTA_SEARCH_WINDOW),
                                            enable_second_peak=two, q_ref=_q_ref())
    except (RuntimeError, ValueError):
        return None
    A, q0, HG_s, HL_s = (float(v) for v in res["main_params"])
    H_s, eta_s = tch_fwhm_and_eta(HG_s, HL_s)
    H_obs, _ = tch_fwhm_and_eta(np.sqrt(inst["H_G"] ** 2 + HG_s ** 2), inst["H_L"] + HL_s)
    y_fit = np.asarray(res["I_total_fit"], float)
    ss_res = float(np.sum((ys - y_fit) ** 2))
    ss_tot = float(np.sum((ys - np.mean(ys)) ** 2))
    Lc = (2.0 * np.pi * k / H_s) * 0.1 if H_s > 0 else np.nan          # A -> nm
    if np.isfinite(Lc) and Lc > LC_RES_LIMIT_NM:
        Lc = np.nan                                                      # resolution-limited
    return {"q_sub": qs, "y_sub": ys, "y_fit": y_fit, "amp": A, "q0": q0, "a_ang": float(2.0 * np.pi / q0),
            "fwhm_obs": float(H_obs), "fwhm_corr": float(H_s), "eta": float(eta_s), "H_G_s": HG_s, "H_L_s": HL_s,
            "r2": 1.0 - ss_res / max(ss_tot, 1e-12), "Lc_nm": float(Lc) if np.isfinite(Lc) else np.nan,
            "n_peaks": int(res["n_peaks"])}


def _lc_ci_from_cov(HG_s, HL_s, cov_GL, inst, k, z):
    """95 % CI of L_c: fit covariance of (H_G_s, H_L_s) + LaB6 width uncertainty, by the delta method."""
    H0, _ = tch_fwhm_and_eta(HG_s, HL_s)
    if not (np.isfinite(H0) and H0 > 0):
        return np.nan
    def _dH(Ga, La, Gb, Lb):
        return (tch_fwhm_and_eta(Ga, La)[0] - tch_fwhm_and_eta(Gb, Lb)[0]) / ((Ga - Gb) if Ga != Gb else (La - Lb))
    e = 1e-6
    dH_dG = _dH(HG_s + e, HL_s, max(HG_s - e, 1e-9), HL_s)
    dH_dL = _dH(HG_s, HL_s + e, HG_s, max(HL_s - e, 1e-9))
    g = np.array([dH_dG, dH_dL])
    var = float(g @ np.asarray(cov_GL, float) @ g)
    # instrument uncertainty: HG_s^2 = HG_obs^2 - HG_inst^2  ->  dHG_s/dHG_inst = -HG_inst/HG_s ;  dHL_s/dHL_inst = -1
    sG = inst["rel_err"] * inst["H_G"]
    sL = inst["rel_err"] * inst["H_L"]
    var += (dH_dG * (inst["H_G"] / max(HG_s, 1e-9)) * sG) ** 2 + (dH_dL * sL) ** 2
    lc = (2.0 * np.pi * k / H0) * 0.1
    return float(z * lc * np.sqrt(max(var, 0.0)) / H0)


def frame_uncertainties(q_arr, I_mat, f_cut, q_win=None, z=1.96, inst=None, scherrer_k=None):
    """Per-frame 95 % CIs for area, q0 and L_c from the covariance of the SAME convolved fit used for the values.
    -> (area_ci, q0_ci, lc_ci) arrays of length n_frames (NaN where there is no usable fit)."""
    q_win = tuple(q_win) if q_win is not None else tuple(Q_100_SEARCH_WINDOW)
    inst = inst or get_instrument()
    k = SCHERRER_K if scherrer_k is None else float(scherrer_k)
    n_frames = I_mat.shape[0]
    area_ci = np.full(n_frames, np.nan)
    q0_ci = np.full(n_frames, np.nan)
    lc_ci = np.full(n_frames, np.nan)
    q_arr = np.asarray(q_arr, float)
    m = (q_arr >= q_win[0]) & (q_arr <= q_win[1])
    qs = q_arr[m]
    if qs.size < KINETICS_MIN_POINTS:
        return area_ci, q0_ci, lc_ci
    sqrt_n_dq = np.sqrt(qs.size) * float(np.median(np.diff(qs)))
    end_f = min(int(f_cut), n_frames - 1)
    peak_y_max = float(np.nanmax(np.asarray(I_mat)[: end_f + 1][:, m]))
    if peak_y_max <= 0:
        return area_ci, q0_ci, lc_ci
    single, _ = _deconvolved_models(inst, _q_ref())
    p_prev = None
    for idx in range(end_f + 1):
        ys = np.asarray(I_mat[idx, m], float)
        ok = np.isfinite(ys)
        y_max = float(np.nanmax(ys)) if ok.any() else 0.0
        if y_max <= KINETICS_MIN_PROM_FRAC * peak_y_max or ok.sum() < KINETICS_MIN_POINTS:
            if ok.any():
                area_ci[idx] = z * float(np.std(ys[ok])) * sqrt_n_dq
            continue
        bg0 = float(np.mean(np.concatenate([ys[:6], ys[-6:]])))
        i0 = int(np.argmax(ys))
        prom = max(y_max - bg0, 1e-3)
        H0 = 0.018
        p0 = p_prev if (p_prev is not None and abs(p_prev[1] - qs[i0]) < 0.015) else \
            [prom * H0, float(qs[i0]), H0 * 0.6, H0 * 0.6, 0.0, bg0]
        p0 = [max(p0[0], 1e-6), float(np.clip(p0[1], q_win[0] + 1e-4, q_win[1] - 1e-4)),
              float(np.clip(p0[2], 1e-5, 0.25)), float(np.clip(p0[3], 1e-5, 0.25)), p0[4], p0[5]]
        bounds = ([0.0, q_win[0], 1e-5, 1e-5, -np.inf, -np.inf], [np.inf, q_win[1], 0.25, 0.25, np.inf, np.inf])
        try:
            import warnings as _w
            with _w.catch_warnings():
                _w.simplefilter("ignore")
                popt, pcov = curve_fit(single, qs[ok], ys[ok], p0=p0, bounds=bounds, maxfev=4000)
            p_prev = popt
            if not np.all(np.isfinite(pcov)):
                continue
            se = np.sqrt(np.maximum(np.diag(pcov), 0.0))
            area_ci[idx] = z * float(se[0])
            q0_ci[idx] = z * float(se[1])
            lc_ci[idx] = _lc_ci_from_cov(float(popt[2]), float(popt[3]), pcov[2:4, 2:4], inst, k, z)
        except Exception:
            continue
    return area_ci, q0_ci, lc_ci


_inst_try = None
try:
    _inst_try = get_instrument()
except KeyError:
    pass
if _inst_try:
    print(f"Instrument broadening (LaB6): H_G = {_inst_try['H_G']:.5f}, H_L = {_inst_try['H_L']:.5f}, "
          f"total = {_inst_try['H_tot']:.5f} A^-1")
print("Shared helpers loaded (windows, exclusions, cutoffs, instrument broadening, t_max, area, (100) fit).")


# ---- frame time axis (single source of truth for the cells that read the cached .npz files) ------------
_TIME_AXIS_WARNED = set()


def resolve_time_axis(name, n_frames, dtype=np.float64):
    """Frame times (s) for one sample: REDUCED_DATA['time_s'] (read from the h5 in Cell 5) if available,
    otherwise n * DT_SEC_FALLBACK with a warning. Used when the cached .npz has no time axis."""
    key = str(name).replace("_reduced", "").strip().lower()
    red = globals().get("REDUCED_DATA") or {}
    for k, v in red.items():
        if str(k).replace("_reduced", "").strip().lower() == key:
            t = v.get("time_s") if isinstance(v, dict) else None
            if t is not None and len(t) == n_frames:
                return np.asarray(t, dtype=dtype)
    if key not in _TIME_AXIS_WARNED:
        _TIME_AXIS_WARNED.add(key)
        print(f"  ! {name}: no stored time axis -> assuming {globals().get('DT_SEC_FALLBACK', 0.5)} s per frame "
              "(rerun Cell 5 so the real frame times are saved in the cache)")
    return np.arange(n_frames, dtype=dtype) * float(globals().get("DT_SEC_FALLBACK", 0.5))


## Cell 3: Auto-Discover Scans & Load Selected `.cif` Reference Phases
**What this cell does:**
1. Finds your `.poni` file, thin-film `LaB6*_master.h5` calibration scan, and all `<SampleID>_anneal_master.h5` + `<SampleID>_shear_master.h5` pairs.
2. Loads **any number of `.cif` files (0, 1, 3, 4, 5, or more)** from `OPEN_CIF_PICKER`, `CIF_FILES`, or auto-discovery in `DATA_DIR`.
3. Computes theoretical Bragg peak positions ($q_{hkl}$) and relative intensities for every selected `.cif`.  

**What to check:** Check the two summary tables below to confirm all sample scans and `.cif` phases loaded properly.


In [ ]:
def _is_valid_data_file(path, exclude_dir_name="outputs"):
    """Ignore macOS AppleDouble sidecars (._*), hidden files/folders (.*), and outputs/."""
    p = Path(path)
    if p.name.startswith(".") or p.name.startswith("._"):
        return False
    for part in p.parts:
        if part == exclude_dir_name or (part not in (".", "..") and part.startswith(".")):
            return False
    return True


def select_and_load_cifs(data_dir, explicit_cifs=None, open_picker=False, q_range=(0.4, 2.2)):
    selected_paths = []

    if open_picker:
        try:
            import tkinter as tk
            from tkinter import filedialog

            root = tk.Tk()
            root.withdraw()
            root.attributes("-topmost", True)
            picked = filedialog.askopenfilenames(
                title="Select 1 to 5+ Reference .cif Files",
                initialdir=str(Path(data_dir).resolve()),
                filetypes=[("CIF Crystallographic Files", "*.cif"), ("All Files", "*.*")],
            )
            root.destroy()
            selected_paths = [Path(p) for p in picked if _is_valid_data_file(p, OUTPUT_DIR.name)]
        except Exception as exc:
            print(f"File picker unavailable in this environment ({exc}); falling back to CIF_FILES / auto-discovery.")

    if not selected_paths and explicit_cifs:
        resolved_cifs = []
        for p in explicit_cifs:
            cp = Path(p)
            if not cp.exists() and (Path(data_dir) / cp).exists():
                cp = Path(data_dir) / cp
            if _is_valid_data_file(cp, OUTPUT_DIR.name):
                resolved_cifs.append(cp)
        selected_paths = resolved_cifs

    if not selected_paths:
        selected_paths = sorted(
            p for p in Path(data_dir).rglob("*.cif") if _is_valid_data_file(p, OUTPUT_DIR.name)
        )

    cif_refs = []
    cif_rows = []
    for p in selected_paths:
        if not p.exists():
            print(f"Warning: CIF file not found -> {p}")
            continue
        try:
            parsed = parse_cif_file(p, q_min=q_range[0], q_max=q_range[1], min_rel_intensity=1.0)
            cif_refs.append(parsed)
            top_peaks = sorted(parsed["peaks"], key=lambda x: -x["I_rel"])[:4]
            top_str = ", ".join(f"{pk['hkl']}@{pk['q']:.3f}" for pk in top_peaks)
            a, b, c = parsed["cell"]
            cif_rows.append({
                "CIF Phase": parsed["label"],
                "Lattice (a, b, c Å)": f"{a:.2f}, {b:.2f}, {c:.2f}",
                "Peaks in Range": len(parsed["peaks"]),
                "Strongest Reflections (q in Å^-1)": top_str or "None in q-range",
            })
        except Exception as exc:
            print(f"Warning: Could not parse {p.name}: {exc}")

    if cif_rows:
        print(f"Loaded {len(cif_refs)} reference .cif structure(s):")
        display(pd.DataFrame(cif_rows))
    else:
        print("No .cif reference files selected/found (plots will run without CIF overlays).")

    return cif_refs


def discover_experiment_files(data_dir, poni_override=None, mask_override=None):
    data_dir = Path(data_dir)

    if poni_override is not None:
        poni_file = Path(poni_override)
        if not poni_file.exists() and (data_dir / poni_file).exists():
            poni_file = data_dir / poni_file
        if not poni_file.exists():
            raise FileNotFoundError(f"PONI_FILE_OVERRIDE not found: {poni_file.resolve()}")
    else:
        poni_candidates = sorted(
            p for p in data_dir.rglob("*.poni") if _is_valid_data_file(p, OUTPUT_DIR.name)
        )
        valid_ponis = []
        for p in poni_candidates:
            txt = p.read_text(encoding="utf-8", errors="ignore")
            if "distance:" in txt.lower():
                valid_ponis.append(p)
        if not valid_ponis:
            raise FileNotFoundError(f"No valid non-hidden .poni file found in {data_dir.resolve()}")
        poni_file = valid_ponis[0]

    lab6_files = sorted(
        p for p in data_dir.rglob("*_master.h5")
        if _is_valid_data_file(p, OUTPUT_DIR.name) and "lab6" in p.name.lower()
    )
    if not lab6_files:
        lab6_files = sorted(
            p for p in data_dir.rglob("*.h5")
            if _is_valid_data_file(p, OUTPUT_DIR.name) and "lab6" in p.name.lower()
        )
    if not lab6_files:
        raise FileNotFoundError(f"No *LaB6*.h5 file found in {data_dir.resolve()}")
    lab6_master = lab6_files[0]

    anneal_files = sorted(
        p for p in data_dir.rglob("*_anneal_master.h5") if _is_valid_data_file(p, OUTPUT_DIR.name)
    )
    if not anneal_files:
        raise FileNotFoundError(f"No *_anneal_master.h5 files found in {data_dir.resolve()}")

    conditions = {}
    summary_rows = []
    for ann_path in anneal_files:
        sample_id = ann_path.name.replace("_anneal_master.h5", "")
        shear_path = ann_path.parent / f"{sample_id}_shear_master.h5"
        csv_matches = sorted(
            p for p in ann_path.parent.glob(f"*{sample_id}*.csv") if _is_valid_data_file(p, OUTPUT_DIR.name)
        )
        log_csv = csv_matches[0] if csv_matches else None

        temp_match = re.search(r"(\d+(?:\.\d+)?)\s*C\b", sample_id, re.IGNORECASE)
        parsed_temp = float(temp_match.group(1)) if temp_match else None

        conditions[sample_id] = {
            "sample_id": sample_id,
            "anneal_master_h5": ann_path,
            "shear_master_h5": shear_path if shear_path.exists() else None,
            "process_log_csv": log_csv,
            "anneal_temp_C": parsed_temp,
        }
        summary_rows.append({
            "Sample ID": sample_id,
            "Anneal Master": ann_path.name,
            "Shear Master": shear_path.name if shear_path.exists() else "Not found",
            "Process Log": log_csv.name if log_csv else "None",
        })

    resolved_mask = None
    if mask_override is False:
        resolved_mask = None
    elif mask_override is not None:
        m_cand = Path(mask_override)
        if not m_cand.exists() and (data_dir / m_cand).exists():
            m_cand = data_dir / m_cand
        if not m_cand.exists():
            raise FileNotFoundError(f"MASK_FILE not found: {m_cand}")
        resolved_mask = m_cand
    else:
        edf_candidates = sorted(
            p for p in data_dir.rglob("*.edf") if _is_valid_data_file(p, OUTPUT_DIR.name)
        )
        mask_edfs = [p for p in edf_candidates if "mask" in p.name.lower()]
        if mask_edfs:
            resolved_mask = mask_edfs[0]
        elif edf_candidates:
            resolved_mask = edf_candidates[0]

    print(f"PONI file   : {poni_file.name}")
    print(f"Mask file   : {resolved_mask.name if resolved_mask else 'None (Eiger auto-mask only)'}")
    print(f"LaB6 master : {lab6_master.name}\n")
    display(pd.DataFrame(summary_rows))
    return poni_file, lab6_master, resolved_mask, conditions


PONI_FILE, LAB6_MASTER_H5, ACTIVE_MASK_FILE, CONDITIONS = discover_experiment_files(
    DATA_DIR, poni_override=PONI_FILE_OVERRIDE, mask_override=MASK_FILE
)
CIF_REFERENCES = select_and_load_cifs(
    DATA_DIR,
    explicit_cifs=CIF_FILES,
    open_picker=OPEN_CIF_PICKER,
    q_range=Q_PLOT_RANGE,
)


## Cell 4: Inspect Calibrant (Raw 2D, 2D Cake, 1D Integration) & Fit Instrumental Broadening
**What this cell does:**
1. Averages your thin-film NIST $\text{LaB}_6$ calibration scan and displays **three individual diagnostic images** inline in the cell output:
   * **1. Raw 2D Detector Image** (with active mask and calibrated beam center),
   * **2. 2D Cake $I(q, \chi)$** (with theoretical $\text{LaB}_6$ Bragg lines),
   * **3. 1D Integrated $I(q)$ Pattern** (also saved as tabular CSV `outputs/lab6_integrated_1d.csv`).
2. Fits **every visible $\text{LaB}_6$ reflection** (`(100)`, `(110)`, `(111)`, `(200)`, etc.) with a single-peak TCH pseudo-Voigt profile and evaluates the instrumental Gaussian ($H_{G,\text{inst}}$) and Lorentzian ($H_{L,\text{inst}}$) widths at $q \approx 0.996\text{ \AA}^{-1}$ for Cell 6.


In [ ]:
def _half_max_fwhm(qs, ys, i_pk, base):
    """FWHM of the peak at index i_pk by linear interpolation of the half-maximum crossings (NaN if not found)."""
    half = base + 0.5 * (ys[i_pk] - base)
    def cross(idx_iter):
        prev = i_pk
        for i in idx_iter:
            if ys[i] <= half:
                f = (ys[prev] - half) / max(ys[prev] - ys[i], 1e-12)
                return qs[prev] + f * (qs[i] - qs[prev])
            prev = i
        return np.nan
    ql = cross(range(i_pk - 1, -1, -1)); qr = cross(range(i_pk + 1, len(qs)))
    return float(qr - ql) if np.isfinite(ql) and np.isfinite(qr) else np.nan


def fit_lab6_peak(q, I, q_theo, half_window=0.08, tol=0.06, min_window=0.03, n_fwhm=6.0):
    """Robust single-peak TCH fit of one LaB6 reflection.
    1) estimate background and FWHM straight from the data,
    2) refit in a window of +-n_fwhm*FWHM around the peak (so the window follows the real peak width and
       neighbouring weak lines / flat tails do not drive the fit),
    3) try several Gaussian/Lorentzian starting mixes and keep the lowest-residual solution.
    -> dict or None"""
    sel = (q >= q_theo - half_window) & (q <= q_theo + half_window) & np.isfinite(I)
    qs, ys = q[sel], I[sel]
    if len(qs) < 10:
        return None
    core = (qs >= q_theo - tol) & (qs <= q_theo + tol)
    if not np.any(core):
        return None
    k = np.where(core)[0]
    ipk = int(k[np.argmax(ys[k])])
    e = max(len(ys) // 10, 3)
    bg0 = float(np.median(np.r_[ys[:e], ys[-e:]]))
    bg_std = max(float(np.std(np.r_[ys[:e], ys[-e:]])), 1e-6)
    if ys[ipk] - bg0 < 3.0 * bg_std:
        return None
    H_est = _half_max_fwhm(qs, ys, ipk, bg0)
    if not np.isfinite(H_est) or H_est <= 0:
        H_est = 0.012
    H_est = float(np.clip(H_est, 0.003, 0.08))
    q_c = float(qs[ipk])
    hw = float(np.clip(n_fwhm * H_est, min_window, half_window))
    w = (q >= q_c - hw) & (q <= q_c + hw) & np.isfinite(I)
    qw, yw = q[w], I[w]
    if len(qw) < 10:
        return None
    e2 = max(len(yw) // 8, 2)
    bg = float(np.median(np.r_[yw[:e2], yw[-e2:]]))
    pk = float(np.max(yw) - bg)
    lo = [0.0, q_theo - tol, 1e-4, 1e-4, -np.inf, -np.inf]
    hi = [np.inf, q_theo + tol, 0.1, 0.1, np.inf, np.inf]
    best = None
    for fg, fl in ((0.9, 0.2), (0.6, 0.6), (0.3, 0.9), (0.15, 1.0)):
        p0 = [max(pk, 1e-3) * H_est * 1.1, float(np.clip(q_c, q_theo - tol + 1e-4, q_theo + tol - 1e-4)),
              fg * H_est, fl * H_est, 0.0, bg]
        try:
            popt, pcov = curve_fit(free_peak_model, qw, yw, p0=p0, bounds=(lo, hi), maxfev=40000)
        except (RuntimeError, ValueError):
            continue
        sse = float(np.sum((yw - free_peak_model(qw, *popt)) ** 2))
        if best is None or sse < best[0]:
            best = (sse, popt, pcov)
    if best is None:
        return None
    sse, popt, pcov = best
    fit = free_peak_model(qw, *popt)
    sst = float(np.sum((yw - np.mean(yw)) ** 2))
    H_tot, eta = tch_fwhm_and_eta(popt[2], popt[3])
    near = np.abs(qw - popt[1]) <= H_tot
    rms_pct = 100.0 * float(np.sqrt(np.mean((yw[near] - fit[near]) ** 2))) / max(pk, 1e-9) if np.any(near) else np.nan
    return {"popt": popt, "perr": np.sqrt(np.diag(pcov)), "q_sub": qw, "I_sub": yw, "I_fit": fit,
            "r2": 1.0 - sse / max(sst, 1e-12), "rms_pct": rms_pct, "window": hw, "H_est": H_est}


def calibrate_lab6_automatically(
    lab6_master_h5,
    poni_file,
    eval_q=0.996,
    npt_rad=3000,
    azimuth_range=None,
    fit_half_window=0.08,
    peak_center_tol=0.06,
    pixel_size_override=None,
    energy_kev_override=None,
    mask_file=None,
):
    """
    Integrate the thin-film LaB6 standard pattern, display individual inline plots for:
      1) Raw 2D detector image (with active mask & beam center),
      2) 2D cake I(q, chi),
      3) 1D integrated I(q) curve,
    then fit all visible LaB6 Bragg peaks to evaluate instrumental broadening at `eval_q`.
    """
    with h5py.File(lab6_master_h5, "r") as f:
        data_path = find_eiger_data_path(f)
        stack = f[data_path][:]
        # Exclude any blank/shutter-closed frames if present
        valid_frames = [
            fr for fr in stack
            if np.nanmax(np.where((fr >= 0) & (fr < 65535), fr, 0)) > 0
        ]
        img_avg = np.mean(valid_frames if valid_frames else stack, axis=0)
        n_avg_frames = len(valid_frames) if valid_frames else len(stack)

    ai = load_azimuthal_integrator(
        poni_file,
        reference_h5=lab6_master_h5,
        img_shape=img_avg.shape,
        pixel_size_override=pixel_size_override,
        energy_kev_override=energy_kev_override,
    )

    mask = build_combined_mask(img_avg, mask_file=mask_file, verbose=True)

    # 1D azimuthal integration
    res1d = ai.integrate1d(
        img_avg,
        npt=npt_rad,
        mask=mask,
        azimuth_range=azimuth_range,
        unit="q_A^-1",
    )
    q = np.asarray(res1d.radial, dtype=float)
    I = np.asarray(res1d.intensity, dtype=float)
    counts = getattr(res1d, "count", None)
    if counts is not None:
        I = np.where(np.asarray(counts) > 0, I, np.nan)

    # 2D cake integration I(q, chi)
    res2d = ai.integrate2d(
        img_avg,
        npt_rad=npt_rad,
        npt_azim=360,
        mask=mask,
        azimuth_range=azimuth_range,
        unit=("q_A^-1", "chi_deg"),
    )
    q_cake = np.asarray(res2d.radial, dtype=float)
    chi_cake = np.asarray(res2d.azimuthal, dtype=float)
    I_cake = np.asarray(res2d.intensity, dtype=float)

    valid_mask = np.isfinite(q) & np.isfinite(I)
    q_min_det = float(np.min(q[valid_mask])) if np.any(valid_mask) else float(np.nanmin(q))
    q_max_det = float(np.max(q[valid_mask])) if np.any(valid_mask) else float(np.nanmax(q))

    # Save & display the integrated 1D LaB6 standard data table
    df_lab6_1d = pd.DataFrame({
        "q_A_inv": q,
        "intensity_au": I,
        "pixel_count": np.asarray(counts) if counts is not None else np.nan,
    })
    lab6_1d_csv = OUTPUT_DIR / "lab6_integrated_1d.csv"
    df_lab6_1d.to_csv(lab6_1d_csv, index=False)

    print(
        f"Active PONI : {Path(poni_file).name} | dist={ai.dist*1e3:.2f} mm | "
        f"poni1={ai.poni1*1e3:.2f} mm, poni2={ai.poni2*1e3:.2f} mm"
    )
    print(
        f"Integrated 1D LaB6 Standard -> {int(np.sum(valid_mask))}/{len(q)} active bins | "
        f"q-range=[{q_min_det:.4f}, {q_max_det:.4f}] A^-1 | "
        f"I-range=[{np.nanmin(I):.2f}, {np.nanmax(I):.2f}] a.u."
    )
    print(f"Saved full 1D standard data  -> {lab6_1d_csv}")
    display(df_lab6_1d.dropna(subset=["intensity_au"]).head(10))

    a_lab6 = 4.15689  # NIST SRM 660c LaB6 lattice parameter (A)
    hkls = [
        (1, 0, 0), (1, 1, 0), (1, 1, 1), (2, 0, 0),
        (2, 1, 0), (2, 1, 1), (2, 2, 0), (3, 0, 0), (3, 1, 0),
    ]
    theoretical_peaks = {
        f"({h}{k}{l})": (2 * np.pi / a_lab6) * np.sqrt(h**2 + k**2 + l**2)
        for h, k, l in hkls
    }

    # Resolve direct-beam pixel coordinates (x_px, y_px) for overlay on the raw 2D detector image
    try:
        fit2d_geom = ai.getFit2D()
        beam_x_px = float(fit2d_geom["centerX"])
        beam_y_px = float(fit2d_geom["centerY"])
    except Exception:
        beam_x_px = float(ai.poni2 / ai.pixel2)
        beam_y_px = float(ai.poni1 / ai.pixel1)

    cmap_2d = plt.get_cmap("inferno").copy()
    cmap_2d.set_bad("slategray")

    # --------------------------------------------------------------------------
    # INDIVIDUAL IMAGE 1: Raw 2D Detector Image (with mask & beam center)
    # --------------------------------------------------------------------------
    # Show the data exactly as integrated: mean counts per pixel, with ONLY the real mask in gray.
    # (Zero-count pixels that are NOT masked stay visible as the darkest colour instead of being hidden in gray.)
    scale = str(globals().get("RAW_IMAGE_SCALE", "log")).lower()          # "log" (default) or "linear"
    is_masked = (mask != 0) | ~np.isfinite(img_avg)
    shown = np.where(is_masked, np.nan, img_avg)
    if scale == "linear":
        raw_plot = shown
        vmin_raw = 0.0
        vmax_raw = float(np.nanpercentile(shown, 99.5)) if np.any(np.isfinite(shown)) else 1.0
        cbar_label = "mean counts per frame (linear)"
    else:
        raw_plot = np.log10(np.maximum(shown, 1e-3))
        vmin_raw = 0.0                                                      # 1 count per frame; lower values show as the darkest colour
        vmax_raw = float(np.nanpercentile(raw_plot, 99.9)) if np.any(np.isfinite(raw_plot)) else 1.0
        cbar_label = "$\\log_{10}$ mean counts per frame"
    raw_log = raw_plot
    n_masked = int(np.sum(is_masked))

    fig_2d, ax_2d = plt.subplots(figsize=(7.6, 6.2))
    im_raw = ax_2d.imshow(
        raw_log,
        origin="lower",
        cmap=cmap_2d,
        vmin=vmin_raw,
        vmax=vmax_raw,
        aspect="equal",
    )
    ax_2d.plot(
        beam_x_px,
        beam_y_px,
        marker="+",
        color="cyan",
        ms=12,
        mew=1.8,
        label=f"Beam Center ({beam_x_px:.1f}, {beam_y_px:.1f} px)",
    )
    ax_2d.set_xlim(0, img_avg.shape[1] - 1)
    ax_2d.set_ylim(0, img_avg.shape[0] - 1)
    ax_2d.set_xlabel("Detector X (pixels)")
    ax_2d.set_ylabel("Detector Y (pixels)")
    ax_2d.set_title(f"1. Raw 2D LaB6 Detector Image — mean of {n_avg_frames} frame(s), mask applied\n"
                    f"{Path(lab6_master_h5).name} | masked (gray): {n_masked:,} px = {100.0 * n_masked / img_avg.size:.1f} %")
    ax_2d.legend(loc="upper center", bbox_to_anchor=(0.5, -0.11), fontsize=8.5, frameon=False)
    plt.colorbar(im_raw, ax=ax_2d, fraction=0.046, pad=0.04, label=cbar_label)
    plt.tight_layout()
    plt.show()

    # --------------------------------------------------------------------------
    # INDIVIDUAL IMAGE 2: 2D Cake I(q, chi)
    # --------------------------------------------------------------------------
    cake_disp = np.where(np.isfinite(I_cake) & (I_cake > 0), I_cake, np.nan)
    cake_log = np.log10(cake_disp)
    if np.any(np.isfinite(cake_log)):
        vmin_cake, vmax_cake = np.nanpercentile(cake_log, [5.0, 99.8])
    else:
        vmin_cake, vmax_cake = 0.0, 1.0

    fig_cake, ax_cake = plt.subplots(figsize=(8.4, 4.2))
    pcm_cake = ax_cake.pcolormesh(
        q_cake,
        chi_cake,
        cake_log,
        shading="auto",
        cmap=cmap_2d,
        vmin=vmin_cake,
        vmax=vmax_cake,
    )
    chi_top = float(np.max(chi_cake))
    for lbl, qt in theoretical_peaks.items():
        if q_min_det <= qt <= q_max_det:
            ax_cake.axvline(qt, color="cyan", ls=":", lw=1.0, alpha=0.85)
            ax_cake.text(
                qt,
                chi_top - 0.03 * (chi_top - float(np.min(chi_cake))),
                f" {lbl}",
                color="cyan",
                rotation=90,
                va="top",
                ha="left",
                fontsize=7.5,
                fontweight="bold",
            )
    ax_cake.set_xlim(q_min_det, q_max_det)
    ax_cake.set_xlabel("$q$ ($\\AA^{-1}$)")
    ax_cake.set_ylabel("Azimuthal Angle $\\chi$ (deg)")
    ax_cake.set_title(f"2. LaB6 2D Cake $I(q, \\chi)$ ({Path(lab6_master_h5).name})")
    plt.colorbar(pcm_cake, ax=ax_cake, fraction=0.03, pad=0.03, label="$\\log_{10} I$ (a.u.)")
    plt.tight_layout()
    plt.show()

    # --------------------------------------------------------------------------
    # INDIVIDUAL IMAGE 3: 1D Integrated I(q) Pattern
    # --------------------------------------------------------------------------
    fig_raw, ax_raw = plt.subplots(figsize=(8.4, 3.2))
    ax_raw.plot(q, I, "k-", lw=1.2, label="Integrated 1D LaB6 Standard")
    y_top = float(np.nanmax(I[valid_mask])) if np.any(valid_mask) else 1.0
    for lbl, qt in theoretical_peaks.items():
        if q_min_det <= qt <= q_max_det:
            ax_raw.axvline(qt, color="tab:blue", ls=":", lw=1.0, alpha=0.85)
            ax_raw.text(qt, y_top * 0.94, f" {lbl}", color="tab:blue", rotation=90, va="top", ha="left", fontsize=7.5)
    ax_raw.set_xlim(q_min_det, q_max_det)
    ax_raw.set_xlabel("$q$ ($\\AA^{-1}$)")
    ax_raw.set_ylabel("Intensity (a.u.)")
    ax_raw.set_title(f"3. Integrated 1D LaB6 Standard Pattern ({Path(lab6_master_h5).name})")
    ax_raw.legend(frameon=False, fontsize=8, loc="upper right")
    plt.tight_layout()
    plt.show()

    # --------------------------------------------------------------------------
    # Fit all visible LaB6 Bragg reflections for instrumental broadening
    # --------------------------------------------------------------------------
    fit_results = []
    for label, q_theo in theoretical_peaks.items():
        if not (q_min_det + 0.03 < q_theo < q_max_det - 0.03):
            continue
        r = fit_lab6_peak(q, I, q_theo, half_window=fit_half_window, tol=peak_center_tol)
        if r is None:
            continue
        A_fit, q0_fit, HG_fit, HL_fit, slope_fit, int_fit = r["popt"]
        perr = r["perr"]
        H_tot, eta = tch_fwhm_and_eta(HG_fit, HL_fit)
        fit_results.append({
            "hkl": label,
            "q_theo": float(q_theo),
            "q0": float(q0_fit),
            "H_G": float(HG_fit),
            "H_L": float(HL_fit),
            "H_total": float(H_tot),
            "eta": float(eta),
            "H_G_err": float(perr[2]),
            "H_L_err": float(perr[3]),
            "r2": float(r["r2"]),
            "rms_resid_pct": float(r["rms_pct"]),
            "fit_window": float(r["window"]),
            "q_sub": r["q_sub"],
            "I_sub": r["I_sub"],
            "I_fit": r["I_fit"],
        })

    print("LaB6 peak fits (residual = rms of data-fit within one FWHM of the peak, as % of peak height):")
    for r_ in fit_results:
        print(f"  {r_['hkl']}: q0-q_theo = {r_['q0'] - r_['q_theo']:+.5f} A^-1 | FWHM = {r_['H_total']:.5f} | eta = {r_['eta']:.2f} | "
              f"R2 = {r_['r2']:.4f} | residual = {r_['rms_resid_pct']:.1f} % | fit window = +-{r_['fit_window']:.3f}")
    _max_rms = globals().get("LAB6_MAX_RMS_PCT", None)
    if _max_rms is not None and len(fit_results) > 2:
        _good = [r_ for r_ in fit_results if r_["rms_resid_pct"] <= _max_rms]
        if len(_good) >= 2 and len(_good) < len(fit_results):
            print(f"  LAB6_MAX_RMS_PCT = {_max_rms}: excluded from the broadening extrapolation:",
                  [r_["hkl"] for r_ in fit_results if r_ not in _good])
            _used = _good
        else:
            _used = fit_results
    else:
        _used = fit_results

    if not fit_results:
        all_ponis = [p.name for p in sorted(DATA_DIR.rglob("*.poni"))]
        raise ValueError(
            f"No LaB6 peaks could be fitted in q=[{q_min_det:.3f}, {q_max_det:.3f}] A^-1. "
            f"PONI files found in DATA_DIR: {all_ponis}. "
            "Check the raw 2D, 2D cake, and 1D LaB6 plots above to verify the PONI calibration and q-range."
        )

    q_peaks = np.array([r["q0"] for r in _used])
    HG_peaks = np.array([r["H_G"] for r in _used])
    HL_peaks = np.array([r["H_L"] for r in _used])

    if len(_used) >= 2:
        HG_inst = float(np.clip(np.polyval(np.polyfit(q_peaks, HG_peaks, 1), eval_q), 1e-4, None))
        HL_inst = float(np.clip(np.polyval(np.polyfit(q_peaks, HL_peaks, 1), eval_q), 1e-4, None))
    else:
        HG_inst = float(HG_peaks[0])
        HL_inst = float(HL_peaks[0])

    H_tot_inst, eta_inst = tch_fwhm_and_eta(HG_inst, HL_inst)

    calib_summary = {
        "poni_file": str(poni_file),
        "mask_file": str(mask_file) if mask_file else None,
        "lab6_master_h5": str(lab6_master_h5),
        "lab6_1d_csv": str(lab6_1d_csv),
        "pixel_size_override": pixel_size_override,
        "beam_energy_kev": energy_kev_override,
        "wavelength_A": float(ai.wavelength * 1e10),
        "npt_rad": npt_rad,
        "azimuth_range": azimuth_range,
        "q_target_A_inv": eval_q,
        "H_G_inst": HG_inst,
        "H_L_inst": HL_inst,
        "H_total_inst": float(H_tot_inst),
        "eta_inst": float(eta_inst),
        "fitted_peaks": [
            {k: v for k, v in r.items() if k not in ("q_sub", "I_sub", "I_fit")}
            for r in fit_results
        ],
    }

    with open(OUTPUT_DIR / "inst_broadening.json", "w") as f:
        json.dump(calib_summary, f, indent=2)

    print(f"Fitted {len(fit_results)} LaB6 peak(s): {[r['hkl'] for r in fit_results]}")
    print(f"Thin-Film Instrumental Broadening evaluated at q = {eval_q:.3f} A^-1:")
    print(f"  H_G_inst = {HG_inst:.5f} A^-1 | H_L_inst = {HL_inst:.5f} A^-1 | Total FWHM_inst = {H_tot_inst:.5f} A^-1\n")

    n = len(fit_results)
    fig, axes = plt.subplots(2, n, figsize=(4.3 * n, 4.6), gridspec_kw={"height_ratios": [3, 1]}, sharex="col")
    if n == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    for idx, r in enumerate(fit_results):
        axes[0, idx].plot(r["q_sub"], r["I_sub"], "ko", ms=3.5, label="LaB6 Data")
        axes[0, idx].plot(r["q_sub"], r["I_fit"], "r-", lw=1.8, label="TCH Fit")
        axes[0, idx].set_title(f"LaB6 {r['hkl']} ($q_0$={r['q0']:.4f} $\\AA^{{-1}}$)")
        axes[0, idx].set_ylabel("Intensity (a.u.)" if idx == 0 else "")
        axes[0, idx].legend(frameon=False, fontsize=8)

        axes[1, idx].plot(r["q_sub"], r["I_sub"] - r["I_fit"], "b-", lw=1)
        axes[1, idx].axhline(0, color="gray", ls="--", lw=0.8)
        axes[1, idx].set_xlabel("$q$ ($\\AA^{-1}$)")
        axes[1, idx].set_ylabel("Residual" if idx == 0 else "")

    plt.tight_layout()
    plt.show()

    # Attach 1D and 2D cake arrays in memory for easy interactive inspection
    calib_summary["q_A_inv"] = q
    calib_summary["I_raw"] = I
    calib_summary["cake_q"] = q_cake
    calib_summary["cake_chi_deg"] = chi_cake
    calib_summary["cake_I"] = I_cake
    return calib_summary


CALIB_DATA = calibrate_lab6_automatically(
    lab6_master_h5=LAB6_MASTER_H5,
    poni_file=PONI_FILE,
    eval_q=Q_TARGET_100,
    npt_rad=NPT_RADIAL,
    azimuth_range=AZIMUTH_RANGE,
    pixel_size_override=PIXEL_SIZE_M,
    energy_kev_override=BEAM_ENERGY_KEV,
    mask_file=ACTIVE_MASK_FILE,
)


## Cell 5: Reduce All Annealing Scans & Auto-Detect Beam-Damage Cutoff
**What this cell does:**
1. Integrates every frame of each sample's `*_anneal_master.h5` into 1D $I(q, t)$ using `pyFAI` (and caches the result in `outputs/cache/` so re-runs are instant).
2. Reads the companion `*_shear_master.h5` duration for timeline reference without wasting time integrating featureless shearing frames.
3. Automatically locates the **$\alpha$-perovskite $(100)$ intensity maximum (`frame_max`)** inside `Q_100_SEARCH_WINDOW` and flags subsequent frames as excluded beam-damage frames.
4. Extracts 1D $I(q)$, 2D $(q_r, q_z)$, and azimuthal $I(\chi)$ snapshots at `frame_max`.  

**What to check:** Confirm the vertical dashed cutoff lines sit at the peak maximum of each sample's $(100)$ intensity trace.


In [ ]:
def process_all_samples(conditions, calib_data, q_100_window=(0.96, 1.05), buffer_frames=15):
    npt_rad = calib_data["npt_rad"]
    azimuth_range = calib_data["azimuth_range"]
    cache_dir = OUTPUT_DIR / "cache"
    cache_dir.mkdir(exist_ok=True)

    ai = None
    reduced = {}
    for sample_id, cfg in conditions.items():
        if is_excluded_sample(sample_id):      # shared rule: excluded names / not on the sample table
            continue
        shear_dur = np.nan
        if cfg["shear_master_h5"] and Path(cfg["shear_master_h5"]).exists():
            with h5py.File(cfg["shear_master_h5"], "r") as fs:
                spath = find_eiger_data_path(fs)
                n_sh = fs[spath].shape[0]
                t_sh = get_eiger_timestamps(fs, n_sh, DT_SEC_FALLBACK)
                shear_dur = float(t_sh[-1] - t_sh[0]) if len(t_sh) > 1 else float(n_sh * DT_SEC_FALLBACK)

        cache_file = cache_dir / f"{sample_id}_reduced.npz"
        with h5py.File(cfg["anneal_master_h5"], "r") as fa:
            apath = find_eiger_data_path(fa)
            dset = fa[apath]
            n_frames = dset.shape[0]
            ts = get_eiger_timestamps(fa, n_frames, DT_SEC_FALLBACK)
            frame0 = dset[0]

            if ai is None:
                ai = load_azimuthal_integrator(
                    calib_data["poni_file"],
                    reference_h5=cfg["anneal_master_h5"],
                    img_shape=frame0.shape,
                    pixel_size_override=calib_data.get("pixel_size_override"),
                    energy_kev_override=calib_data.get("beam_energy_kev"),
                )

            mask_file = calib_data.get("mask_file")
            if mask_file:
                _mp = Path(mask_file).resolve()
                mask_sig = f"{_mp}|{_mp.stat().st_size}|{int(_mp.stat().st_mtime)}"   # changes when the mask is edited
            else:
                mask_sig = "none"
            use_cache = False
            cache_has_time = False
            if cache_file.exists() and not FORCE_RECOMPUTE:
                try:
                    with np.load(cache_file, allow_pickle=False) as cached:      # 'with' closes the file handle (Windows cannot overwrite an open file)
                        cached_sig = str(cached["mask_signature"]) if "mask_signature" in cached else "missing"
                        cached_npt = len(cached["q"]) if "q" in cached else -1
                        if cached_sig == mask_sig and cached_npt == npt_rad:
                            cache_has_time = "time_s" in cached
                            q_grid = cached["q"]
                            I_matrix = cached["I_matrix"]
                            use_cache = True
                        else:
                            print(f"[{sample_id}] Mask or radial bin count changed -> re-integrating 1D frames...")
                except Exception as _ce:      # truncated / corrupt cache file (e.g. a run was interrupted while saving)
                    print(f"[{sample_id}] Cache file unreadable ({type(_ce).__name__}) -> deleting it and re-integrating...")
                    try:
                        cache_file.unlink()
                    except OSError:
                        pass

            if not use_cache:
                mask = build_combined_mask(frame0, mask_file=mask_file, verbose=False)
                # Warm up the CSR lookup table on frame 0, then integrate in RAM-safe batches
                fast_method = ("no", "csr", "cython")
                res0 = ai.integrate1d(
                    frame0,
                    npt=npt_rad,
                    mask=mask,
                    azimuth_range=azimuth_range,
                    unit="q_A^-1",
                    method=fast_method,
                    safe=True,
                )
                q_grid = res0.radial
                I_matrix = np.empty((n_frames, len(q_grid)), dtype=np.float32)
                I_matrix[0] = res0.intensity

                batch_size = 32  # Reads 32 frames per HDF5 chunk (~120-500 MB RAM) to cut drive I/O overhead
                for start in range(1, n_frames, batch_size):
                    end = min(start + batch_size, n_frames)
                    block = dset[start:end]
                    for offset in range(end - start):
                        res1d = ai.integrate1d(
                            block[offset],
                            npt=npt_rad,
                            mask=mask,
                            azimuth_range=azimuth_range,
                            unit="q_A^-1",
                            method=fast_method,
                            safe=False,
                        )
                        I_matrix[start + offset] = res1d.intensity
                _tmp = cache_file.with_name(cache_file.stem + "_tmp.npz")      # write then rename, so an interrupted save cannot leave a half-written cache
                np.savez_compressed(_tmp, q=q_grid, I_matrix=I_matrix, mask_signature=mask_sig, time_s=ts)
                try:
                    _tmp.replace(cache_file)
                except OSError:                                                # some drives refuse to overwrite -> remove the old file, then rename; last resort: save directly
                    try:
                        cache_file.unlink()
                        _tmp.rename(cache_file)
                    except OSError:
                        np.savez_compressed(cache_file, q=q_grid, I_matrix=I_matrix, mask_signature=mask_sig, time_s=ts)
                        try:
                            _tmp.unlink()
                        except OSError:
                            pass

            if use_cache and not cache_has_time:      # older cache without the time axis: add it (no re-integration needed)
                try:
                    np.savez_compressed(cache_file, q=q_grid, I_matrix=I_matrix, mask_signature=mask_sig, time_s=ts)
                except OSError as _e:
                    print(f"[{sample_id}] could not add time_s to the cache ({_e}); later cells will fall back to DT_SEC_FALLBACK")

            roi = (q_grid >= q_100_window[0]) & (q_grid <= q_100_window[1])
            I_roi = I_matrix[:, roi]
            baseline = 0.5 * (I_roi[:, 0] + I_roi[:, -1])
            net_100 = np.max(I_roi - baseline[:, None], axis=1)

            frame_max = find_alpha100_frame_max(q_grid, I_matrix, t_arr=ts, t_cutoff=get_sample_cutoff(sample_id))   # same t_max rule as Cells 5A-7
            frame_plot_end = min(n_frames - 1, frame_max + buffer_frames)

            mask = build_combined_mask(dset[frame_max], mask_file=mask_file, verbose=False)
            res2d = ai.integrate2d(
                dset[frame_max],
                npt_rad=600,
                npt_azim=180,
                mask=mask,
                unit=("q_A^-1", "chi_deg"),
            )
            q2d, chi2d, I2d = res2d.radial, res2d.azimuthal, res2d.intensity
            ring_mask = (q2d >= q_100_window[0]) & (q2d <= q_100_window[1])
            I_chi_100 = np.nanmean(I2d[:, ring_mask], axis=1)

            Q_mesh, CHI_mesh = np.meshgrid(q2d, np.deg2rad(chi2d))
            qr_map = Q_mesh * np.cos(CHI_mesh)
            qz_map = Q_mesh * np.sin(CHI_mesh)

        reduced[sample_id] = {
            "sample_id": sample_id,
            "time_s": ts,
            "shear_duration_s": shear_dur,
            "anneal_temp_C": cfg["anneal_temp_C"],
            "process_log_csv": cfg["process_log_csv"],
            "q": q_grid,
            "I_matrix": I_matrix,
            "I_at_max": I_matrix[frame_max],
            "net_100": net_100,
            "frame_max": frame_max,
            "frame_plot_end": frame_plot_end,
            "chi_deg": chi2d,
            "I_chi_100": I_chi_100,
            "qr_map": qr_map,
            "qz_map": qz_map,
            "I2d_map": I2d,
        }
        print(
            f"[{sample_id}] Shear={shear_dur:.1f}s | Anneal frames={n_frames} | "
            f"(100) Max at frame {frame_max} (t = {ts[frame_max]:.1f} s) | frame spacing {float(np.median(np.diff(ts))) if len(ts) > 1 else float('nan'):.3f} s"
        )

    fig, ax = plt.subplots(figsize=(11.0, 4.2), constrained_layout=True)
    for sample_id, r in reduced.items():
        norm_trace = r["net_100"] / max(np.max(r["net_100"]), 1e-6)
        line, = ax.plot(
            r["time_s"], norm_trace, lw=1.6,
            label=f"{sample_id} (cutoff={r['time_s'][r['frame_max']]:.1f}s)"
        )
        ax.axvline(r["time_s"][r["frame_max"]], color=line.get_color(), ls="--", lw=1.2)
    ax.set_xlabel("Annealing Time (s)")
    ax.set_ylabel("Norm. (100) Peak Intensity")
    ax.set_title("Auto-Detected Perovskite (100) Max & Beam-Damage Cutoff")
    ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1.0),
              ncol=max(1, int(np.ceil(len(reduced) / 18))))   # legend outside the axes, never over the traces
    plt.show()

    return reduced


REDUCED_DATA = process_all_samples(CONDITIONS, CALIB_DATA, q_100_window=Q_100_SEARCH_WINDOW)
reduced_data = REDUCED_DATA  # Alias for convenience


### 5A

In [ ]:
import math
import re
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import grey_opening, gaussian_filter1d, median_filter
from scipy.signal import find_peaks, peak_widths

# ==============================================================================
# SETTINGS
# ==============================================================================
OUTPUT_DIR = Path(globals().get("OUTPUT_DIR", r"H:\17.2 beam stuff\2026_05_Stone\outputs"))
Q_100_SEARCH_WINDOW = globals().get("Q_100_SEARCH_WINDOW", (0.96, 1.05))
Q_DELTA_SEARCH_WINDOW = globals().get("Q_DELTA_SEARCH_WINDOW", (0.82, 0.96))
Q_PLOT_RANGE = None                             # None = plot entire scanned q-range; or set e.g. (0.35, 2.25)
DT_SEC_FALLBACK = globals().get("DT_SEC_FALLBACK", 0.5)

# (exclusion lists, MIN_FRAMES, cut-offs and t_max rule come from the shared helper cell)
SERIES_ORDER = ["Y", "S", "P", "F"]

# Glass substrate halo & high-q ripple filtering
SUBTRACT_GLASS_BG = True
BG_WINDOW_Q = 0.20
BG_SCALE = 1.03
BG_LIFT_SIGMA = 1.0

DESPIKE_1D = True
DESPIKE_WINDOW_BINS = 7
FILTER_NON_PEAK_NOISE = True
LOW_Q_PROTECT_MAX = 1.05                        # Preserves all low-q & (100) features <= 1.05 A^-1
MIN_PEAK_PROM_REL = 0.018
MIN_PEAK_PROM_ABS = 18.0
MIN_PEAK_FWHM_Q = 0.004
MAX_PEAK_FWHM_Q = 0.080

MAX_PER_PANEL = 5
STACK_WATERFALL = True


def is_excluded_scan(name):
    return is_excluded_sample(name)


def _natural_key(name):
    return [int(tok) if tok.isdigit() else tok.lower() for tok in re.split(r"(\d+)", str(name))]


def _series_rank(name):
    m = re.match(r"^[A-Za-z]+", str(name).strip())
    prefix = m.group(0)[0].upper() if m else "Z"
    return SERIES_ORDER.index(prefix) if prefix in SERIES_ORDER else len(SERIES_ORDER)


def despike_1d_subresolution(I_mat, win_bins=7, thresh_sigma=4.5):
    med = median_filter(I_mat, size=(1, win_bins), mode="nearest")
    resid = I_mat - med
    mad = np.median(np.abs(resid - np.median(resid, axis=1, keepdims=True)), axis=1, keepdims=True)
    noise_floor = np.maximum(1.4826 * mad * thresh_sigma, 1.2)
    spike_mask = (resid > noise_floor) & (resid > 0.18 * np.maximum(med, 1.0))
    return np.where(spike_mask, med, I_mat)


def subtract_glass_background_2d(q_arr, I_mat_in, window_q=0.20, bg_scale=1.03, lift_sigma=1.0):
    dq = float(np.median(np.diff(q_arr)))
    win_bins = max(15, int(round(window_q / dq))) | 1
    opened = grey_opening(I_mat_in, size=(1, win_bins))
    bg_base = gaussian_filter1d(opened, sigma=win_bins / 3.5, axis=1, mode="nearest")
    pos_resid = np.maximum(I_mat_in - bg_base, 0.0)
    floor_lift = np.percentile(pos_resid, 40, axis=1, keepdims=True) * (lift_sigma / 1.5)
    bg = bg_base * bg_scale + floor_lift
    return np.maximum(I_mat_in - bg, 0.0), bg


def filter_non_defined_peaks_2d(
    I_mat, q_arr, min_prom_rel=0.018, min_prom_abs=18.0, min_fwhm_q=0.004, max_fwhm_q=0.080, protected_windows=()
):
    dq_loc = float(np.median(np.diff(q_arr)))
    min_w_bins = max(2.5, min_fwhm_q / dq_loc)
    max_w_bins = max(min_w_bins + 2.0, max_fwhm_q / dq_loc)

    prot_mask = np.zeros(len(q_arr), dtype=bool)
    for q_lo, q_hi in protected_windows:
        prot_mask |= (q_arr >= q_lo) & (q_arr <= q_hi)
    prot_weights = (
        gaussian_filter1d(prot_mask.astype(np.float32), sigma=2.0)
        if np.any(prot_mask)
        else np.zeros(len(q_arr), dtype=np.float32)
    )

    out = np.empty_like(I_mat)
    for i in range(I_mat.shape[0]):
        y = I_mat[i]
        y_max = float(np.max(y))
        if y_max <= 0:
            out[i] = y
            continue
        prom_thresh = max(min_prom_abs, min_prom_rel * y_max)
        y_det = gaussian_filter1d(y, sigma=1.5)
        peaks, _ = find_peaks(y_det, prominence=prom_thresh, width=(min_w_bins, max_w_bins))
        keep = np.zeros(len(y), dtype=bool)
        if len(peaks) > 0:
            _, _, left_ips, right_ips = peak_widths(y_det, peaks, rel_height=0.90)
            for l_i, r_i in zip(left_ips, right_ips):
                w = r_i - l_i
                pad = max(3, int(round(0.15 * w)))
                keep[max(0, int(np.floor(l_i)) - pad) : min(len(y), int(np.ceil(r_i)) + pad + 1)] = True
        weights = np.maximum(gaussian_filter1d(keep.astype(np.float32), sigma=1.5), prot_weights)
        out[i] = y * weights
    return out




npz_files = sorted(OUTPUT_DIR.rglob("*.npz"))
assert npz_files, f"No .npz files found under {OUTPUT_DIR}."

records = []
for npz_p in npz_files:
    clean_name = npz_p.stem.replace("_reduced", "").strip()
    if is_excluded_scan(clean_name):
        continue

    with np.load(npz_p, allow_pickle=False) as d:
        payload = {k: d[k] for k in d.files}

    keys = set(payload.keys())
    q_key = next((k for k in ("q", "q_A_inv", "q_grid", "radial") if k in keys), None)
    I_key = next((k for k in ("I_matrix", "I_1d", "I_qt", "I_q", "I", "intensities", "intensity") if k in keys), None)
    t_key = next((k for k in ("time_s", "t_sec", "times", "time", "t") if k in keys), None)
    if q_key is None or I_key is None:
        continue

    q_arr = np.asarray(payload[q_key], dtype=np.float64)
    I_raw = np.asarray(payload.get("I_matrix_raw", payload[I_key]), dtype=np.float32)
    if I_raw.ndim != 2 or I_raw.shape[0] < MIN_FRAMES:
        continue

    t_arr = (
        np.asarray(payload[t_key], dtype=np.float64)
        if t_key is not None
        else resolve_time_axis(clean_name, I_raw.shape[0], np.float64)
    )

    I_work = despike_1d_subresolution(I_raw, win_bins=DESPIKE_WINDOW_BINS) if DESPIKE_1D else I_raw.copy()
    if SUBTRACT_GLASS_BG:
        I_sub, I_bg = subtract_glass_background_2d(q_arr, I_work, BG_WINDOW_Q, BG_SCALE, BG_LIFT_SIGMA)
    else:
        I_sub, I_bg = I_work, np.zeros_like(I_work)

    if FILTER_NON_PEAK_NOISE:
        I_clean = filter_non_defined_peaks_2d(
            I_sub, q_arr, MIN_PEAK_PROM_REL, MIN_PEAK_PROM_ABS, MIN_PEAK_FWHM_Q, MAX_PEAK_FWHM_Q,
            protected_windows=[(0.0, LOW_Q_PROTECT_MAX)],
        ).astype(np.float32)
    else:
        I_clean = I_sub.astype(np.float32)

    I_bg = I_bg.astype(np.float32)
    f_max = int(np.clip(find_alpha100_frame_max(q_arr, I_clean, t_arr=t_arr, t_cutoff=get_sample_cutoff(clean_name)), 0, len(t_arr) - 1))

    # Locate frame where transient low-q signal (0.35 - 0.85 A^-1) is strongest
    low_q_mask = (q_arr >= 0.35) & (q_arr <= 0.85)
    if np.any(low_q_mask):
        low_q_sub = I_clean[:, low_q_mask]
        f_lowq = int(np.argmax(np.max(low_q_sub, axis=1)))
        q_lowq_pk = float(q_arr[low_q_mask][np.argmax(low_q_sub[f_lowq])])
        val_lowq_pk = float(np.max(low_q_sub[f_lowq]))
    else:
        f_lowq, q_lowq_pk, val_lowq_pk = 0, 0.0, 0.0

    payload["I_matrix_raw"] = I_raw
    payload["I_glass_bg"] = I_bg
    payload[I_key] = I_clean
    for max_key in ("I_at_max", "I_max", "I_1d_max"):
        if max_key in payload:
            payload[max_key] = I_clean[f_max]
    for fmax_key in ("frame_max", "max_frame", "peak_frame", "idx_max", "max_idx"):
        if fmax_key in payload:
            payload[fmax_key] = np.int32(f_max)
    np.savez_compressed(npz_p, **payload)

    records.append({
        "name": clean_name,
        "npz_path": npz_p,
        "q": q_arr,
        "I_raw_max": I_raw[f_max],
        "I_bg_max": I_bg[f_max],
        "I_clean_max": I_clean[f_max],
        "f_max": f_max,
        "t_max": float(t_arr[f_max]),
        "f_lowq": f_lowq,
        "t_lowq": float(t_arr[f_lowq]),
        "q_lowq_pk": q_lowq_pk,
        "val_lowq_pk": val_lowq_pk,
    })

assert records, "No valid sample .npz records left after filtering."
records.sort(key=lambda r: (_series_rank(r["name"]), _natural_key(r["name"])))

# Exact-name sync into REDUCED_DATA in RAM
if "REDUCED_DATA" in globals() and isinstance(REDUCED_DATA, dict):
    rec_by_name = {r["name"]: r for r in records}
    for k in list(REDUCED_DATA.keys()):
        ck = str(k).replace("_reduced", "").strip()
        if is_excluded_scan(ck) or ck not in rec_by_name:
            del REDUCED_DATA[k]
            continue
        r = rec_by_name[ck]
        with np.load(r["npz_path"], allow_pickle=False) as ds:
            ik = next(x for x in ("I_matrix", "I_1d", "I_qt", "I_q", "I") if x in ds.files)
            cmat = ds[ik]
        entry = REDUCED_DATA[k]
        for ak, av in list(entry.items()):
            if isinstance(av, np.ndarray) and ak != "q":
                if av.shape == r["I_clean_max"].shape:
                    entry[ak] = r["I_clean_max"]
                elif av.ndim == 2 and av.shape == cmat.shape:
                    entry[ak] = cmat
        for fmk in ("frame_max", "max_frame", "peak_frame", "idx_max", "max_idx"):
            if fmk in entry:
                entry[fmk] = r["f_max"]

print(f"Cleaned & synced {len(records)} sample(s) (sorted Y → S → P → F; low-q <= {LOW_Q_PROTECT_MAX} A^-1 preserved):")
for r in records:
    print(
        f"  • {r['name']}: max (100) @ frame {r['f_max']} ({r['t_max']:.1f} s) | "
        f"strongest low-q signal ({r['val_lowq_pk']:.1f} a.u. @ q={r['q_lowq_pk']:.3f} A^-1) @ frame {r['f_lowq']} ({r['t_lowq']:.1f} s)"
    )

# ==============================================================================
# PLOT: [Panel 0] Baseline & Peak-Filter Check + [Panels 1..N] Full 1D Patterns
# ==============================================================================
n_1d_panels = max(1, math.ceil(len(records) / MAX_PER_PANEL))
total_panels = n_1d_panels + 1
ncols = 2
nrows = math.ceil(total_panels / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(7.2 * ncols, 4.4 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()
cmap = plt.get_cmap("tab10")

# Panel 0: Verify glass baseline & peak-filter on the first sample
ax0 = axes[0]
r0 = records[0]
q_lo0, q_hi0 = Q_PLOT_RANGE if Q_PLOT_RANGE is not None else (float(r0["q"].min()), float(r0["q"].max()))
mp0 = (r0["q"] >= q_lo0) & (r0["q"] <= q_hi0)
ax0.plot(r0["q"][mp0], r0["I_raw_max"][mp0], color="#888888", lw=1.1, label="Raw 1D (with glass)")
ax0.plot(r0["q"][mp0], r0["I_bg_max"][mp0], color="#d62728", lw=1.5, ls="--", label="Glass background")
ax0.plot(r0["q"][mp0], r0["I_clean_max"][mp0], color="#1f77b4", lw=1.5, label="Cleaned (low-q preserved)")
ax0.set_xlim(q_lo0, q_hi0)
ax0.set_xlabel(r"$q$ ($\mathrm{\AA}^{-1}$)")
ax0.set_ylabel("Intensity (a.u.)")
ax0.set_title(f"Baseline & Peak-Filter Check: {r0['name']} (t = {r0['t_max']:.1f} s)")
ax0.grid(True, alpha=0.25)
ax0.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), fontsize=8, frameon=False, ncol=3)

# Panels 1..N: Full 1D patterns at max (100) frame
for p_idx in range(n_1d_panels):
    ax = axes[1 + p_idx]
    chunk = records[p_idx * MAX_PER_PANEL : (p_idx + 1) * MAX_PER_PANEL]

    y_offset = 0.0
    for c_idx, rec in enumerate(chunk):
        col = cmap(c_idx % 10)
        q_lo, q_hi = Q_PLOT_RANGE if Q_PLOT_RANGE is not None else (float(rec["q"].min()), float(rec["q"].max()))
        mp = (rec["q"] >= q_lo) & (rec["q"] <= q_hi)
        q_sub = rec["q"][mp]
        y_sub = rec["I_clean_max"][mp]

        ax.plot(
            q_sub,
            y_sub + y_offset,
            lw=1.4,
            color=col,
            label=f"{rec['name']} (t_max={rec['t_max']:.1f} s)",
        )
        if STACK_WATERFALL:
            step = max(float(np.nanpercentile(y_sub, 99.2)) * 0.55, 1.0)
            y_offset += step

    ax.set_xlim(q_lo, q_hi)
    ax.set_xlabel(r"$q$ ($\mathrm{\AA}^{-1}$)")
    ax.set_ylabel("Intensity (a.u., offset)" if STACK_WATERFALL else "Intensity (a.u.)")
    ax.set_title(f"Full 1D GIWAXS at Max (100) Peak — Group {p_idx + 1}/{n_1d_panels}")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), fontsize=8, frameon=False, ncol=2)

for extra_ax in axes[total_panels:]:
    extra_ax.set_visible(False)

plt.show()

## Cell 5B (Optional): Interactive Sample & Frame Explorer (Dropdown + Slider)
**What this cell does:** Opens a lightweight interactive viewer on the **already-reduced 1D patterns** (`REDUCED_DATA`) so you can switch conditions with a dropdown and scrub through frames/annealing time without re-integrating raw detector images or using extra memory.


In [ ]:
from pathlib import Path
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

OUTPUT_DIR = Path(globals().get("OUTPUT_DIR", r"H:\17.2 beam stuff\2026_05_Stone\outputs"))
Q_100_SEARCH_WINDOW = globals().get("Q_100_SEARCH_WINDOW", (0.96, 1.05))
Q_DELTA_SEARCH_WINDOW = globals().get("Q_DELTA_SEARCH_WINDOW", (0.82, 0.96))
DT_SEC_FALLBACK = globals().get("DT_SEC_FALLBACK", 0.5)

_trapz = getattr(np, "trapezoid", None) or getattr(np, "trapz")


def launch_bg_subtracted_explorer():
    npz_files = [
        p for p in sorted(OUTPUT_DIR.rglob("*.npz"))
        if not is_excluded_sample(p.stem)
    ]
    assert npz_files, f"No valid .npz files found under {OUTPUT_DIR}."

    scans = {}
    for npz_p in npz_files:
        with np.load(npz_p, allow_pickle=False) as d:
            keys = set(d.files)
            q_key = next((k for k in ("q", "q_A_inv", "q_grid", "radial") if k in keys), None)
            I_key = next((k for k in ("I_matrix", "I_1d", "I_qt", "I_q", "I", "intensities", "intensity") if k in keys), None)
            t_key = next((k for k in ("time_s", "t_sec", "times", "time", "t") if k in keys), None)
            if q_key is None or I_key is None:
                continue
            q_arr = np.asarray(d[q_key], dtype=np.float32)
            I_clean = np.asarray(d[I_key], dtype=np.float32)
            if I_clean.ndim != 2:
                continue
            I_raw = np.asarray(d["I_matrix_raw"], dtype=np.float32) if "I_matrix_raw" in keys else I_clean
            I_bg = np.asarray(d["I_glass_bg"], dtype=np.float32) if "I_glass_bg" in keys else np.zeros_like(I_clean)
            t_arr = (
                np.asarray(d[t_key], dtype=np.float32)
                if t_key is not None
                else resolve_time_axis(npz_p.stem, I_clean.shape[0], np.float32)
            )

            q_win = (q_arr >= Q_100_SEARCH_WINDOW[0]) & (q_arr <= Q_100_SEARCH_WINDOW[1])
            fmax_key = next((k for k in ("frame_max", "max_frame", "peak_frame", "idx_max", "max_idx") if k in keys), None)
            f_max = int(d[fmax_key]) if fmax_key is not None else int(np.argmax(_trapz(I_clean[:, q_win], q_arr[q_win], axis=1)))
            f_max = int(np.clip(f_max, 0, len(t_arr) - 1))

            name = npz_p.stem.replace("_reduced", "")
            scans[name] = {
                "q": q_arr,
                "I_clean": I_clean,
                "I_raw": I_raw,
                "I_bg": I_bg,
                "time_s": t_arr,
                "f_max": f_max,
            }

    scan_names = list(scans.keys())
    init_name = scan_names[0]
    init_scan = scans[init_name]
    q_floor = float(np.nanmin(init_scan["q"]))
    q_ceil = float(np.nanmax(init_scan["q"]))

    init_roi_lo = max(q_floor, round(min(Q_DELTA_SEARCH_WINDOW[0], Q_100_SEARCH_WINDOW[0]) - 0.05, 3))
    init_roi_hi = min(q_ceil, round(max(Q_DELTA_SEARCH_WINDOW[1], Q_100_SEARCH_WINDOW[1]) + 0.05, 3))

    # --- Top Bar: Sample, View Mode, and Full-Scan Reset ---
    sample_dd = widgets.Dropdown(options=scan_names, value=init_name, description="Sample:", layout=widgets.Layout(width="270px"))
    view_dd = widgets.Dropdown(
        options=[
            ("Background-Subtracted", "clean"),
            ("Overlay (Raw + Glass Fit + Clean)", "overlay"),
            ("Raw (with Glass)", "raw"),
        ],
        value="clean",
        description="View:",
        layout=widgets.Layout(width="280px"),
    )
    reset_q_btn = widgets.Button(description="Reset Full q", icon="refresh", layout=widgets.Layout(width="125px"))

    # --- Frame Slider ---
    play_btn = widgets.Play(value=init_scan["f_max"], min=0, max=len(init_scan["time_s"]) - 1, step=1, interval=250)
    slider = widgets.IntSlider(
        value=init_scan["f_max"], min=0, max=len(init_scan["time_s"]) - 1, step=1,
        description="Frame:", layout=widgets.Layout(width="420px")
    )
    widgets.jslink((play_btn, "value"), (slider, "value"))

    # --- Full Plot q-Range Slider + Text Boxes (defaults to entire scan range) ---
    plot_q_slider = widgets.FloatRangeSlider(
        value=[q_floor, q_ceil],
        min=q_floor,
        max=q_ceil,
        step=0.005,
        description="Plot q (Å⁻¹):",
        readout=False,
        layout=widgets.Layout(width="360px"),
    )
    plot_q_min = widgets.FloatText(value=round(q_floor, 4), step=0.01, description="min:", layout=widgets.Layout(width="145px"))
    plot_q_max = widgets.FloatText(value=round(q_ceil, 4), step=0.01, description="max:", layout=widgets.Layout(width="145px"))

    # --- ROI Range Slider + Text Boxes (controls shaded ROI & right zoomed panel) ---
    roi_slider = widgets.FloatRangeSlider(
        value=[init_roi_lo, init_roi_hi],
        min=q_floor,
        max=q_ceil,
        step=0.002,
        description="ROI q (Å⁻¹):",
        readout=False,
        layout=widgets.Layout(width="360px"),
    )
    roi_min = widgets.FloatText(value=round(init_roi_lo, 4), step=0.01, description="min:", layout=widgets.Layout(width="145px"))
    roi_max = widgets.FloatText(value=round(init_roi_hi, 4), step=0.01, description="max:", layout=widgets.Layout(width="145px"))

    status_html = widgets.HTML()

    # Create figure and wrap in widgets.Output if running under inline FigureCanvasAgg
    plt.ioff()
    fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(13.5, 4.5), constrained_layout=True)
    use_widget_canvas = isinstance(fig.canvas, widgets.Widget)
    if use_widget_canvas:
        plt.ion()
        canvas_widget = fig.canvas
    else:
        plt.close(fig)  # Prevent duplicate static figure below the VBox in inline mode
        canvas_widget = widgets.Output()

    def _safe_y_top(arr_slice):
        if arr_slice.size == 0:
            return 1.0
        finite = arr_slice[np.isfinite(arr_slice)]
        if finite.size == 0:
            return 1.0
        return max(float(np.max(finite)) * 1.10, 1.0)

    def _draw_traces(ax, q, mask, sc, idx, mode):
        if mode == "clean":
            ax.plot(q[mask], sc["I_clean"][idx, mask], color="#1f77b4", lw=1.6, label="Glass-subtracted")
            return _safe_y_top(sc["I_clean"][idx, mask])
        elif mode == "raw":
            ax.plot(q[mask], sc["I_raw"][idx, mask], color="#555555", lw=1.6, label="Raw (with glass)")
            return _safe_y_top(sc["I_raw"][idx, mask])
        else:
            ax.plot(q[mask], sc["I_raw"][idx, mask], color="#888888", lw=1.1, alpha=0.8, label="Raw")
            ax.plot(q[mask], sc["I_bg"][idx, mask], color="#d62728", lw=1.4, ls="--", label="Glass baseline")
            ax.plot(q[mask], sc["I_clean"][idx, mask], color="#1f77b4", lw=1.6, label="Subtracted")
            return _safe_y_top(sc["I_raw"][idx, mask])

    def _render():
        name = sample_dd.value
        idx = slider.value
        mode = view_dd.value
        p_lo, p_hi = plot_q_slider.value
        r_lo, r_hi = roi_slider.value

        sc = scans[name]
        q = sc["q"]
        t_arr = sc["time_s"]
        f_max = sc["f_max"]
        idx = int(np.clip(idx, 0, len(t_arr) - 1))

        mz = (q >= r_lo) & (q <= r_hi)
        roi_peak = float(np.nanmax(sc["I_clean"][idx, mz])) if np.any(mz) else 0.0

        badge = (
            "<span style='color:#1b7e3c;font-weight:bold;'>● Active Growth (≤ Max 100)</span>"
            if idx <= f_max
            else "<span style='color:#c62828;font-weight:bold;'>● Post-Peak Degradation (> Max 100)</span>"
        )
        status_html.value = (
            f"<div style='font-size:13px;padding:4px 8px;background:#f4f6f8;border-radius:4px;'>"
            f"<b>Frame {idx} / {len(t_arr)-1}</b> &nbsp;|&nbsp; t = <b>{t_arr[idx]:.2f} s</b> "
            f"(max @ frame {f_max}, {t_arr[f_max]:.2f} s) &nbsp;|&nbsp; "
            f"ROI [{r_lo:.3f}, {r_hi:.3f}] peak = <b>{roi_peak:.1f} a.u.</b> &nbsp;|&nbsp; {badge}</div>"
        )

        # Left panel: Adjustable Plot q-Range (defaults to entire scan)
        ax_full.clear()
        mp = (q >= p_lo) & (q <= p_hi)
        y_top_full = _draw_traces(ax_full, q, mp, sc, idx, mode)
        ax_full.axvspan(r_lo, r_hi, color="#ffd54f", alpha=0.28, label=f"Selected ROI ({r_lo:.2f}–{r_hi:.2f})")
        ax_full.set_xlim(p_lo, p_hi)
        ax_full.set_ylim(0, y_top_full)
        ax_full.set_xlabel(r"$q$ ($\mathrm{\AA}^{-1}$)")
        ax_full.set_ylabel("Intensity (a.u.)")
        ax_full.set_title(f"{name} — 1D Pattern ({p_lo:.2f}–{p_hi:.2f} Å⁻¹)")
        ax_full.grid(True, alpha=0.2)
        ax_full.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), fontsize=8, frameon=False, ncol=3)

        # Right panel: Zoomed to User-Selected ROI
        ax_zoom.clear()
        y_top_zoom = _draw_traces(ax_zoom, q, mz, sc, idx, mode)
        ax_zoom.set_xlim(r_lo, r_hi)
        ax_zoom.set_ylim(0, y_top_zoom)
        ax_zoom.set_xlabel(r"$q$ ($\mathrm{\AA}^{-1}$)")
        ax_zoom.set_ylabel("Intensity (a.u.)")
        ax_zoom.set_title(f"Zoomed ROI ({r_lo:.3f}–{r_hi:.3f} Å⁻¹)")
        ax_zoom.grid(True, alpha=0.2)
        ax_zoom.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), fontsize=8, frameon=False, ncol=3)

        if use_widget_canvas:
            fig.canvas.draw_idle()
        else:
            with canvas_widget:
                clear_output(wait=True)
                display(fig)

    # --- Synchronized Observer Callbacks (with recursion guard) ---
    syncing = False

    def _on_plot_q_slider(change):
        nonlocal syncing
        if syncing:
            return
        syncing = True
        lo, hi = change["new"]
        plot_q_min.value = round(lo, 4)
        plot_q_max.value = round(hi, 4)
        syncing = False
        _render()

    def _on_plot_q_text(_):
        nonlocal syncing
        if syncing:
            return
        syncing = True
        s_min, s_max = plot_q_slider.min, plot_q_slider.max
        lo = float(np.clip(plot_q_min.value, s_min, s_max - 0.005))
        hi = float(np.clip(plot_q_max.value, lo + 0.005, s_max))
        plot_q_min.value = round(lo, 4)
        plot_q_max.value = round(hi, 4)
        plot_q_slider.value = (lo, hi)
        syncing = False
        _render()

    def _on_roi_slider(change):
        nonlocal syncing
        if syncing:
            return
        syncing = True
        lo, hi = change["new"]
        roi_min.value = round(lo, 4)
        roi_max.value = round(hi, 4)
        syncing = False
        _render()

    def _on_roi_text(_):
        nonlocal syncing
        if syncing:
            return
        syncing = True
        s_min, s_max = roi_slider.min, roi_slider.max
        lo = float(np.clip(roi_min.value, s_min, s_max - 0.002))
        hi = float(np.clip(roi_max.value, lo + 0.002, s_max))
        roi_min.value = round(lo, 4)
        roi_max.value = round(hi, 4)
        roi_slider.value = (lo, hi)
        syncing = False
        _render()

    def _on_reset_q(_):
        plot_q_slider.value = (plot_q_slider.min, plot_q_slider.max)

    def _on_sample(change):
        nonlocal syncing
        sc = scans[change["new"]]
        q_lo_new = float(np.nanmin(sc["q"]))
        q_hi_new = float(np.nanmax(sc["q"]))

        syncing = True
        slider.max = len(sc["time_s"]) - 1
        play_btn.max = len(sc["time_s"]) - 1
        slider.value = sc["f_max"]

        plot_q_slider.min = q_lo_new
        plot_q_slider.max = q_hi_new
        plot_q_slider.value = (q_lo_new, q_hi_new)
        plot_q_min.value = round(q_lo_new, 4)
        plot_q_max.value = round(q_hi_new, 4)

        roi_slider.min = q_lo_new
        roi_slider.max = q_hi_new
        r_lo = float(np.clip(roi_slider.value[0], q_lo_new, q_hi_new - 0.002))
        r_hi = float(np.clip(roi_slider.value[1], r_lo + 0.002, q_hi_new))
        roi_slider.value = (r_lo, r_hi)
        roi_min.value = round(r_lo, 4)
        roi_max.value = round(r_hi, 4)
        syncing = False
        _render()

    sample_dd.observe(_on_sample, names="value")
    view_dd.observe(lambda _: _render(), names="value")
    slider.observe(lambda _: _render(), names="value")
    reset_q_btn.on_click(_on_reset_q)

    plot_q_slider.observe(_on_plot_q_slider, names="value")
    plot_q_min.observe(_on_plot_q_text, names="value")
    plot_q_max.observe(_on_plot_q_text, names="value")

    roi_slider.observe(_on_roi_slider, names="value")
    roi_min.observe(_on_roi_text, names="value")
    roi_max.observe(_on_roi_text, names="value")

    controls = widgets.VBox([
        widgets.HBox([sample_dd, view_dd, reset_q_btn]),
        widgets.HBox([play_btn, slider]),
        widgets.HBox([plot_q_slider, plot_q_min, plot_q_max]),
        widgets.HBox([roi_slider, roi_min, roi_max]),
        status_html,
        canvas_widget,
    ])
    display(controls)
    _render()


launch_bg_subtracted_explorer()

## Cell 5C: Annealing Temperature + Motion Velocity / Acceleration
**What this cell does:** For every sample in `REDUCED_DATA`, finds its run folder under `HEATING_ROOT`, reads the **event log** (`motion_start`, `motion_end`, `anneal_start`), the **temperature csv** (3 substrate thermocouples) and the **`DATA_RAPID_RUN` motion csv** (position vs. time), then:
- aligns temperature to *time since anneal start* (starts exactly at the `anneal_start` timestamp) and computes average / peak heating rate (°C/s), plateau, setpoint and temperature at the (100) max;
- takes motion `Time_s = 0` as `motion_start`, windows to `motion_end`, and differentiates position to get **velocity (m/min)** and **acceleration (m/min²)**, flagging any run whose clocks look offset;
- plots every run plus an all-run summary and saves everything to `outputs/heating/` (summary CSV included).

Run after Cell 5 / 5A. Edit `HEATING_ROOT` in the settings block if the data folder moves.

In [ ]:
# =============================================================================
# CELL 5C: ANNEALING TEMPERATURE (thermocouples) + MOTION VELOCITY, per sample
# Run AFTER Cell 3 (CONDITIONS) and Cell 5 / 5A (REDUCED_DATA). Fully self-contained.
#
# For every sample in REDUCED_DATA it:
#   1. finds that sample's folder under HEATING_ROOT (matched by sample ID)
#   2. reads the event log  -> motion start / motion end / anneal start (UTC)
#   3. reads the temperature csv -> 3 thermocouples on the substrate holder
#   4. aligns temperature to "time since anneal start" (= GIWAXS anneal time)
#   5. computes heating rate (degC/s), plateau T, T at the (100) max, velocity
#   6. plots every run + an all-sample summary, saves a summary csv
# =============================================================================
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d
from scipy.signal import savgol_filter

# ------------------------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------------------------
HEATING_ROOT = Path(r"C:\Users\matth\OneDrive\Desktop\RAPID_GUI_AUTOMATION\RAPID_GUI_AUTOMATION\data\data")

# Force a specific folder for a sample if auto-matching picks the wrong one, e.g.
# {"S1A-try2": Path(r"C:\...\data\data\S1A_try2_2026-05-12")}
HEATING_FOLDER_OVERRIDES = {}

SAMPLES = None                 # None = every sample in REDUCED_DATA, or e.g. ["F1A", "S10A"]
SHOW_PER_SAMPLE = True         # False = only save per-sample PNGs, show just the summary figures

# --- Time alignment ----------------------------------------------------------
ALIGN_TO_H5_UTC = True         # True = line the GIWAXS clock up with the heating clock using the UTC start time stored in the anneal .h5
                               # (GIWAXS frame 0 = h5 start time; heating clock = anneal_start event / thermocouple UTC).
                               # False, or if the h5 has no readable start time = use ANNEAL_T0_OFFSET_S below.
ANNEAL_T0_OFFSET_S = 0.0       # EXTRA manual shift (s) added on top (or used alone when ALIGN_TO_H5_UTC is off):
                               # GIWAXS frame 0 is taken to be this many seconds after the anneal_start event.
MAX_PLAUSIBLE_OFFSET_S = 120.0 # |h5 start - anneal event| larger than this is reported as suspicious (clock / timezone problem?)
TEMP_TZ_OFFSET_HOURS = 0.0     # Only if the temperature csv timestamps are LOCAL time (not UTC),
                               # e.g. Phoenix = -7. Left at 0 the code auto-tries whole-hour shifts
                               # if the anneal-start event falls outside the temperature data.

# --- Heating analysis --------------------------------------------------------
PRE_ANNEAL_S = 0.0             # 0 = temperature data starts exactly at the anneal_start timestamp.
                               # Set e.g. 20 to also keep 20 s of pre-anneal data (and show motion on the plot).
POST_ANNEAL_S_FALLBACK = 300.0 # analysis length if a sample has no GIWAXS time axis
SMOOTH_S = 2.0                 # smoothing window for rate calculations (s)
PLATEAU_S = 20.0               # plateau temp = median of the last PLATEAU_S seconds of the window
MIN_RISE_C = 5.0               # rises smaller than this are treated as "no ramp found"

# --- Velocity ----------------------------------------------------------------
# Units: position in m, velocity in m/min, acceleration in m/min^2.
# Velocity/acceleration come from the DATA_RAPID_RUN csv's Position_m. MOTION_DISTANCE_M is only a
# FALLBACK for runs with no such file: velocity = distance / (motion_end - motion_start).
# Scalar (same for all runs) or dict per sample, e.g. 0.05 or {"F1A": 0.05, "S1A": 0.04}. None = duration only.
MOTION_DISTANCE_M = None
VELOCITY_UNIT = "m/min"
ACCEL_UNIT = "m/min$^2$"
MOTION_SMOOTH_S = 0.10         # Savitzky-Golay window for position -> velocity -> acceleration (s).
                               # Acceleration is noisy; raise this (e.g. 0.2) if the accel trace looks spiky.
MOTION_PAD_S = 1.0             # extra seconds either side of motion_start..motion_end used for derivatives/plots
MOTION_SYNC_TOL_S = 0.5        # warn if the stage starts/stops more than this far from the event timestamps
MOTION_MOVING_FRAC = 0.05      # "moving" = speed above this fraction of peak (onset/end detection)
MOTION_CRUISE_FRAC = 0.90      # cruise velocity = median speed where speed >= this fraction of peak

# --- Event names (regex on the 'event' column; underscores are treated as spaces) --
EVENT_PATTERNS = {
    "motion_start": r"^motion start$",
    "motion_end":   r"^motion end$",
    "anneal_start": r"^anneal start$",
}
# Note: exact names on purpose, so motion_delay_start / motion_delay_end / motion_stop_requested
# in your log are NOT mistaken for the motion events.

HEATING_OUT_DIR = OUTPUT_DIR / "heating"
HEATING_OUT_DIR.mkdir(exist_ok=True)


# ------------------------------------------------------------------------------
# CSV / TIME HELPERS
# ------------------------------------------------------------------------------
def _read_csv(path):
    with open(path, "r", encoding="utf-8", errors="replace") as fh:
        head = fh.readline()
    sep = "\t" if head.count("\t") > head.count(",") else ","
    try:
        df = pd.read_csv(path, sep=sep, encoding_errors="replace", skipinitialspace=True)
    except Exception:
        df = pd.read_csv(path, sep=None, engine="python", encoding_errors="replace")
    df.columns = [str(c).strip() for c in df.columns]
    return df


def _to_epoch(series):
    """Return absolute time as float epoch seconds (UTC), or None if the column isn't an absolute time."""
    if pd.api.types.is_numeric_dtype(series):
        v = pd.to_numeric(series, errors="coerce").astype(float)
        if v.notna().sum() == 0:
            return None
        med = float(np.nanmedian(v))
        if med > 1e17:
            v = v / 1e9
        elif med > 1e14:
            v = v / 1e6
        elif med > 1e11:
            v = v / 1e3
        elif med <= 1e8:
            return None                      # relative seconds, not a wall-clock time
        return v.to_numpy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        dt = pd.to_datetime(series, errors="coerce", utc=True)
    if dt.notna().mean() < 0.9:
        return None
    return (dt - pd.Timestamp("1970-01-01", tz="UTC")).dt.total_seconds().to_numpy()


def _pick_time_col(df, prefer=("utc",)):
    cols = list(df.columns)
    order = (
        [c for c in cols if any(p in c.lower() for p in prefer)]
        + [c for c in cols if re.search(r"time|stamp|date", c, re.I)]
        + cols
    )
    seen = set()
    for c in order:
        if c in seen:
            continue
        seen.add(c)
        ep = _to_epoch(df[c])
        if ep is not None:
            return c, ep
    raise ValueError(f"No absolute timestamp column found. Columns: {cols}")


def h5_start_epoch(h5_path):
    """UTC epoch (s) of the start of an Eiger/DECTRIS master .h5 (frame 0). -> (epoch or None, where it was found)."""
    import h5py
    keys = ["/entry/start_time", "/entry/instrument/detector/detectorSpecific/data_collection_date",
            "/entry/instrument/detector/detectorSpecific/collection_date", "/entry/instrument/detector/start_time",
            "/entry/instrument/NDAttributes/NDArrayTimeStamp", "/entry/data/start_time"]
    def _to_epoch(v):
        if isinstance(v, bytes):
            v = v.decode(errors="ignore")
        if isinstance(v, np.ndarray):
            if v.size == 0:
                return None
            v = v.flat[0]
            if isinstance(v, bytes):
                v = v.decode(errors="ignore")
        if isinstance(v, (int, float, np.integer, np.floating)):
            f = float(v)
            return f if f > 1e9 else None
        dt = pd.to_datetime(str(v).strip(), errors="coerce", utc=True)        # naive strings are taken as UTC
        return None if pd.isna(dt) else float(dt.timestamp())
    try:
        with h5py.File(h5_path, "r") as f:
            for k in keys:
                if k in f:
                    e = _to_epoch(f[k][()])
                    if e is not None:
                        return e, k
            for a in ("file_time", "start_time", "creation_time"):
                if a in f.attrs:
                    e = _to_epoch(f.attrs[a])
                    if e is not None:
                        return e, f"root attribute '{a}'"
    except Exception:
        pass
    return None, "no readable start time"


def _fmt_utc(epoch):
    if epoch is None or not np.isfinite(epoch):
        return ""
    return pd.Timestamp(epoch, unit="s", tz="UTC").strftime("%Y-%m-%d %H:%M:%S")


# ------------------------------------------------------------------------------
# FOLDER + FILE DISCOVERY
# ------------------------------------------------------------------------------
def _is_hidden(p):
    return any(part.startswith(".") for part in Path(p).parts)


def find_run_dir(sample_id, root, csv_dirs):
    """Match a sample ID to a run folder under root. Returns (dir, exact_match, ambiguous_dirs)."""
    sid = sample_id.lower()
    pat = re.compile(r"(?<![a-z0-9])" + re.escape(sid) + r"(?![a-z0-9])")
    cands = {}
    for d in csv_dirs:
        rel = d.relative_to(root).parts
        for k, part in enumerate(rel):
            low = part.lower()
            if low == sid or re.fullmatch(re.escape(sid) + r"[_-]\d{4}-\d{2}-\d{2}[_-]\d{2}-\d{2}-\d{2}", low):
                score = (0, len(low), k)
            elif pat.search(low):
                score = (1, len(low), k)
            else:
                continue
            run_dir = root.joinpath(*rel[: k + 1])
            if run_dir not in cands or score < cands[run_dir]:
                cands[run_dir] = score
            break
    if not cands:
        return None, False, []
    best_score = min(cands.values())
    best = sorted(d for d, s in cands.items() if s == best_score)
    return best[0], best_score[0] == 0, best[1:]


def _looks_like_event_log(df):
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            continue
        txt = df[c].astype(str).str.lower().str.replace("_", " ", regex=False)
        if any(txt.str.contains(p, regex=True).any() for p in EVENT_PATTERNS.values()):
            return True
    return False


def _looks_like_motion(df):
    return any(re.fullmatch(r"position_m", c.strip(), re.I) for c in df.columns)


def classify_csvs(csvs):
    """Return (event_log_path, temperature_path, motion_path) from the csvs in a run folder."""
    ev = tp = mo = None
    for p in csvs:
        n = p.name.lower()
        if mo is None and re.search(r"data_rapid|rapid_run", n):
            mo = p
        elif ev is None and "event" in n:
            ev = p
        elif tp is None and re.search(r"temp|thermo", n):
            tp = p
    for p in csvs:                            # content-based fallback for oddly named files
        if p in (ev, tp, mo):
            continue
        try:
            df = _read_csv(p).head(500)
        except Exception:
            continue
        if _looks_like_motion(df):
            mo = mo or p
        elif _looks_like_event_log(df):
            ev = ev or p
        else:
            tp = tp or p
    return ev, tp, mo


# ------------------------------------------------------------------------------
# EVENT LOG + TEMPERATURE LOADERS
# ------------------------------------------------------------------------------
def _pick_event_col(df):
    best_n, best_c = 0, None
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            continue
        txt = df[c].astype(str).str.lower().str.replace("_", " ", regex=False)
        n = sum(int(txt.str.contains(p, regex=True).sum()) for p in EVENT_PATTERNS.values())
        if n > best_n:
            best_n, best_c = n, c
    return best_c


def _numeric_col(df, pattern):
    for c in df.columns:
        if re.search(pattern, c, re.I) and pd.api.types.is_numeric_dtype(df[c]):
            return c
    return None


def load_event_log(path):
    df = _read_csv(path)
    ev_col = _pick_event_col(df)
    if ev_col is None:
        raise ValueError(f"No column with motion/anneal events found. Columns: {list(df.columns)}")
    _, epoch = _pick_time_col(df)
    txt = df[ev_col].astype(str).str.lower().str.replace("_", " ", regex=False).to_numpy()

    found = {k: [] for k in EVENT_PATTERNS}          # key -> [(epoch, row_index)]
    for i, (t, s) in enumerate(zip(epoch, txt)):
        if not np.isfinite(t):
            continue
        for key, pat in EVENT_PATTERNS.items():
            if re.search(pat, s):
                found[key].append((t, i))
                break
    for key in found:
        found[key].sort()

    # 'value' column: anneal_start -> setpoint (degC), experiment_start -> sample ID, arduino DISPENSE -> amount
    val_col = next((c for c in df.columns if c.lower() == "value"), None)
    allev = pd.Series(txt)
    def _val(event_regex):
        if val_col is None:
            return None
        m = allev.str.contains(event_regex, regex=True).to_numpy()
        vals = df[val_col][m].dropna().astype(str).str.strip()
        return vals.iloc[0] if len(vals) else None
    sp = _val(r"^anneal start$")
    try:
        setpoint = float(sp) if sp is not None else np.nan
    except ValueError:
        setpoint = np.nan
    disp = _val(r"^arduino command$")

    # pair each motion start with the next unused motion end
    segments, ends = [], list(found["motion_end"])
    for t_s, i_s in found["motion_start"]:
        nxt = next(((t_e, i_e) for t_e, i_e in ends if t_e > t_s), None)
        if nxt is not None:
            ends.remove(nxt)
            segments.append({"t_start": t_s, "t_end": nxt[0], "i_start": i_s, "i_end": nxt[1]})

    return {
        "df": df,
        "event_col": ev_col,
        "segments": segments,
        "anneal_start": found["anneal_start"][0][0] if found["anneal_start"] else np.nan,
        "n_anneal_events": len(found["anneal_start"]),
        "anneal_setpoint_C": setpoint,
        "experiment_id": _val(r"^experiment start$"),
        "arduino_cmd": disp,
    }


def load_temperature(path):
    df = _read_csv(path)
    tcol, epoch = _pick_time_col(df, prefer=("utc", "timestamp"))
    num = [
        c for c in df.columns
        if c != tcol and pd.api.types.is_numeric_dtype(df[c]) and not re.search(r"time|stamp|elapsed", c, re.I)
    ]
    named = [c for c in num if re.search(r"substrate|tc|therm|temp", c, re.I)]
    tc_cols = (named if len(named) >= 3 else num)[:3]
    if not tc_cols:
        raise ValueError(f"No thermocouple columns found. Columns: {list(df.columns)}")
    out = pd.DataFrame({"epoch": epoch})
    for c in tc_cols:
        out[c] = pd.to_numeric(df[c], errors="coerce").to_numpy()
    out = out.dropna(subset=["epoch"]).sort_values("epoch").reset_index(drop=True)
    return out, tc_cols


# ------------------------------------------------------------------------------
# HEATING ANALYSIS
# ------------------------------------------------------------------------------
def analyze_heating(t, T, post_s, pre_s):
    """t = seconds since anneal start, T = mean thermocouple temperature (degC)."""
    win = (t >= -pre_s) & (t <= post_s) & np.isfinite(T)
    tw, Tw = t[win], T[win]
    nan = {k: np.nan for k in ("T_start_C", "T_plateau_C", "rise_C", "t_onset_s", "t90_s", "avg_rate_C_per_s", "peak_rate_C_per_s")}
    nan.update(t10=np.nan, t90=np.nan, tw=tw, Ts=Tw)
    if len(tw) < 10:
        return nan

    dt = float(np.median(np.diff(tw)))
    n = max(1, int(round(SMOOTH_S / dt)))
    Ts = uniform_filter1d(Tw, size=n, mode="nearest")

    base = Tw[tw <= 0]
    if len(base) < 3:                      # no pre-anneal data: use the first few readings only
        base = Tw[:3]                      # (anything longer includes heating that has already started)
    T0 = float(np.median(base))
    n_plat = max(3, int(round(PLATEAU_S / dt)))
    Tp = float(np.median(Ts[-n_plat:]))
    rise = Tp - T0
    nan.update(T_start_C=T0, T_plateau_C=Tp, rise_C=rise, tw=tw, Ts=Ts)
    if rise < MIN_RISE_C:
        return nan

    def first_cross(level):
        idx = np.where(Ts >= level)[0]
        return float(tw[idx[0]]) if len(idx) else np.nan

    t02, t10, t90 = first_cross(T0 + 0.02 * rise), first_cross(T0 + 0.10 * rise), first_cross(T0 + 0.90 * rise)
    if not (np.isfinite(t10) and np.isfinite(t90)) or t90 <= t10:
        return nan

    m = (tw >= t10) & (tw <= t90)
    if m.sum() >= 3:
        rate = float(np.polyfit(tw[m], Tw[m], 1)[0])
    else:
        rate = 0.8 * rise / (t90 - t10)
    peak = float(np.nanmax(np.gradient(Ts, tw)))

    nan.update(
        t_onset_s=t02, t90_s=t90, avg_rate_C_per_s=rate, peak_rate_C_per_s=peak, t10=t10, t90=t90,
    )
    return nan


def _col(df, pattern):
    for c in df.columns:
        if re.fullmatch(pattern, c.strip(), re.I):
            return c
    return None


def analyze_motion(mdf, dur):
    """
    Velocity + acceleration from the DATA_RAPID_RUN csv.
    Time_s = 0 is taken as the motion_start event; the window ends at motion_end (dur seconds later).
    Outputs: position in m (always positive from the start), velocity in m/min, acceleration in m/min^2.
    """
    ct, cx = _col(mdf, r"time_s"), _col(mdf, r"position_m")
    if ct is None or cx is None:
        raise ValueError(f"motion csv needs Time_s and Position_m columns; found {list(mdf.columns)}")
    cv, ce = _col(mdf, r"velocity_m_min"), _col(mdf, r"tracking_error_counts")

    t_all = pd.to_numeric(mdf[ct], errors="coerce").to_numpy(float)
    x_all = pd.to_numeric(mdf[cx], errors="coerce").to_numpy(float)                   # m
    v_ctrl = pd.to_numeric(mdf[cv], errors="coerce").to_numpy(float) if cv else np.full(len(t_all), np.nan)  # already m/min
    e_all = pd.to_numeric(mdf[ce], errors="coerce").to_numpy(float) if ce else np.full(len(t_all), np.nan)

    sel = np.isfinite(t_all) & np.isfinite(x_all) & (t_all >= -MOTION_PAD_S) & (t_all <= dur + MOTION_PAD_S)
    if sel.sum() < 15:
        raise ValueError(f"only {int(sel.sum())} motion samples between -{MOTION_PAD_S}s and motion_end+{MOTION_PAD_S}s")
    order = np.argsort(t_all[sel])
    t, x = t_all[sel][order], x_all[sel][order]
    vc, ee = v_ctrl[sel][order], e_all[sel][order]

    dt = float(np.median(np.diff(t)))
    if not dt > 0:
        raise ValueError("Time_s is not increasing")
    tg = np.arange(t[0], t[-1], dt)
    xg = np.interp(tg, t, x)
    w = max(7, int(round(MOTION_SMOOTH_S / dt)))
    w += (w % 2 == 0)
    w = min(w, len(tg) - (1 - len(tg) % 2))
    v = savgol_filter(xg, w, 3, deriv=1, delta=dt) * 60.0        # m/s -> m/min
    a = savgol_filter(xg, w, 3, deriv=2, delta=dt) * 3600.0      # m/s^2 -> m/min^2

    x0, x1 = float(np.interp(0.0, t, x)), float(np.interp(dur, t, x))
    disp = x1 - x0
    sign = 1.0 if disp >= 0 else -1.0
    speed, accel = sign * v, sign * a

    inwin = (tg >= 0) & (tg <= dur)
    if not inwin.any():
        raise ValueError("no motion samples inside the motion_start..motion_end window")
    peak = float(np.nanmax(speed[inwin]))
    moving = tg[speed >= MOTION_MOVING_FRAC * peak]
    cruise_mask = inwin & (speed >= MOTION_CRUISE_FRAC * peak)
    cruise = float(np.median(speed[cruise_mask])) if cruise_mask.sum() >= 3 else peak

    win_raw = (t >= 0) & (t <= dur)
    return {
        "tg": tg, "x_m": sign * (xg - x0), "speed": speed, "accel": accel, "dur": dur,
        "t_ctrl": t, "v_ctrl": np.abs(vc),
        "displacement_m": abs(disp),
        "mean_m_min": abs(disp) / dur * 60.0,
        "cruise_m_min": cruise,
        "peak_m_min": peak,
        "accel_max_m_min2": float(np.nanmax(accel[inwin])),
        "decel_max_m_min2": float(-np.nanmin(accel[inwin])),
        "move_onset_s": float(moving[0]) if len(moving) else np.nan,
        "move_end_s": float(moving[-1]) if len(moving) else np.nan,
        "ctrl_peak_m_min": float(np.nanmax(np.abs(vc[win_raw]))) if np.isfinite(vc[win_raw]).any() else np.nan,
        "peak_tracking_err": float(np.nanmax(np.abs(ee[win_raw]))) if np.isfinite(ee[win_raw]).any() else np.nan,
    }


def plot_motion(sample_id, mot):
    dur = mot["dur"]
    fig, axs = plt.subplots(3, 1, figsize=(8, 7.2), sharex=True)
    for a_ in axs:
        a_.axvspan(0, dur, color="tab:blue", alpha=0.10)
        a_.axvline(0, color="gray", ls="--", lw=1.0)
        a_.axvline(dur, color="gray", ls="--", lw=1.0)
    axs[0].plot(mot["tg"], mot["x_m"], color="k", lw=1.6)
    axs[0].set_ylabel("Position from start (m)")
    axs[0].set_title(f"{sample_id}: motion (t = 0 at motion_start, dashed = motion_start / motion_end)")

    axs[1].plot(mot["tg"], mot["speed"], color="tab:blue", lw=1.6, label="From position (smoothed)")
    axs[1].plot(mot["t_ctrl"], mot["v_ctrl"], color="tab:orange", lw=0.9, ls="--", alpha=0.8, label="Controller Velocity_m_min")
    axs[1].axhline(mot["cruise_m_min"], color="tab:red", ls=":", lw=1.2, label=f"Cruise {mot['cruise_m_min']:.3f} {VELOCITY_UNIT}")
    axs[1].set_ylabel(f"Speed ({VELOCITY_UNIT})")
    axs[1].legend(frameon=False, fontsize=8, loc="center right")

    axs[2].plot(mot["tg"], mot["accel"], color="tab:green", lw=1.2)
    axs[2].axhline(0, color="gray", lw=0.8)
    axs[2].set_ylabel(f"Acceleration ({ACCEL_UNIT})")
    axs[2].set_xlabel("Time since motion_start (s)")
    t_stop = mot["move_end_s"] if np.isfinite(mot["move_end_s"]) else dur
    axs[2].set_xlim(0, np.ceil(t_stop * 2) / 2)
    axs[2].text(0.98, 0.95, f"max accel {mot['accel_max_m_min2']:.1f}\nmax decel {mot['decel_max_m_min2']:.1f} {ACCEL_UNIT}",
                transform=axs[2].transAxes, ha="right", va="top", fontsize=9)
    plt.tight_layout()
    fig.savefig(HEATING_OUT_DIR / f"{sample_id}_motion.png", dpi=300)
    if SHOW_PER_SAMPLE:
        plt.show()
    plt.close(fig)


def resolve_velocity(seg_list, ev, sample_id):
    """Fallback: return (duration_s, mean velocity m/min, source_label) from MOTION_DISTANCE_M / duration."""
    if not seg_list:
        return np.nan, np.nan, "no motion events"
    seg = seg_list[0]
    dur = seg["t_end"] - seg["t_start"]
    dist = MOTION_DISTANCE_M.get(sample_id) if isinstance(MOTION_DISTANCE_M, dict) else MOTION_DISTANCE_M
    if dist:
        return dur, float(dist) / dur * 60.0, f"{dist:g} m / duration"
    return dur, np.nan, "duration only (no motion csv; set MOTION_DISTANCE_M)"


# ------------------------------------------------------------------------------
# PLOTTING
# ------------------------------------------------------------------------------
def plot_sample(sample_id, rec, red):
    t, Tall, Tm, tc_cols = rec["t"], rec["T_all"], rec["T_mean"], rec["tc_cols"]
    res, seg = rec["analysis"], rec["segment_rel"]
    t_max = rec["t_100max_s"]

    left = -PRE_ANNEAL_S
    if seg is not None and PRE_ANNEAL_S > 0 and seg[0] > -120:
        left = min(left, seg[0] - 3)
    right = rec["post_s"]

    fig, (ax, ax2) = plt.subplots(
        2, 1, figsize=(8.5, 6.2), sharex=True, gridspec_kw={"height_ratios": [3, 1.4]}
    )
    show = (t >= left) & (t <= right)
    for j, c in enumerate(tc_cols):
        ax.plot(t[show], Tall[show, j], lw=0.9, alpha=0.55, label=c)
    ax.plot(t[show], Tm[show], color="k", lw=2.0, label="Mean (3 TC)")

    if np.isfinite(res["t10"]):
        ax.axvspan(res["t10"], res["t90"], color="orange", alpha=0.18, label="10-90% ramp (rate fit)")
    if np.isfinite(rec["setpoint"]):
        ax.axhline(rec["setpoint"], color="tab:red", ls=":", lw=1.2, label=f"Setpoint ({rec['setpoint']:g} degC)")
    ax.axvline(0, color="gray", ls="--", lw=1.1)
    ax.text(0, ax.get_ylim()[1], " anneal start", va="top", ha="left", fontsize=8, color="gray")
    if np.isfinite(t_max):
        ax.axvline(t_max, color="green", ls="--", lw=1.2, label=f"(100) max (t={t_max:.0f}s)")
    if seg is not None and seg[1] > left:
        ax.axvspan(seg[0], seg[1], color="tab:blue", alpha=0.22, label="Motion")
        for a in (ax, ax2):
            a.axvspan(seg[0], seg[1], color="tab:blue", alpha=0.12)

    r = res["avg_rate_C_per_s"]
    txt = [
        f"Avg heating rate: {r:.2f} degC/s" if np.isfinite(r) else "Avg heating rate: n/a (no ramp)",
        f"Plateau: {res['T_plateau_C']:.1f} degC" if np.isfinite(res["T_plateau_C"]) else "",
    ]
    if np.isfinite(rec["velocity"]):
        txt.append(f"Cruise velocity: {rec['velocity']:.3f} {VELOCITY_UNIT}")
    elif np.isfinite(rec["motion_duration_s"]):
        txt.append(f"Motion: {rec['motion_duration_s']:.2f} s")
    ax.text(0.98, 0.04, "\n".join(s for s in txt if s), transform=ax.transAxes, ha="right", va="bottom",
            fontsize=9)
    ax.set_ylabel("Substrate-holder T (degC)")
    ax.set_title(f"{sample_id}: annealing temperature & motion")
    ax.legend(frameon=False, fontsize=7.5, ncol=2, loc="center right")

    if red is not None and red.get("net_100") is not None:
        tg = np.asarray(red["time_s"], float) + float(rec.get("gi_off_s", ANNEAL_T0_OFFSET_S))
        y = np.asarray(red["net_100"], float)
        ax2.plot(tg, y / max(np.nanmax(y), 1e-9), color="tab:purple", lw=1.4)
    if np.isfinite(t_max):
        ax2.axvline(t_max, color="green", ls="--", lw=1.2)
    ax2.axvline(0, color="gray", ls="--", lw=1.1)
    ax2.set_ylabel("Norm. (100)")
    ax2.set_xlabel("Time since anneal start (s)")
    ax2.set_xlim(left, np.ceil(right * 2) / 2)
    plt.tight_layout()
    fig.savefig(HEATING_OUT_DIR / f"{sample_id}_heating.png", dpi=300)
    if SHOW_PER_SAMPLE:
        plt.show()
    plt.close(fig)


def plot_summary(records, summary):
    ids = list(records)
    has_motion = any(r["motion"] is not None for r in records.values())
    nrow = 4 if has_motion else 3
    fig, axes = plt.subplots(nrow, 1, figsize=(10, 11.5 + (3.5 if has_motion else 0)),
                             gridspec_kw={"height_ratios": [2.4, 1.2, 1.2, 1.6][:nrow]})
    ax = axes[0]
    post = float(np.nanmedian([r["post_s"] for r in records.values()]))
    grid = np.arange(-PRE_ANNEAL_S, post, 0.5)
    stack = []
    for sid, r in records.items():
        ok = np.isfinite(r["T_mean"])
        y = np.interp(grid, r["t"][ok], r["T_mean"][ok], left=None if PRE_ANNEAL_S == 0 else np.nan, right=np.nan)
        stack.append(y)
        ax.plot(grid, y, color="tab:blue", lw=0.8, alpha=0.30)
    stack = np.array(stack)
    mean, sd = np.nanmean(stack, axis=0), np.nanstd(stack, axis=0)
    ax.fill_between(grid, mean - sd, mean + sd, color="tab:red", alpha=0.20, label="+/- 1 SD across runs")
    ax.plot(grid, mean, color="tab:red", lw=2.4, label=f"Average of {len(ids)} runs")
    ax.axvline(0, color="gray", ls="--", lw=1.1)
    ax.set_xlabel("Time since anneal start (s)")
    ax.set_xlim(-PRE_ANNEAL_S, np.ceil(post * 2) / 2)
    ax.set_ylabel("Mean TC temperature (degC)")
    ax.set_title("Annealing temperature: all runs, aligned at anneal start")
    ax.legend(frameon=False)

    x = np.arange(len(ids))
    ax = axes[1]
    rates = summary["avg_rate_C_per_s"].to_numpy(float)
    ax.bar(x, rates, color="tab:orange")
    if np.isfinite(rates).any():
        m, s = np.nanmean(rates), np.nanstd(rates)
        ax.axhline(m, color="k", ls="--", lw=1.2, label=f"mean = {m:.2f} +/- {s:.2f} degC/s")
        ax.legend(frameon=False)
    ax.set_xticks(x, ids, rotation=90, fontsize=8)
    ax.set_ylabel("Avg heating rate (degC/s)")

    ax = axes[2]
    vel = summary["velocity_cruise_m_min"].to_numpy(float)
    if not np.isfinite(vel).any():
        vel = summary["velocity_mean_m_min"].to_numpy(float)
    if np.isfinite(vel).any():
        ax.bar(x, vel, color="tab:blue")
        ax.set_ylabel(f"Cruise velocity ({VELOCITY_UNIT})")
    else:
        ax.bar(x, summary["motion_duration_s"].to_numpy(float), color="tab:blue")
        ax.set_ylabel("Motion duration (s)  [set MOTION_DISTANCE_M for velocity]")
    ax.set_xticks(x, ids, rotation=90, fontsize=8)
    if has_motion:
        ax = axes[3]
        for sid, r in records.items():
            if r["motion"] is not None:
                ax.plot(r["motion"]["tg"], r["motion"]["speed"], lw=0.9, alpha=0.5)
        ax.set_xlabel("Time since motion_start (s)")
        ax.set_ylabel(f"Speed ({VELOCITY_UNIT})")
        t_end = max(np.nanmax([r["motion"]["move_end_s"] for r in records.values() if r["motion"] is not None] + [0.0]), 0.5)
        ax.set_xlim(0, np.ceil(t_end * 2) / 2)
        ax.set_title("Velocity profiles, all runs", fontsize=10)
    plt.tight_layout()
    fig.savefig(HEATING_OUT_DIR / "heating_velocity_summary.png", dpi=300)
    plt.show()
    plt.close(fig)


# ------------------------------------------------------------------------------
# MAIN DRIVER
# ------------------------------------------------------------------------------
def run_heating_velocity():
    if "REDUCED_DATA" not in globals():
        raise NameError("REDUCED_DATA not found. Run Cell 5 first.")
    root = Path(HEATING_ROOT)
    if not root.exists():
        raise FileNotFoundError(f"HEATING_ROOT not found: {root}")

    csv_dirs = {}
    for p in root.rglob("*.csv"):
        if not _is_hidden(p.relative_to(root)):
            csv_dirs.setdefault(p.parent, []).append(p)
    print(f"Scanning {root}  ({sum(len(v) for v in csv_dirs.values())} csv files in {len(csv_dirs)} folders)\n")

    sample_ids = [s for s in REDUCED_DATA if SAMPLES is None or s in SAMPLES]
    records, rows, problems = {}, [], []

    for sid in sample_ids:
        red = REDUCED_DATA[sid]
        try:
            if sid in HEATING_FOLDER_OVERRIDES:
                run_dir, exact, amb = Path(HEATING_FOLDER_OVERRIDES[sid]), True, []
            else:
                run_dir, exact, amb = find_run_dir(sid, root, csv_dirs)
            if run_dir is None:
                problems.append((sid, "no matching folder"))
                continue
            if not exact:
                problems.append((sid, f"approximate folder match -> {run_dir.name} (check!)"))
            if amb:
                problems.append((sid, f"ambiguous folder, also matches: {[a.name for a in amb]} (using {run_dir.name})"))

            csvs = sorted(p for p in run_dir.rglob("*.csv") if not _is_hidden(p.relative_to(run_dir)))
            ev_path, tp_path, mo_path = classify_csvs(csvs)
            if ev_path is None or tp_path is None:
                problems.append((sid, f"could not identify event/temperature csv in {run_dir.name}: {[p.name for p in csvs]}"))
                continue

            ev = load_event_log(ev_path)
            temp, tc_cols = load_temperature(tp_path)
            t0 = ev["anneal_start"]
            if not np.isfinite(t0):
                problems.append((sid, f"no 'anneal start' event found in {ev_path.name} (event column '{ev['event_col']}')"))
                continue
            if ev["experiment_id"] and ev["experiment_id"].lower() != sid.lower():
                problems.append((sid, f"event log says experiment_start = '{ev['experiment_id']}' (folder {run_dir.name}); wrong folder?"))
            if ev["n_anneal_events"] > 1:
                problems.append((sid, f"{ev['n_anneal_events']} anneal-start events; using the first"))
            # --- GIWAXS clock vs heating clock: gi_off = seconds from the anneal event to GIWAXS frame 0 ---
            gi_off, off_src = float(ANNEAL_T0_OFFSET_S), "ANNEAL_T0_OFFSET_S"
            h5_epoch = None
            if ALIGN_TO_H5_UTC:
                _cfgs = globals().get("CONDITIONS", {}) or {}
                _cfg = _cfgs.get(sid) or next((v for k, v in _cfgs.items() if str(k).lower() == str(sid).lower()), None)
                if _cfg and _cfg.get("anneal_master_h5"):
                    h5_epoch, h5_key = h5_start_epoch(_cfg["anneal_master_h5"])
                    if h5_epoch is None:
                        problems.append((sid, f"{Path(_cfg['anneal_master_h5']).name}: no readable UTC start time -> using ANNEAL_T0_OFFSET_S = {ANNEAL_T0_OFFSET_S:g}"))
                    else:
                        raw_off = h5_epoch - t0
                        shift_h = round(raw_off / 3600.0)
                        if abs(raw_off) > MAX_PLAUSIBLE_OFFSET_S and shift_h != 0 and abs(raw_off - shift_h * 3600.0) <= MAX_PLAUSIBLE_OFFSET_S:
                            problems.append((sid, f"h5 start time is {raw_off:+.0f} s from the anneal event = {shift_h:+d} h timezone mismatch -> corrected by {-shift_h:+d} h"))
                            raw_off -= shift_h * 3600.0
                            h5_epoch -= shift_h * 3600.0
                        elif abs(raw_off) > MAX_PLAUSIBLE_OFFSET_S:
                            problems.append((sid, f"h5 start is {raw_off:+.0f} s from the anneal event (more than {MAX_PLAUSIBLE_OFFSET_S:g} s): check the clocks / timezones"))
                        gi_off = raw_off + float(ANNEAL_T0_OFFSET_S)
                        off_src = f"h5 UTC start ({h5_key})"
                else:
                    problems.append((sid, "no anneal .h5 path found in CONDITIONS -> using ANNEAL_T0_OFFSET_S"))
            print(f"[{sid}] GIWAXS frame 0 = anneal event {gi_off:+.2f} s  ({off_src})")

            # timestamp sanity check / whole-hour timezone rescue
            ep = temp["epoch"].to_numpy() - TEMP_TZ_OFFSET_HOURS * 3600.0
            if not (ep.min() <= t0 <= ep.max()):
                fixed = None
                for h in sorted(range(-14, 15), key=abs):
                    if (ep.min() - h * 3600) <= t0 <= (ep.max() - h * 3600):
                        fixed = h
                        break
                if fixed is None:
                    problems.append((sid, "anneal start is outside the temperature data's time range (timestamps don't overlap)"))
                    continue
                ep = ep - fixed * 3600.0
                problems.append((sid, f"temperature timestamps shifted by {-fixed:+d} h to line up with the event log (set TEMP_TZ_OFFSET_HOURS={fixed} to silence)"))

            t = ep - t0
            T_all = temp[tc_cols].to_numpy(float)
            keep = t >= -PRE_ANNEAL_S                  # data starts at anneal_start (t = 0)
            t, T_all = t[keep], T_all[keep]
            T_mean = np.nanmean(T_all, axis=1)

            gt = np.asarray(red["time_s"], float)
            post_s = float(gt[-1] + gi_off) if len(gt) else POST_ANNEAL_S_FALLBACK
            post_s = min(post_s, float(t.max()))
            analysis = analyze_heating(t, T_mean, post_s, PRE_ANNEAL_S)

            fm = red.get("frame_max")
            t_max = float(gt[fm] + gi_off) if fm is not None and len(gt) > fm else np.nan
            T_at_max = float(np.interp(t_max, analysis["tw"], analysis["Ts"])) if np.isfinite(t_max) and len(analysis["tw"]) else np.nan

            dur, vel, vsrc = resolve_velocity(ev["segments"], ev, sid)
            seg_rel = None
            if ev["segments"]:
                s0 = ev["segments"][0]
                seg_rel = (s0["t_start"] - t0, s0["t_end"] - t0)

            mot = None
            if ev["segments"]:
                if mo_path is None:
                    problems.append((sid, "no DATA_RAPID_RUN csv found -> velocity falls back to MOTION_DISTANCE_M / duration"))
                else:
                    try:
                        mot = analyze_motion(_read_csv(mo_path), dur)
                        onset, mend = mot["move_onset_s"], mot["move_end_s"]
                        if not (np.isfinite(onset) and np.isfinite(mend)) or onset > MOTION_SYNC_TOL_S or abs(mend - dur) > MOTION_SYNC_TOL_S:
                            problems.append((sid, f"stage moves {onset:.2f}-{mend:.2f} s after Time_s=0 but events say 0-{dur:.2f} s: clocks may be offset (check)"))
                        if len(ev["segments"]) > 1:
                            problems.append((sid, f"{len(ev['segments'])} motion segments in log; using the first"))
                    except Exception as exc:
                        problems.append((sid, f"motion csv {mo_path.name}: {type(exc).__name__}: {exc}"))
            if mot is not None:
                vel_mean, vel_cruise = mot["mean_m_min"], mot["cruise_m_min"]
                vsrc = f"{mo_path.name} (position derivative)"
            else:
                vel_mean, vel_cruise = vel, np.nan
            g = (lambda k: mot[k]) if mot is not None else (lambda k: np.nan)

            tc_end = T_all[t >= (post_s - PLATEAU_S)] if post_s > PLATEAU_S else T_all
            tc_spread = float(np.nanmean(np.nanmax(tc_end, axis=1) - np.nanmin(tc_end, axis=1))) if len(tc_end) else np.nan

            records[sid] = dict(
                t=t, T_all=T_all, T_mean=T_mean, tc_cols=tc_cols, analysis=analysis, post_s=post_s,
                t_100max_s=t_max, gi_off_s=gi_off, segment_rel=seg_rel, setpoint=ev["anneal_setpoint_C"], velocity=(vel_cruise if np.isfinite(vel_cruise) else vel_mean),
                motion_duration_s=dur, motion=mot,
            )
            rows.append({
                "sample_id": sid,
                "folder": run_dir.name,
                "anneal_start_utc": _fmt_utc(t0),
                "giwaxs_start_minus_anneal_event_s": gi_off,
                "giwaxs_start_utc": _fmt_utc(h5_epoch) if h5_epoch is not None else "n/a",
                "T_start_C": analysis["T_start_C"],
                "setpoint_C": ev["anneal_setpoint_C"],
                "T_plateau_C": analysis["T_plateau_C"],
                "plateau_minus_setpoint_C": analysis["T_plateau_C"] - ev["anneal_setpoint_C"],
                "rise_C": analysis["rise_C"],
                "avg_rate_C_per_s": analysis["avg_rate_C_per_s"],
                "peak_rate_C_per_s": analysis["peak_rate_C_per_s"],
                "t90_s": analysis["t90_s"],
                "T_at_100max_C": T_at_max,
                "t_100max_s": t_max,
                "TC_spread_C": tc_spread,
                "motion_duration_s": dur,
                "shear_h5_duration_s": red.get("shear_duration_s", np.nan),
                "displacement_m": g("displacement_m"),
                "velocity_mean_m_min": vel_mean,
                "velocity_cruise_m_min": vel_cruise,
                "velocity_peak_m_min": g("peak_m_min"),
                "accel_max_m_min2": g("accel_max_m_min2"),
                "decel_max_m_min2": g("decel_max_m_min2"),
                "move_onset_s": g("move_onset_s"),
                "move_end_s": g("move_end_s"),
                "ctrl_peak_velocity_m_min": g("ctrl_peak_m_min"),
                "peak_tracking_err_counts": g("peak_tracking_err"),
                "velocity_source": vsrc,
                "n_motion_segments": len(ev["segments"]),
                "arduino_cmd": ev["arduino_cmd"],
            })
        except Exception as exc:
            problems.append((sid, f"{type(exc).__name__}: {exc}"))

    if not rows:
        print("No samples could be processed:")
        for sid, msg in problems:
            print(f"  [{sid}] {msg}")
        return {}, pd.DataFrame()

    summary = pd.DataFrame(rows)
    for sid in summary["sample_id"]:
        plot_sample(sid, records[sid], REDUCED_DATA[sid])
        if records[sid]["motion"] is not None:
            plot_motion(sid, records[sid]["motion"])
    plot_summary(records, summary)

    summary.to_csv(HEATING_OUT_DIR / "heating_velocity_summary.csv", index=False)
    pd.set_option("display.width", 220, "display.max_columns", 30)
    try:
        display(summary.round(3))
    except NameError:
        print(summary.round(3).to_string())

    r = summary["avg_rate_C_per_s"].dropna()
    if len(r):
        print(f"\nHeating rate across {len(r)} runs: mean {r.mean():.2f} +/- {r.std(ddof=0):.2f} degC/s "
              f"(range {r.min():.2f} to {r.max():.2f}); mean plateau {summary['T_plateau_C'].mean():.1f} degC")
    if problems:
        print("\nNotes / skipped:")
        for sid, msg in problems:
            print(f"  [{sid}] {msg}")
    print(f"\nSaved figures + summary csv -> {HEATING_OUT_DIR}")
    return records, summary


HEATING_DATA, HEATING_SUMMARY = run_heating_velocity()


## Cell 5D: (100) Kinetics Fitting Engine (defines `fit_all_kinetics`)
**What this cell does:** Defines `fit_all_kinetics()`, which Cell 6 calls. For every sample it fits the $\alpha$-$(100)$ peak in each frame from frame 0 up to `frame_max` with the thin-film $\text{LaB}_6$ instrumental broadening (`H_G_inst`, `H_L_inst` from Cell 4) deconvolved, so the fitted widths are sample-only. Per frame it returns the integrated area, peak position $q_0$ and Scherrer coherence length $L_c$ ($K = 0.9$), saving `outputs/<SampleID>_100_kinetics.csv`. Frames before the peak appears (< 4% of the strongest frame's prominence) are left blank.

Run once after Cell 4; it only *defines* the function (no fitting happens here).

In [ ]:
# ==============================================================================
# CELL 5D: (100) KINETICS FITTING ENGINE  ->  defines fit_all_kinetics()
# Run AFTER Cell 2 (peak models) and Cell 4 (CALIB_DATA), BEFORE Cell 6.
#
# For every sample in REDUCED_DATA, fits the alpha-(100) peak in each frame from
# frame 0 up to frame_max, using the thin-film LaB6 instrumental broadening
# (H_G_inst, H_L_inst from CALIB_DATA) so the fitted widths are SAMPLE-ONLY widths.
#
# Returns {sample_id: DataFrame} with one row per frame (NaN where no peak /
# no fit, and after the fitted range) and the columns Cell 6 reads:
#     time_s, area_100, q0_100, Lc_nm
# plus: frame, fwhm_sample_Ainv, eta_sample, fwhm_total_Ainv, n_peaks,
#       area_delta, q0_delta, r2
# and saves outputs/<SampleID>_100_kinetics.csv  (…_100_kinetics_isolated.csv when
# enable_second_peak=True).
# ==============================================================================
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

# SCHERRER_K, KINETICS_MIN_POINTS and KINETICS_MIN_PROM_FRAC come from the shared helper cell.


def fit_all_kinetics(
    reduced_data,
    calib_data,
    q_100_window=(0.96, 1.05),
    q_delta_window=(0.82, 0.96),
    enable_second_peak=False,
    scherrer_k=SCHERRER_K,
    frames_after_max=0,
    save_csv=True,
    verbose=True,
):
    for key in ("H_G_inst", "H_L_inst"):
        if not isinstance(calib_data, dict) or key not in calib_data:
            raise KeyError(f"CALIB_DATA is missing '{key}'. Run Cell 4 (LaB6 calibration) first.")
    HG_inst, HL_inst = float(calib_data["H_G_inst"]), float(calib_data["H_L_inst"])
    q_ref = _q_ref()
    single_model = make_deconvolved_peak_model(HG_inst, HL_inst, q_ref=q_ref)
    double_model = make_double_deconvolved_peak_model(HG_inst, HL_inst, q_ref=q_ref)

    out_dir = Path(globals().get("OUTPUT_DIR", "."))
    suffix = "_100_kinetics_isolated.csv" if enable_second_peak else "_100_kinetics.csv"
    mode = "2-peak isolated mode" if enable_second_peak else "single-peak un-isolated mode"

    tables = {}
    for raw_name, entry in reduced_data.items():
        name = str(raw_name).replace("_reduced", "").strip()
        mat_key = next((k for k in ("I_matrix", "I_1d", "I_qt", "I_q", "I") if k in entry), None)
        if mat_key is None or "q" not in entry:
            print(f"[{name}] skipped: no intensity matrix / q grid in REDUCED_DATA entry.")
            continue

        I = np.asarray(entry[mat_key], dtype=float)
        q = np.asarray(entry["q"], dtype=float)
        n_frames = I.shape[0]
        t = np.asarray(entry.get("time_s", np.arange(n_frames) * globals().get("DT_SEC_FALLBACK", 0.5)), dtype=float)

        # window: alpha-(100) only, or alpha + delta region when isolating a second peak
        q_lo = min(q_100_window[0], q_delta_window[0]) if enable_second_peak else q_100_window[0]
        win = (q >= q_lo) & (q <= q_100_window[1])
        qs = q[win]
        if qs.size < KINETICS_MIN_POINTS:
            print(f"[{name}] skipped: only {qs.size} q-points inside the fit window (need >= {KINETICS_MIN_POINTS}).")
            continue
        sub_all = I[:, win]

        # prominence of the alpha-(100) peak in every frame (screens out pre-nucleation frames)
        a_sel = (qs >= q_100_window[0]) & (qs <= q_100_window[1])
        sa = sub_all[:, a_sel]
        bg_edge = 0.5 * (np.nanmean(sa[:, :3], axis=1) + np.nanmean(sa[:, -3:], axis=1))
        prom = np.nanmax(sa, axis=1) - bg_edge

        f_max = entry.get("frame_max")
        f_max = int(np.nanargmax(prom)) if f_max is None else int(f_max)
        f_end = int(np.clip(f_max + int(frames_after_max), 0, n_frames - 1))
        prom_ref = np.nanmax(prom[: f_end + 1])
        if not np.isfinite(prom_ref) or prom_ref <= 0:
            print(f"[{name}] skipped: no alpha-(100) signal found up to frame {f_end}.")
            continue
        thr = KINETICS_MIN_PROM_FRAC * prom_ref

        cols = {k: np.full(n_frames, np.nan) for k in (
            "area_100", "q0_100", "Lc_nm", "fwhm_sample_Ainv", "eta_sample",
            "fwhm_total_Ainv", "n_peaks", "area_delta", "q0_delta", "r2")}

        for i in range(f_end + 1):
            ys = sub_all[i]
            ok = np.isfinite(ys)
            if prom[i] < thr or ok.sum() < KINETICS_MIN_POINTS:
                continue
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    res = fit_single_or_double_peak(
                        qs[ok], ys[ok], single_model, double_model,
                        q_alpha_bounds=tuple(q_100_window), q_delta_bounds=tuple(q_delta_window),
                        enable_second_peak=enable_second_peak, q_ref=q_ref,
                    )
            except (RuntimeError, ValueError):
                continue

            A, q0, HG_s, HL_s = (float(v) for v in res["main_params"])
            H_s, eta_s = tch_fwhm_and_eta(HG_s, HL_s)                              # sample-only FWHM (A^-1)
            H_tot, _ = tch_fwhm_and_eta(np.sqrt(HG_inst**2 + HG_s**2), HL_inst + HL_s)  # observed FWHM (A^-1)

            y_fit = np.asarray(res["I_total_fit"], dtype=float)
            ss_res = float(np.sum((ys[ok] - y_fit) ** 2))
            ss_tot = float(np.sum((ys[ok] - np.mean(ys[ok])) ** 2))

            cols["area_100"][i] = A
            cols["q0_100"][i] = q0
            cols["fwhm_sample_Ainv"][i] = H_s
            cols["eta_sample"][i] = eta_s
            cols["fwhm_total_Ainv"][i] = H_tot
            cols["Lc_nm"][i] = (2.0 * np.pi * scherrer_k / H_s) * 0.1 if H_s > 0 else np.nan   # Angstrom -> nm
            cols["n_peaks"][i] = res["n_peaks"]
            cols["r2"][i] = 1.0 - ss_res / max(ss_tot, 1e-12)
            if res["delta_params"] is not None:
                cols["area_delta"][i] = float(res["delta_params"][0])
                cols["q0_delta"][i] = float(res["delta_params"][1])

        tbl = pd.DataFrame({"frame": np.arange(n_frames), "time_s": t, **cols})
        tables[name] = tbl

        n_ok = int(np.isfinite(tbl["area_100"]).sum())
        if verbose:
            print(f"[{name}] Fitted {n_ok} frames up to frame {f_end} ({mode}).")
        if save_csv:
            try:
                tbl.to_csv(out_dir / f"{name}{suffix}", index=False)
            except OSError as exc:
                print(f"[{name}] could not save csv: {exc}")

    return tables


## Cell 6: Fit Instrumentally Deconvolved Perovskite $(100)$ Kinetics
**What this cell does:** Fits every sample frame **up to `frame_max`** using the locked thin-film $\text{LaB}_6$ instrumental broadening ($H_{G,\text{inst}}, H_{L,\text{inst}}$) from Cell 4 (`ENABLE_SECONDARY_PEAK = False` by default so you first obtain the clean instrumentally corrected $(100)$ peak fit), saving the results to `outputs/<SampleID>_100_kinetics.csv`.  
**Want to isolate $\alpha$-$(100)$ from an overlapping $\delta$-phase peak afterward?** Run **Cell 6B** right below to apply 2-peak isolation on top of the instrumentally corrected data and compare both results side by side.


In [ ]:
import math
import re
from pathlib import Path
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import norm
import matplotlib.pyplot as plt

# ==============================================================================
# USER SETTINGS FOR CELL 6
# ==============================================================================
_out_dir = Path(globals().get("OUTPUT_DIR", r"H:\17.2 beam stuff\2026_05_Stone\outputs"))

# High-quality PNG export of every Cell 6 figure
SAVE_CELL6_PNGS = True
CELL6_DPI = 300
_C6_DIR = Path(globals().get("EXPORT_PNG_DIR", _out_dir / "exports_png_by_condition")) / "cell6_growth_and_coherence"


def _c6_slug(text):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(text)).strip("_")[:90]


def _c6_save(fig, stem):
    """Save a Cell 6 figure as a high-resolution PNG (tight bbox so the outside legend is never cut off)."""
    if not SAVE_CELL6_PNGS:
        return
    _C6_DIR.mkdir(parents=True, exist_ok=True)
    path = _C6_DIR / f"{_c6_slug(stem)}.png"
    saver = globals().get("_c7_savefig")
    try:
        if callable(saver):
            saver(fig, path, dpi=CELL6_DPI, bbox_inches="tight", facecolor="white")
        else:
            fig.savefig(path, dpi=CELL6_DPI, bbox_inches="tight", facecolor="white")
    except Exception as e:
        print(f"  ! could not save {path.name}: {e}")

# (exclusions, cut-offs, instrument broadening, t_max rule and the (100) fit all come from the shared helper cell)

SERIES_ORDER = list(globals().get("GROUP_ORDER", ["Y", "S", "P", "F"]))   # experiment groups (Cell 1: EXPERIMENT_GROUPS) in plot order
_GROUP_TITLE = {k: v["title"] for k, v in globals().get("EXPERIMENT_GROUPS", {}).items()}
_GROUP_SHORT = {k: v["short"] for k, v in globals().get("EXPERIMENT_GROUPS", {}).items()}
_GROUP_TITLE.update(globals().get("FAMILY_TITLES", {}))     # overview groups (one per letter series)
_GROUP_SHORT.update({k: k for k in globals().get("FAMILY_ORDER", [])})
MAX_CURVES_PER_SERIES_PLOT = 10       # Y (4), P (4), F (9) fit on 1 plot each; S (19) splits into 2 clean S-only plots
STOP_AT_MAX_PEAK = True               # True = show crystallization up to t_max (or cutoff); False = up to cutoff / full scan

# Confidence Interval Settings
CI_LEVEL = 0.95                       # 0.95 -> 95% CI (z = 1.96); set to 0.68 for +/- 1-sigma
SHOW_FIT_CI = True                    # True = show shaded 95% confidence ribbons & summary error bars
SHOW_SERIES_MEAN_CI = False           # Optional: True = overlay dark series mean +/- 95% CI band across each family

_Z_SCORE = float(norm.ppf(0.5 + 0.5 * CI_LEVEL))
_CI_PCT_LABEL = f"{int(round(CI_LEVEL * 100))}% CI"


_get_sample_cutoff = get_sample_cutoff


_is_excluded = is_excluded_sample


def _natural_key(name):
    return [int(tok) if tok.isdigit() else tok.lower() for tok in re.split(r"(\d+)", str(name))]


def _series_label(name):
    """Experiment group (what the sample changes); falls back to the letter prefix if Cell 1 metadata is missing."""
    fn = globals().get("sample_info")
    if callable(fn):
        return fn(name)["group"]
    m = re.match(r"^[A-Za-z]+", str(name).strip())
    return m.group(0)[0].upper() if m else "Other"


def _sample_meta(name):
    fn = globals().get("sample_info")
    return fn(name) if callable(fn) else dict(group=_series_label(name), level=np.nan, label=str(name), rep="A", numeric=False, known=False)




_find_true_alpha100_frame_max = find_alpha100_frame_max








def _compute_frame_uncertainties(q_arr, I_mat, f_cut, fwhm_inst=None, se_fwhm_inst=None, q_win=None, z=1.96, lc_ref_arr=None):
    """95 % CIs from the covariance of the same LaB6-convolved fit used for the values (shared helper)."""
    return frame_uncertainties(q_arr, I_mat, f_cut, q_win=q_win, z=z)


def _estimate_tmax_ci(t_arr, area_arr, area_ci_arr, f_max, f_cut, z=1.96):
    """Estimate 95% CI on t_max from weighted local quadratic apex curvature around f_max, floored at 0.5*dt."""
    t = np.asarray(t_arr, dtype=float)
    y = np.asarray(area_arr, dtype=float)
    dt = float(np.median(np.diff(t))) if len(t) > 1 else 0.5
    min_ci = 0.5 * dt

    i0 = max(0, f_max - 4)
    i1 = min(f_cut + 1, f_max + 5)
    if i1 - i0 < 4:
        return min_ci

    ts = t[i0:i1] - t[f_max]
    ys = y[i0:i1]
    if not np.all(np.isfinite(ts)) or not np.all(np.isfinite(ys)):
        return min_ci

    se = np.asarray(area_ci_arr[i0:i1], dtype=float) / max(z, 1e-6)
    fallback_se = max(float(np.nanmax(ys)) * 0.008, 1e-3)
    se = np.where(np.isfinite(se) & (se > 0), se, fallback_se)
    w = 1.0 / (se ** 2)

    X = np.column_stack([ts ** 2, ts, np.ones_like(ts)])
    Xw = X * np.sqrt(w)[:, None]
    yw = ys * np.sqrt(w)
    try:
        beta, _, _, _ = np.linalg.lstsq(Xw, yw, rcond=None)
        c2, c1, _ = beta
        if c2 >= -1e-9:
            return min_ci
        cov_beta = np.linalg.inv(Xw.T @ Xw)
        grad = np.array([c1 / (c2 ** 2), -1.0 / (2.0 * c2), 0.0])
        se_t = float(np.sqrt(max(grad @ cov_beta @ grad, 0.0)))
        return float(np.clip(z * se_t, min_ci, 8.0 * dt))
    except Exception:
        return min_ci


# ==============================================================================
# 1. VERIFY REDUCED_DATA, EXCLUDE NON-REAL SCANS, & COMPUTE ACCURATE FRAME_MAX
# ==============================================================================
_npz_by_name = {}
for _p in sorted(_out_dir.rglob("*.npz")):
    _clean = _p.stem.replace("_reduced", "")
    if _is_excluded(_clean):
        continue
    _npz_by_name[_clean] = _p

_true_fmax_map = {}
for _k in list(REDUCED_DATA.keys()):
    _clean_k = str(_k).replace("_reduced", "").strip()
    if _is_excluded(_clean_k):
        del REDUCED_DATA[_k]
        continue

    _entry = REDUCED_DATA[_k]
    if _clean_k in _npz_by_name:
        with np.load(_npz_by_name[_clean_k], allow_pickle=False) as _d:
            _ik = next(_x for _x in ("I_matrix", "I_1d", "I_qt", "I_q", "I") if _x in _d.files)
            _clean_mat = _d[_ik]
            for _arr_k, _arr_v in list(_entry.items()):
                if isinstance(_arr_v, np.ndarray) and _arr_k != "q":
                    if _arr_v.ndim == 2 and _arr_v.shape[1] == _clean_mat.shape[1]:
                        _entry[_arr_k] = _clean_mat

    _mat_key = next((_x for _x in ("I_matrix", "I_1d", "I_qt", "I_q", "I") if _x in _entry), None)
    if _mat_key is not None and isinstance(_entry[_mat_key], np.ndarray) and _entry[_mat_key].ndim == 2:
        _n_frames = _entry[_mat_key].shape[0]
        if _n_frames < MIN_FRAMES:
            del REDUCED_DATA[_k]
            continue

        _q_vec = np.asarray(_entry["q"], dtype=float)
        _t_vec = np.asarray(_entry.get("time_s", np.arange(_n_frames) * globals().get("DT_SEC_FALLBACK", 0.5)), dtype=float)
        _t_cut = _get_sample_cutoff(_clean_k)

        _fmax = _find_true_alpha100_frame_max(_q_vec, np.asarray(_entry[_mat_key], dtype=float), t_arr=_t_vec, t_cutoff=_t_cut)
        _fmax = int(np.clip(_fmax, 0, _n_frames - 1))
        _true_fmax_map[_clean_k] = _fmax
        for _fmk in ("frame_max", "max_frame", "peak_frame", "idx_max", "max_idx"):
            if _fmk in _entry:
                _entry[_fmk] = _fmax
        if "I_at_max" in _entry:
            _entry["I_at_max"] = _entry[_mat_key][_fmax]

# ==============================================================================
# 2. COMPUTE FIT_TABLES (SUPPRESSING EMPTY/CRAMMED INTERNAL FIGURES)
# ==============================================================================
_before_figs = set(plt.get_fignums())
plt.ioff()
try:
    FIT_TABLES = fit_all_kinetics(
        REDUCED_DATA,
        CALIB_DATA,
        q_100_window=Q_100_SEARCH_WINDOW,
        q_delta_window=Q_DELTA_SEARCH_WINDOW,
        enable_second_peak=ENABLE_SECONDARY_PEAK,
    )
finally:
    for _fn in set(plt.get_fignums()) - _before_figs:
        plt.close(_fn)
    plt.ion()

# ==============================================================================
# 3. PARSE, PROPAGATE 95% CONFIDENCE INTERVALS, & RENDER SINGLE PLOTS
# ==============================================================================
def _extract_col(tbl, candidates):
    cols = list(tbl.columns) if hasattr(tbl, "columns") else list(tbl.keys())
    for cand in candidates:
        if cand in cols:
            return np.asarray(tbl[cand], dtype=float)
    for cand in candidates:
        for c in cols:
            if cand.lower() in str(c).lower():
                return np.asarray(tbl[c], dtype=float)
    return None


def _clean_lc_series(lc_arr, ci_arr=None, max_jump_ratio=2.2, min_valid_run=3):
    """Remove isolated stray dots and single-frame noise spikes while keeping corresponding CIs aligned."""
    arr = np.asarray(lc_arr, dtype=float).copy()
    ci = np.asarray(ci_arr, dtype=float).copy() if ci_arr is not None else None
    finite = np.isfinite(arr)
    if np.sum(finite) < min_valid_run:
        return np.full_like(arr, np.nan), (np.full_like(ci, np.nan) if ci is not None else None)

    padded = np.concatenate([[False], finite, [False]])
    isolated = finite & (~padded[:-2]) & (~padded[2:])
    arr[isolated] = np.nan
    if ci is not None:
        ci[isolated] = np.nan

    n = len(arr)
    for i in range(n):
        if not np.isfinite(arr[i]):
            continue
        win = arr[max(0, i - 2) : min(n, i + 3)]
        win_f = win[np.isfinite(win)]
        if win_f.size >= 3:
            med = float(np.median(win_f))
            if med > 0 and arr[i] > max_jump_ratio * med:
                arr[i] = np.nan
                if ci is not None:
                    ci[i] = np.nan

    if np.sum(np.isfinite(arr)) < min_valid_run:
        return np.full_like(arr, np.nan), (np.full_like(ci, np.nan) if ci is not None else None)
    return arr, ci


_inst = get_instrument()
_fwhm_inst, _se_fwhm_inst = _inst["H_tot"], _inst["H_tot"] * _inst["rel_err"]
records_by_name = {}

for raw_name, tbl in FIT_TABLES.items():
    clean_name = str(raw_name).replace("_reduced", "").strip()
    if _is_excluded(clean_name):
        continue

    t = _extract_col(tbl, ["time_s", "t_sec", "time", "t", "frame"])
    area = _extract_col(tbl, ["area_100", "area_alpha", "area", "integrated", "amp_100", "amplitude", "intensity"])
    lc_raw = _extract_col(tbl, ["Lc_nm", "L_c_nm", "coherence_length", "Lc", "L_c", "size_nm"])
    q0 = _extract_col(tbl, ["q0_100", "q0", "q_center", "center", "q_100"])

    if t is None or area is None or len(t) == 0:
        continue

    t_cut = _get_sample_cutoff(clean_name)
    if t_cut is not None:
        valid_idx = np.where(t <= t_cut)[0]
        f_cut = int(valid_idx[-1]) if len(valid_idx) > 0 else len(t) - 1
    else:
        f_cut = len(t) - 1

    if clean_name in _true_fmax_map:
        f_max = int(np.clip(_true_fmax_map[clean_name], 0, f_cut))
    else:
        area_search = np.where((np.arange(len(t)) <= f_cut) & np.isfinite(area), area, -np.inf)
        f_max = int(np.argmax(area_search))

    area_ci = np.full(len(t), np.nan)
    q0_ci = np.full(len(t), np.nan)
    lc_ci_raw = np.full(len(t), np.nan)

    red_entry = next(
        (v for k, v in REDUCED_DATA.items() if str(k).replace("_reduced", "").strip().lower() == clean_name.lower()),
        None,
    )
    if SHOW_FIT_CI and isinstance(red_entry, dict) and "q" in red_entry:
        mat_k = next((x for x in ("I_matrix", "I_1d", "I_qt", "I_q", "I") if x in red_entry), None)
        if mat_k is not None:
            a_ci, q_ci, l_ci = _compute_frame_uncertainties(
                red_entry["q"], red_entry[mat_k], f_cut=f_cut,
                fwhm_inst=_fwhm_inst, se_fwhm_inst=_se_fwhm_inst,
                q_win=Q_100_SEARCH_WINDOW, z=_Z_SCORE, lc_ref_arr=lc_raw
            )
            m_len = min(len(t), len(a_ci))
            area_ci[:m_len] = a_ci[:m_len]
            q0_ci[:m_len] = q_ci[:m_len]
            lc_ci_raw[:m_len] = l_ci[:m_len]

    if lc_raw is not None:
        valid_lc_mask = (lc_raw > 0) & (lc_raw <= LC_RES_LIMIT_NM)
        lc_bounded = np.where(valid_lc_mask, lc_raw, np.nan)
        lc_bounded[f_cut + 1 :] = np.nan
        lc_ci_bounded = np.where(valid_lc_mask, lc_ci_raw, np.nan)
        lc_ci_bounded[f_cut + 1 :] = np.nan

        lc_clean, lc_ci_clean = _clean_lc_series(lc_bounded, lc_ci_bounded)
        peak_lc = float(lc_clean[f_max]) if np.isfinite(lc_clean[f_max]) else np.nan
        peak_lc_ci = float(lc_ci_clean[f_max]) if (lc_ci_clean is not None and np.isfinite(lc_ci_clean[f_max])) else np.nan
        if np.isfinite(peak_lc) and np.isfinite(peak_lc_ci):
            peak_lc_ci = min(peak_lc_ci, 0.85 * peak_lc)
    else:
        lc_clean, lc_ci_clean = None, None
        peak_lc, peak_lc_ci = np.nan, np.nan

    t_max_ci = _estimate_tmax_ci(t, area, area_ci, f_max, f_cut, z=_Z_SCORE) if SHOW_FIT_CI else np.nan

    records_by_name[clean_name] = {
        "name": clean_name,
        "series": _series_label(clean_name),
        "family": re.match(r"^[A-Za-z]", clean_name).group(0).upper() if re.match(r"^[A-Za-z]", clean_name) else "?",
        "known": _sample_meta(clean_name)["known"], "level": _sample_meta(clean_name)["level"],
        "numeric": _sample_meta(clean_name)["numeric"], "rep": _sample_meta(clean_name)["rep"],
        "label": (f"{_sample_meta(clean_name)['label']} · {clean_name}" if _sample_meta(clean_name)["known"] else clean_name),
        "t": t,
        "area": area,
        "area_ci": area_ci,
        "lc": lc_clean,
        "lc_ci": lc_ci_clean,
        "q0": q0,
        "q0_ci": q0_ci,
        "f_cut": f_cut,
        "t_cut": t_cut,
        "f_max": f_max,
        "t_max": float(t[f_max]),
        "t_max_ci": t_max_ci,
        "peak_area": float(area[f_max]),
        "peak_lc": peak_lc,
        "peak_lc_ci": peak_lc_ci,
    }

all_series = SERIES_ORDER + [s for s in sorted({r["series"] for r in records_by_name.values()}) if s not in SERIES_ORDER]
grouped_records = {}
ordered_records = []
_records_ext = add_baseline_copies(list(records_by_name.values()), key="series") if callable(globals().get("add_baseline_copies")) else list(records_by_name.values())
for s in all_series:
    s_items = sorted([r for r in _records_ext if r["series"] == s], key=lambda r: _natural_key(r["name"]))
    if s_items:
        if callable(globals().get("assign_group_styles")):
            assign_group_styles(s_items)          # sorts by level; colour = level, B replicate dashed
        else:
            for _i, _r in enumerate(s_items):
                _r["color"], _r["ls"] = plt.get_cmap("tab10")(_i % 10), "-"
        grouped_records[s] = s_items
        ordered_records.extend(r for r in s_items if not r.get("baseline_copy"))   # summaries list each sample once

assert ordered_records, "No valid sample records left after filtering."
print(
    f"Loaded {len(ordered_records)} sample(s) with {_CI_PCT_LABEL} (LaB₆: H_G = {_inst['H_G']:.5f}, H_L = {_inst['H_L']:.5f} Å⁻¹): "
    + ", ".join(f"{k} ({len(v)})" for k, v in grouped_records.items())
)

cmap = plt.get_cmap("tab10")


def _plot_by_series_one(groups, metric_key, ci_key, ylabel, title_prefix, filter_above_thresh=False, clip_lower_zero=False):
    for s_name, s_list in groups.items():
        n_chunks = max(1, math.ceil(len(s_list) / MAX_CURVES_PER_SERIES_PLOT))
        for c_idx in range(n_chunks):
            chunk = s_list[c_idx * MAX_CURVES_PER_SERIES_PLOT : (c_idx + 1) * MAX_CURVES_PER_SERIES_PLOT]
            fig, ax = plt.subplots(figsize=(13.0, 4.8), constrained_layout=True)
            plotted = 0
            interp_curves = []

            for idx_r, rec in enumerate(chunk):
                y_arr = rec[metric_key]
                if y_arr is None or not np.any(np.isfinite(y_arr)):
                    continue

                end_idx = (min(rec["f_max"], rec["f_cut"]) + 1) if STOP_AT_MAX_PEAK else (rec["f_cut"] + 1)
                t_sub = rec["t"][:end_idx]
                y_sub = y_arr[:end_idx].copy()
                ci_sub = rec[ci_key][:end_idx].copy() if (SHOW_FIT_CI and rec.get(ci_key) is not None) else None

                if filter_above_thresh and rec["peak_area"] > 0:
                    a_sub = rec["area"][:end_idx]
                    keep_m = a_sub >= 0.10 * rec["peak_area"]
                    y_sub = np.where(keep_m, y_sub, np.nan)
                    if ci_sub is not None:
                        ci_sub = np.where(keep_m, ci_sub, np.nan)

                if not np.any(np.isfinite(y_sub)):
                    continue

                col = rec.get("color", cmap(idx_r % 10))
                cut_note = f", cut≤{rec['t_cut']:.0f}s" if rec["t_cut"] is not None else ""
                ax.plot(
                    t_sub, y_sub, lw=2.0, color=col, ls=rec.get("ls", "-"),
                    label=f"{rec['label']} (t_max={rec['t_max']:.1f} s{cut_note})"
                )

                if ci_sub is not None and np.any(np.isfinite(ci_sub)):
                    ci_capped = np.where(np.isfinite(ci_sub), np.minimum(ci_sub, 0.90 * np.abs(y_sub)), 0.0)
                    y_lo = y_sub - ci_capped
                    y_hi = y_sub + ci_capped
                    if clip_lower_zero:
                        y_lo = np.maximum(y_lo, 0.0)
                    valid_band = np.isfinite(y_sub) & np.isfinite(ci_sub)
                    ax.fill_between(t_sub, y_lo, y_hi, where=valid_band, color=col, alpha=0.16, linewidth=0)

                interp_curves.append((t_sub, y_sub))
                plotted += 1

            if plotted == 0:
                plt.close(fig)
                continue

            if SHOW_SERIES_MEAN_CI and len(interp_curves) >= 3:
                t_min_all = min(tc[0] for tc, _ in interp_curves)
                t_max_all = max(tc[-1] for tc, _ in interp_curves)
                t_grid = np.linspace(t_min_all, t_max_all, 120)
                mat_interp = np.full((len(interp_curves), len(t_grid)), np.nan)
                for r_i, (tc, yc) in enumerate(interp_curves):
                    fin = np.isfinite(tc) & np.isfinite(yc)
                    if np.sum(fin) >= 2:
                        mat_interp[r_i] = np.interp(t_grid, tc[fin], yc[fin], left=np.nan, right=np.nan)
                n_eff = np.sum(np.isfinite(mat_interp), axis=0)
                mean_curve = np.nanmean(mat_interp, axis=0)
                sem_curve = np.nanstd(mat_interp, axis=0, ddof=1) / np.sqrt(np.maximum(n_eff, 1))
                ci_group = np.where(n_eff >= 2, _Z_SCORE * sem_curve, np.nan)
                ax.plot(t_grid, mean_curve, color="#111111", lw=2.3, ls="--", label=f"{_GROUP_SHORT.get(s_name, s_name)} mean (±{_CI_PCT_LABEL})")
                ax.fill_between(
                    t_grid,
                    np.maximum(mean_curve - ci_group, 0.0) if clip_lower_zero else (mean_curve - ci_group),
                    mean_curve + ci_group,
                    where=np.isfinite(ci_group),
                    color="#111111", alpha=0.12, linewidth=0
                )

            part_tag = f" — Part {c_idx + 1}/{n_chunks}" if n_chunks > 1 else ""
            ci_tag = f" (±{_CI_PCT_LABEL})" if SHOW_FIT_CI else ""
            ax.set_xlabel("Annealing Time (s)", fontsize=11)
            ax.set_ylabel(ylabel, fontsize=11)
            ax.set_title(f"{title_prefix}\n{_GROUP_TITLE.get(s_name, s_name + ' series')}{part_tag}{ci_tag}", fontsize=12, fontweight="bold")
            ax.grid(True, alpha=0.25)
            ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=8.8, frameon=False,
                      ncol=2 if plotted > 14 else 1)   # outside the axes so it never covers data
            _c6_save(fig, f"{title_prefix}_{s_name}{'_part' + str(c_idx + 1) if n_chunks > 1 else ''}")
            plt.show()


# --- 1. α-(100) Crystallization Growth + 95% CI (Grouped by Y, S, P, F) ---
_ov = globals().get("build_overview_groups")
overview_records = _ov(ordered_records, key="series") if callable(_ov) else {}


def _plot_by_series(*args, **kwargs):
    """General (letter-series) overview plots first, then the per-experiment breakdown."""
    if overview_records:
        _plot_by_series_one(overview_records, *args, **kwargs)
    _plot_by_series_one(grouped_records, *args, **kwargs)


_plot_by_series(
    "area",
    "area_ci",
    ylabel="Integrated α-(100) Peak Area (a.u.)",
    title_prefix="α-(100) Crystallization Growth",
    filter_above_thresh=False,
    clip_lower_zero=True,
)

# --- 2. Instrument-Corrected Coherence Length Lc(t) + 95% CI (Grouped by Y, S, P, F) ---
_plot_by_series(
    "lc",
    "lc_ci",
    ylabel="Coherence Length $L_c$ (nm)",
    title_prefix="Coherence Length Evolution $L_c(t)$ (LaB₆-Corrected)",
    filter_above_thresh=True,
    clip_lower_zero=True,
)

# --- 3. α-(100) Peak Center q0(t) + 95% CI (Grouped by Y, S, P, F) ---
_plot_by_series(
    "q0",
    "q0_ci",
    ylabel=r"α-(100) Peak Center $q_0$ ($\mathrm{\AA}^{-1}$)",
    title_prefix=r"α-(100) Peak Position $q_0(t)$",
    filter_above_thresh=True,
    clip_lower_zero=False,
)

# --- 4. All-Sample Summary: Instrument-Corrected Coherence Length Lc ± 95% CI (nm) ---
_GROUP_COLORS = {g: plt.get_cmap("tab10")(i % 10) for i, g in enumerate(grouped_records)}   # one colour per experiment
valid_upper = [
    r["peak_lc"] + (r["peak_lc_ci"] if (SHOW_FIT_CI and np.isfinite(r["peak_lc_ci"])) else 0.0)
    for r in ordered_records if np.isfinite(r["peak_lc"])
]
x_max_lc = max(valid_upper) * 1.26 if valid_upper else 500.0

fig_h = max(4.2, 0.38 * len(ordered_records) + 1.5)
fig, ax = plt.subplots(figsize=(10.8, fig_h), constrained_layout=True)
y_pos = np.arange(len(ordered_records))

for idx_r, rec in enumerate(ordered_records):
    col = _GROUP_COLORS.get(rec["series"], "#555555")
    if np.isfinite(rec["peak_lc"]):
        has_ci = SHOW_FIT_CI and np.isfinite(rec["peak_lc_ci"])
        xerr_val = rec["peak_lc_ci"] if has_ci else None
        ax.barh(
            idx_r, rec["peak_lc"], xerr=xerr_val, color=col, height=0.64,
            error_kw={"ecolor": "#111111", "elinewidth": 1.3, "capsize": 3.5}
        )
        txt_x = rec["peak_lc"] + (rec["peak_lc_ci"] if has_ci else 0.0) + x_max_lc * 0.012
        lbl_str = f"{rec['peak_lc']:.1f} ± {rec['peak_lc_ci']:.1f} nm" if has_ci else f"{rec['peak_lc']:.1f} nm"
        ax.text(txt_x, idx_r, lbl_str, va="center", ha="left", fontsize=8.6, fontweight="bold", color="#111111")
    else:
        ax.barh(idx_r, x_max_lc * 0.82, color="none", edgecolor=col, hatch="//", lw=1.1, height=0.60, alpha=0.65)
        ax.text(
            x_max_lc * 0.02, idx_r, "Resolution-limited (FWHM ≤ LaB₆ inst. limit)",
            va="center", ha="left", fontsize=8.5, style="italic", color="#333333"
        )

ax.set_yticks(y_pos)
ax.set_yticklabels([r["label"] for r in ordered_records], fontsize=9.2)
ax.invert_yaxis()
ax.set_xlim(0, x_max_lc)
ax.set_xlabel(f"Instrument-Corrected Coherence Length $L_c$ at Max (100) (nm, ±{_CI_PCT_LABEL})", fontsize=11)
ax.set_title(f"Summary: Coherence Length $L_c$ ± {_CI_PCT_LABEL} Across All Samples (Grouped by Experiment)", fontsize=12, fontweight="bold")
ax.grid(True, axis="x", alpha=0.25)
_c6_save(fig, "Summary_coherence_length_Lc_all_samples")
globals().setdefault("KEY_FIGURES", {})["Summary_coherence_length_Lc"] = fig
plt.show()

# --- 5. All-Sample Summary: Time to Max (100) Peak (t_max ± 95% CI) ---
fig, ax = plt.subplots(figsize=(10.8, fig_h), constrained_layout=True)
t_upper = [
    r["t_max"] + (r["t_max_ci"] if (SHOW_FIT_CI and np.isfinite(r["t_max_ci"])) else 0.0)
    for r in ordered_records
]
x_max_t = max(max(t_upper) * 1.22, 1.0)

for idx_r, rec in enumerate(ordered_records):
    col = _GROUP_COLORS.get(rec["series"], "#2e7d32")
    has_ci = SHOW_FIT_CI and np.isfinite(rec["t_max_ci"])
    xerr_val = rec["t_max_ci"] if has_ci else None
    ax.barh(
        idx_r, rec["t_max"], xerr=xerr_val, color=col, height=0.64,
        error_kw={"ecolor": "#111111", "elinewidth": 1.3, "capsize": 3.5}
    )
    txt_x = rec["t_max"] + (rec["t_max_ci"] if has_ci else 0.0) + x_max_t * 0.012
    lbl_str = f"{rec['t_max']:.1f} ± {rec['t_max_ci']:.1f} s" if has_ci else f"{rec['t_max']:.1f} s"
    ax.text(txt_x, idx_r, lbl_str, va="center", ha="left", fontsize=8.6, fontweight="bold", color="#111111")

ax.set_yticks(y_pos)
ax.set_yticklabels([r["label"] for r in ordered_records], fontsize=9.2)
ax.invert_yaxis()
ax.set_xlim(0, x_max_t)
ax.set_xlabel(f"Time to Max α-(100) Peak, $t_{{\\mathrm{{max}}}}$ (s, ±{_CI_PCT_LABEL})", fontsize=11)
ax.set_title(f"Summary: Time to Peak Crystallinity ($t_{{\\mathrm{{max}}}}$ ± {_CI_PCT_LABEL}) Across All Samples", fontsize=12, fontweight="bold")
ax.grid(True, axis="x", alpha=0.25)
_c6_save(fig, "Summary_time_to_peak_tmax_all_samples")
globals().setdefault("KEY_FIGURES", {})["Summary_time_to_peak_tmax"] = fig
plt.show()


## Cell 6B (Optional): Isolate $\alpha$-$(100)$ from Overlapping $\delta$-Phase Peak (Instrumentally Corrected)
**What this cell does:** Runs the **2-peak isolated fit** (`enable_second_peak=True`, using the exact same $\text{LaB}_6$ instrumental broadening deconvolution from Cell 4) to separate the $\alpha$-$(100)$ peak (`Q_100_SEARCH_WINDOW`) from any overlapping low-$q$ $\delta$-phase / hexagonal polytype shoulder (`Q_DELTA_SEARCH_WINDOW`), saves `outputs/<SampleID>_100_kinetics_isolated.csv`, and compares the isolated vs. un-isolated instrumentally corrected results side by side:
* **Row 1:** Visual fit overlay at each sample's strongest $\delta$-overlap frame.
* **Row 2 & 3:** Time-resolved comparison of **$q_{100}(t)$** and **deconvolved coherence length $L_c(t)$** with vs. without $\delta$-peak isolation.
* **Want to use the isolated fits in Cell 7?** Uncomment `FIT_TABLES = FIT_TABLES_ISOLATED` at the bottom of this cell before running Cell 7.


In [ ]:
import re
from pathlib import Path
import numpy as np
from scipy.optimize import curve_fit
from scipy.ndimage import median_filter, uniform_filter
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

try:
    import fabio
except ImportError:
    fabio = None

try:
    import h5py
except ImportError:
    h5py = None

# ==============================================================================
# SETTINGS — ALIGNED WITH CELL 6 (ANNEAL DATA ONLY)
# ==============================================================================
OUTPUT_DIR = Path(globals().get("OUTPUT_DIR", r"H:\17.2 beam stuff\2026_05_Stone\outputs"))
Q_100_SEARCH_WINDOW = globals().get("Q_100_SEARCH_WINDOW", (0.96, 1.05))
DT_SEC_FALLBACK = globals().get("DT_SEC_FALLBACK", 0.5)

# (exclusions, cut-offs, instrument broadening and the (100) fit come from the shared helper cell)

DEFAULT_ALPHA_I_DEG = 1.0
DEFAULT_BEAM_DX_PX = 0.0   # e.g. set to -310.0 if beam center sits at qr ≈ -0.50 Å⁻¹
DEFAULT_BEAM_DY_PX = 0.0
QR_GRID_RANGE = (-2.25, 2.25)
QZ_GRID_RANGE = (0.0, 2.65)
QR_NBINS = 1500
QZ_NBINS = 883

_trapz = getattr(np, "trapezoid", None) or getattr(np, "trapz")

QRQZ_DATA = {}
globals()["QRQZ_DATA"] = QRQZ_DATA


_is_excluded = is_excluded_sample


def _natural_key(name):
    return [int(tok) if tok.isdigit() else tok.lower() for tok in re.split(r"(\d+)", str(name))]


def _sample_pattern(sample_name):
    esc = re.escape(sample_name)
    return re.compile(rf"(^|[^A-Za-z0-9\-]){esc}(?:$|_(?!try)|(?i:anneal)|(?i:ann)(?![a-z0-9]))", re.IGNORECASE)


_get_sample_cutoff = get_sample_cutoff


def _detect_lab6_fwhm():
    return get_instrument()["H_tot"]


_find_true_alpha100_frame_max = find_alpha100_frame_max




def _fit_single_100_corrected(q, y, q_win=None, f_inst=None, scherrer_k=None):
    """Shared LaB6-convolved (100) fit. f_inst (optional, from the widget) rescales the measured instrument
    widths so their total FWHM equals f_inst; None / measured value = use the calibration unchanged."""
    inst = get_instrument()
    if f_inst is not None and f_inst > 0 and abs(f_inst / inst["H_tot"] - 1.0) > 1e-6:
        s = float(f_inst) / inst["H_tot"]
        inst = dict(inst, H_G=inst["H_G"] * s, H_L=inst["H_L"] * s, H_tot=float(f_inst))
    return fit_alpha100(q, y, q_win=q_win, inst=inst, scherrer_k=scherrer_k, enable_second_peak=False)


# ==============================================================================
# 2D DETECTOR FRAME LOADER (ANNEAL ONLY) & EWALD-SPHERE (qr, qz) MAPPER
# ==============================================================================
def _get_detector_mask(shape):
    calib = globals().get("CALIB_DATA")
    if isinstance(calib, dict):
        for k in ("combined_mask", "total_mask", "mask", "detector_mask"):
            m = calib.get(k)
            if isinstance(m, np.ndarray) and m.shape == shape:
                return m.astype(bool)
    for k in ("COMBINED_MASK", "TOTAL_MASK", "DETECTOR_MASK", "MASK"):
        m = globals().get(k)
        if isinstance(m, np.ndarray) and m.shape == shape:
            return m.astype(bool)
    sat_p = OUTPUT_DIR / "saturation_mask.npy"
    if sat_p.exists():
        m = np.load(sat_p)
        if m.shape == shape:
            return m.astype(bool)
    return None


def _read_frame_from_h5(h5_path, frame_idx):
    if h5py is None:
        return None
    with h5py.File(h5_path, "r") as f:
        if "entry/data" in f:
            grp = f["entry/data"]
            if "data" in grp and grp["data"].ndim == 3:
                ds = grp["data"]
                idx = int(np.clip(frame_idx, 0, ds.shape[0] - 1))
                return np.asarray(ds[idx], dtype=np.float32)
            dkeys = sorted([k for k in grp.keys() if k.startswith("data_")], key=_natural_key)
            offset = 0
            for dk in dkeys:
                ds = grp[dk]
                if ds.ndim == 3:
                    n = ds.shape[0]
                    if offset <= frame_idx < offset + n:
                        return np.asarray(ds[frame_idx - offset], dtype=np.float32)
                    offset += n
        found = []
        f.visititems(lambda _, obj: found.append(obj) if isinstance(obj, h5py.Dataset) and obj.ndim == 3 else None)
        if found:
            ds = found[0]
            idx = int(np.clip(frame_idx, 0, ds.shape[0] - 1))
            return np.asarray(ds[idx], dtype=np.float32)
    return None


def _load_2d_frame_at_tmax(sample_name, npz_path, f_max):
    red = globals().get("REDUCED_DATA")
    entry = None
    if isinstance(red, dict):
        entry = next((v for k, v in red.items() if str(k).replace("_reduced", "").strip().lower() == sample_name.lower()), None)

    if isinstance(entry, dict):
        for k in ("I_2d_max", "img_max", "raw_img_max", "frame_2d_max", "img_at_max", "detector_img"):
            arr = entry.get(k)
            if isinstance(arr, np.ndarray) and arr.ndim == 2:
                return arr.astype(np.float32), "REDUCED_DATA (in-memory)"
        for k in ("I_2d", "frames_2d", "raw_2d", "images"):
            arr = entry.get(k)
            if isinstance(arr, np.ndarray) and arr.ndim == 3:
                return np.asarray(arr[min(f_max, arr.shape[0] - 1)], dtype=np.float32), "REDUCED_DATA (3D stack)"

    npz_recorded_paths = []
    with np.load(npz_path, allow_pickle=False) as d:
        for k in ("I_2d_max", "img_max", "raw_img_max", "frame_2d_max", "img_at_max"):
            if k in d.files and d[k].ndim == 2:
                return np.asarray(d[k], dtype=np.float32), f"{npz_path.name} ({k})"
        for k in ("I_2d", "frames_2d", "raw_2d"):
            if k in d.files and d[k].ndim == 3:
                return np.asarray(d[k][min(f_max, d[k].shape[0] - 1)], dtype=np.float32), f"{npz_path.name} ({k})"
        for k in ("file_path", "raw_path", "h5_path", "master_file", "source_file", "path"):
            if k in d.files:
                val = str(d[k].item() if d[k].ndim == 0 else d[k][0])
                if Path(val).exists() and not _is_excluded(val):
                    npz_recorded_paths.append(Path(val))

    if isinstance(entry, dict):
        for k in ("file_path", "raw_path", "h5_path", "master_file", "source_file", "path"):
            val = entry.get(k)
            if isinstance(val, (str, Path)) and Path(val).exists() and not _is_excluded(val):
                npz_recorded_paths.append(Path(val))
        for k in ("files", "file_list", "frame_files", "source_files"):
            val = entry.get(k)
            if isinstance(val, (list, tuple)) and len(val) > 0:
                sorted_f = sorted(
                    [Path(x) for x in val if Path(x).exists() and not _is_excluded(x)],
                    key=lambda p: _natural_key(p.name)
                )
                if sorted_f:
                    target_p = sorted_f[min(f_max, len(sorted_f) - 1)]
                    if fabio is not None:
                        return np.asarray(fabio.open(str(target_p)).data, dtype=np.float32), target_p.name

    for rp in npz_recorded_paths:
        if rp.is_file():
            if rp.suffix.lower() == ".h5":
                img = _read_frame_from_h5(rp, f_max)
                if img is not None:
                    return img, rp.name
            elif fabio is not None and rp.suffix.lower() in (".tif", ".tiff", ".cbf", ".edf"):
                f_obj = fabio.open(str(rp))
                n_fr = getattr(f_obj, "nframes", 1)
                img = f_obj.getframe(min(f_max, n_fr - 1)).data if n_fr > 1 else f_obj.data
                return np.asarray(img, dtype=np.float32), rp.name

    search_roots = []
    for g_key in ("ANNEAL_DIR", "DATA_DIR", "RAW_DIR", "SCAN_DIR", "IMAGE_DIR", "BEAMLINE_DIR"):
        g_v = globals().get(g_key)
        if isinstance(g_v, (str, Path)) and Path(g_v).exists():
            search_roots.append(Path(g_v))
    search_roots.extend([OUTPUT_DIR, OUTPUT_DIR.parent])

    seen_roots = []
    for r in search_roots:
        rp = r.resolve()
        if rp not in seen_roots and rp.exists():
            seen_roots.append(rp)

    pat = _sample_pattern(sample_name)

    def _anneal_priority(p):
        full_low = str(p).lower()
        stem_low = p.stem.lower()
        if "anneal" in full_low:
            tier = 0
        elif re.search(r"(^|[^a-z])ann([^a-z]|$)", full_low):
            tier = 1
        else:
            tier = 2
        return (tier, 0 if "master" in stem_low else 1, len(p.name))

    for root in seen_roots:
        h5_matches = [
            p for p in root.rglob("*.h5")
            if pat.search(p.stem) and "data_" not in p.stem.lower() and not _is_excluded(p)
        ]
        if h5_matches:
            h5_matches.sort(key=_anneal_priority)
            img = _read_frame_from_h5(h5_matches[0], f_max)
            if img is not None:
                return img, h5_matches[0].name

        if fabio is not None:
            img_matches = [
                p for p in root.rglob("*")
                if p.suffix.lower() in (".tif", ".tiff", ".cbf", ".edf")
                and pat.search(p.stem)
                and "mask" not in p.stem.lower()
                and not _is_excluded(p)
            ]
            if img_matches:
                img_matches.sort(key=lambda p: (_anneal_priority(p)[0], _natural_key(p.name)))
                best_tier = _anneal_priority(img_matches[0])[0]
                tier_matches = [p for p in img_matches if _anneal_priority(p)[0] == best_tier]
                if len(tier_matches) == 1:
                    f_obj = fabio.open(str(tier_matches[0]))
                    n_fr = getattr(f_obj, "nframes", 1)
                    if n_fr > 1:
                        return np.asarray(f_obj.getframe(min(f_max, n_fr - 1)).data, dtype=np.float32), tier_matches[0].name
                    return np.asarray(f_obj.data, dtype=np.float32), tier_matches[0].name
                else:
                    target_p = tier_matches[min(f_max, len(tier_matches) - 1)]
                    return np.asarray(fabio.open(str(target_p)).data, dtype=np.float32), target_p.name

    return None, "Not found"


class EwaldQrQzMapper:
    """Computes exact GIWAXS (qr, qz) maps using pyFAI geometry + optional beam-center pixel offset."""

    def __init__(self):
        self._lut_cache = {}
        self.ai, self.geom_source = self._find_pyfai_ai()
        self.wavelength_A, self.dist_m, self.poni1_m, self.poni2_m, self.px1_m, self.px2_m = self._extract_geom()

    def _find_pyfai_ai(self):
        calib = globals().get("CALIB_DATA")
        if isinstance(calib, dict):
            for k, v in calib.items():
                if hasattr(v, "position_array") or hasattr(v, "calc_pos_zyx"):
                    return v, f"CALIB_DATA['{k}']"
            for k in ("poni_file", "poni_path", "poni", "calib_file", "file"):
                pf = calib.get(k)
                if isinstance(pf, (str, Path)) and Path(pf).exists():
                    try:
                        import pyFAI
                        return pyFAI.load(str(pf)), Path(pf).name
                    except Exception:
                        pass
        for name_g, v in list(globals().items()):
            if hasattr(v, "position_array") or hasattr(v, "calc_pos_zyx"):
                return v, f"global '{name_g}'"
        for root in (OUTPUT_DIR, OUTPUT_DIR.parent):
            if root.exists():
                ponis = sorted(root.rglob("*.poni"))
                if ponis:
                    try:
                        import pyFAI
                        return pyFAI.load(str(ponis[0])), ponis[0].name
                    except Exception:
                        pass
        return None, "CALIB_DATA / fallback"

    def _extract_geom(self):
        if self.ai is not None:
            wl_A = float(self.ai.wavelength) * 1e10 if getattr(self.ai, "wavelength", None) else 0.97625
            return (
                wl_A,
                float(self.ai.dist),
                float(self.ai.poni1),
                float(self.ai.poni2),
                float(self.ai.pixel1),
                float(self.ai.pixel2),
            )
        calib = globals().get("CALIB_DATA") if isinstance(globals().get("CALIB_DATA"), dict) else {}
        wl = float(calib.get("wavelength_A", calib.get("wavelength", 0.97625)))
        if wl < 1e-5:
            wl *= 1e10
        dist = float(calib.get("dist", calib.get("distance", calib.get("sdd", 0.280))))
        if dist > 10.0:
            dist *= 1e-3
        px1 = float(calib.get("pixel1", calib.get("pixel_size", 75e-6)))
        px2 = float(calib.get("pixel2", px1))
        if px1 > 1e-2:
            px1 *= 1e-6
            px2 *= 1e-6
        bx = calib.get("beam_x", calib.get("bc_x", calib.get("x0", 1035.0)))
        by = calib.get("beam_y", calib.get("bc_y", calib.get("y0", 1780.0)))
        poni1 = float(calib.get("poni1", float(by) * px1))
        poni2 = float(calib.get("poni2", float(bx) * px2))
        return wl, dist, poni1, poni2, px1, px2

    def _compute_lab_coords(self, shape):
        ny, nx = shape
        if self.ai is not None and hasattr(self.ai, "position_array"):
            try:
                pos = self.ai.position_array(shape)
                fwd_lab = np.asarray(pos[..., 0], dtype=np.float32)
                down_lab = np.asarray(pos[..., 1], dtype=np.float32)
                right_lab = np.asarray(pos[..., 2], dtype=np.float32)
                return fwd_lab, -down_lab, right_lab
            except Exception:
                pass
        if self.ai is not None and hasattr(self.ai, "calc_pos_zyx"):
            try:
                yy, xx = np.indices((ny, nx), dtype=np.float64)
                z_flat, y_flat, x_flat = self.ai.calc_pos_zyx(d0=None, d1=yy.ravel(), d2=xx.ravel())
                fwd_lab = np.asarray(z_flat, dtype=np.float32).reshape(ny, nx)
                down_lab = np.asarray(y_flat, dtype=np.float32).reshape(ny, nx)
                right_lab = np.asarray(x_flat, dtype=np.float32).reshape(ny, nx)
                return fwd_lab, -down_lab, right_lab
            except Exception:
                pass
        yy, xx = np.indices((ny, nx), dtype=np.float32)
        fwd_lab = np.full((ny, nx), self.dist_m, dtype=np.float32)
        right_lab = xx * self.px2_m - self.poni2_m
        up_lab = self.poni1_m - yy * self.px1_m
        return fwd_lab, up_lab, right_lab

    def _get_lut(self, shape, alpha_i_deg, dx_px=0.0, dy_px=0.0):
        key = (shape, round(float(alpha_i_deg), 4), round(float(dx_px), 2), round(float(dy_px), 2))
        if key in self._lut_cache:
            return self._lut_cache[key]

        alpha_i = np.deg2rad(float(alpha_i_deg))
        k0 = 2.0 * np.pi / self.wavelength_A
        fwd_lab, up_lab, right_lab = self._compute_lab_coords(shape)

        right_lab = right_lab - float(dx_px) * self.px2_m
        up_lab = up_lab + float(dy_px) * self.px1_m

        r_det = np.sqrt(fwd_lab**2 + right_lab**2 + up_lab**2)
        qx = k0 * (right_lab / r_det)
        qy_lab = k0 * (fwd_lab / r_det - 1.0)
        qz_lab = k0 * (up_lab / r_det)

        qy = qy_lab * np.cos(alpha_i) + qz_lab * np.sin(alpha_i)
        qz = -qy_lab * np.sin(alpha_i) + qz_lab * np.cos(alpha_i)
        qr = np.where(qx >= 0, 1.0, -1.0) * np.hypot(qx, qy)

        qr_edges = np.linspace(QR_GRID_RANGE[0], QR_GRID_RANGE[1], QR_NBINS + 1, dtype=np.float32)
        qz_edges = np.linspace(QZ_GRID_RANGE[0], QZ_GRID_RANGE[1], QZ_NBINS + 1, dtype=np.float32)
        qr_c = 0.5 * (qr_edges[:-1] + qr_edges[1:])
        qz_c = 0.5 * (qz_edges[:-1] + qz_edges[1:])

        iz = np.digitize(qz.ravel(), qz_edges) - 1
        ir = np.digitize(qr.ravel(), qr_edges) - 1
        in_bounds = (iz >= 0) & (iz < QZ_NBINS) & (ir >= 0) & (ir < QR_NBINS)
        flat_idx = (iz[in_bounds] * QR_NBINS + ir[in_bounds]).astype(np.int32)

        QR_G, QZ_G = np.meshgrid(qr_c, qz_c)
        sin_af_g = (QZ_G / k0) - np.sin(alpha_i)
        valid_af = np.abs(sin_af_g) <= 1.0
        cos_af_g = np.where(valid_af, np.sqrt(np.maximum(1.0 - sin_af_g**2, 0.0)), np.nan)
        qr_min_wedge = k0 * np.abs(cos_af_g - np.cos(alpha_i))
        ewald_wedge_mask = (~valid_af) | (np.abs(QR_G) < qr_min_wedge)

        lut = {
            "in_bounds": in_bounds,
            "flat_idx": flat_idx,
            "qr": qr_c,
            "qz": qz_c,
            "qr_edges": qr_edges,
            "qz_edges": qz_edges,
            "ewald_wedge_mask": ewald_wedge_mask,
        }
        self._lut_cache[key] = lut
        return lut

    def map_to_qrqz(self, img_2d, alpha_i_deg, dx_px=0.0, dy_px=0.0):
        lut = self._get_lut(img_2d.shape, alpha_i_deg, dx_px=dx_px, dy_px=dy_px)
        w = img_2d.ravel()[lut["in_bounds"]]
        finite = np.isfinite(w)
        f_idx = lut["flat_idx"][finite]
        f_w = w[finite].astype(np.float64)

        n_tot = QZ_NBINS * QR_NBINS
        sums = np.bincount(f_idx, weights=f_w, minlength=n_tot).reshape(QZ_NBINS, QR_NBINS)
        cnts = np.bincount(f_idx, minlength=n_tot).reshape(QZ_NBINS, QR_NBINS)
        with np.errstate(divide="ignore", invalid="ignore"):
            I_qrqz = np.where(cnts > 0, sums / cnts, np.nan).astype(np.float32)

        I_qrqz = np.where(lut["ewald_wedge_mask"], np.nan, I_qrqz)
        return I_qrqz, lut


def _compute_tight_qrqz_bounds(records, pos_qr_only=False, pad_frac=0.015):
    """Tightens (qr, qz) axis limits to the actual non-empty detector footprint."""
    qr_mins, qr_maxs, qz_mins, qz_maxs = [], [], [], []
    for rec in records:
        if rec is None:
            continue
        arr = rec["I_qrqz"]
        qrv = rec["qr"]
        qzv = rec["qz"]
        if pos_qr_only:
            col_sel = qrv >= 0.0
            arr = arr[:, col_sel]
            qrv = qrv[col_sel]
        valid = np.isfinite(arr)
        if not np.any(valid):
            continue
        cols = np.where(np.any(valid, axis=0))[0]
        rows = np.where(np.any(valid, axis=1))[0]
        qr_mins.append(float(qrv[cols[0]]))
        qr_maxs.append(float(qrv[cols[-1]]))
        qz_mins.append(float(qzv[rows[0]]))
        qz_maxs.append(float(qzv[rows[-1]]))

    if not qr_mins:
        return ((0.0, 1.75) if pos_qr_only else (-1.75, 1.75)), (0.0, 2.25)

    r_lo = 0.0 if pos_qr_only else min(qr_mins)
    r_hi = max(qr_maxs)
    z_lo = min(qz_mins)
    z_hi = max(qz_maxs)

    r_pad = pad_frac * max(r_hi - r_lo, 0.1)
    z_pad = pad_frac * max(z_hi - z_lo, 0.1)

    xlim = (0.0 if pos_qr_only else (r_lo - r_pad), r_hi + r_pad)
    ylim = (max(0.0, z_lo - z_pad), z_hi + z_pad)
    return xlim, ylim


def launch_two_dataset_comparator():
    npz_files = [
        p for p in sorted(OUTPUT_DIR.rglob("*.npz"), key=lambda p: _natural_key(p.stem))
        if not _is_excluded(p)
    ]
    assert npz_files, f"No valid anneal .npz files found under {OUTPUT_DIR}."

    scans = {}
    for npz_p in npz_files:
        name = npz_p.stem.replace("_reduced", "").strip()
        if _is_excluded(name):
            continue
        with np.load(npz_p, allow_pickle=False) as d:
            keys = set(d.files)
            q_key = next((k for k in ("q", "q_A_inv", "q_grid", "radial") if k in keys), None)
            I_key = next((k for k in ("I_matrix", "I_1d", "I_qt", "I_q", "I", "intensities", "intensity") if k in keys), None)
            t_key = next((k for k in ("time_s", "t_sec", "times", "time", "t") if k in keys), None)
            if q_key is None or I_key is None:
                continue
            q_arr = np.asarray(d[q_key], dtype=np.float64)
            I_mat = np.asarray(d[I_key], dtype=np.float64)
            if I_mat.ndim != 2 or I_mat.shape[0] < 10:
                continue
            t_arr = (
                np.asarray(d[t_key], dtype=np.float64)
                if t_key is not None
                else resolve_time_axis(name, I_mat.shape[0], np.float64)
            )

            t_cut = _get_sample_cutoff(name)
            if t_cut is not None:
                v_idx = np.where(t_arr <= t_cut)[0]
                f_cut = int(v_idx[-1]) if len(v_idx) > 0 else len(t_arr) - 1
            else:
                f_cut = len(t_arr) - 1

            area_100 = baseline_subtracted_area(q_arr, I_mat, Q_100_SEARCH_WINDOW)
            f_max = _find_true_alpha100_frame_max(q_arr, I_mat, t_arr=t_arr, t_cutoff=t_cut)
            f_max = int(np.clip(f_max, 0, f_cut))

            scans[name] = {
                "npz_path": npz_p,
                "q": q_arr,
                "I_mat": I_mat,
                "time_s": t_arr,
                "area_100": area_100,
                "f_max": f_max,
                "f_cut": f_cut,
                "t_cut": t_cut,
            }

    names = list(scans.keys())
    init_a = "Y2A" if "Y2A" in scans else names[0]
    init_b = "S1A-try2" if "S1A-try2" in scans else (names[1] if len(names) > 1 else names[0])
    q_floor = float(np.nanmin(scans[init_a]["q"]))
    q_ceil = float(np.nanmax(scans[init_a]["q"]))

    mapper = EwaldQrQzMapper()
    det_mask_cache = {}
    raw_2d_cache = {}

    def _get_sample_qrqz(name, alpha_i_deg, dx_px=0.0, dy_px=0.0):
        cache_key = (name, round(float(alpha_i_deg), 4), round(float(dx_px), 2), round(float(dy_px), 2))
        if (
            name in QRQZ_DATA
            and round(float(QRQZ_DATA[name].get("alpha_i_deg", -999)), 4) == cache_key[1]
            and round(float(QRQZ_DATA[name].get("beam_dx_px", -99999)), 2) == cache_key[2]
            and round(float(QRQZ_DATA[name].get("beam_dy_px", -99999)), 2) == cache_key[3]
        ):
            return QRQZ_DATA[name]

        sc = scans[name]
        if name not in raw_2d_cache:
            img, src_label = _load_2d_frame_at_tmax(name, sc["npz_path"], sc["f_max"])
            if img is not None:
                if img.shape not in det_mask_cache:
                    det_mask_cache[img.shape] = _get_detector_mask(img.shape)
                dmask = det_mask_cache[img.shape]
                if dmask is not None:
                    img = np.where(dmask, np.nan, img)
                img = np.where(img < 0, np.nan, img)

                finite_m = np.isfinite(img)
                med_bg = median_filter(np.where(finite_m, img, 0.0), size=3)
                enough_neighbors = uniform_filter(finite_m.astype(np.float32), size=3) > 0.55
                hot_px = finite_m & enough_neighbors & (img > np.maximum(med_bg * 4.0, med_bg + 50.0))
                img = np.where(hot_px, np.nan, img)

            raw_2d_cache[name] = (img, src_label)

        img_2d, src_label = raw_2d_cache[name]
        if img_2d is None:
            return None

        I_qrqz, lut = mapper.map_to_qrqz(img_2d, alpha_i_deg, dx_px=dx_px, dy_px=dy_px)
        record = {
            "I_qrqz": I_qrqz,
            "qr": lut["qr"],
            "qz": lut["qz"],
            "qr_edges": lut["qr_edges"],
            "qz_edges": lut["qz_edges"],
            "ewald_wedge_mask": lut["ewald_wedge_mask"],
            "raw_2d_max": img_2d,
            "f_max": sc["f_max"],
            "t_max": float(sc["time_s"][sc["f_max"]]),
            "alpha_i_deg": float(alpha_i_deg),
            "beam_dx_px": float(dx_px),
            "beam_dy_px": float(dy_px),
            "wavelength_A": mapper.wavelength_A,
            "source_file": src_label,
        }
        QRQZ_DATA[name] = record
        if isinstance(globals().get("REDUCED_DATA"), dict) and name in REDUCED_DATA:
            REDUCED_DATA[name]["I_qrqz_max"] = I_qrqz
            REDUCED_DATA[name]["qr_grid"] = lut["qr"]
            REDUCED_DATA[name]["qz_grid"] = lut["qz"]
            REDUCED_DATA[name]["ewald_wedge_mask"] = lut["ewald_wedge_mask"]
        return record

    # --- Interactive Widgets ---
    dd_a = widgets.Dropdown(options=names, value=init_a, description="Sample A:", layout=widgets.Layout(width="245px"))
    dd_b = widgets.Dropdown(options=names, value=init_b, description="Sample B:", layout=widgets.Layout(width="245px"))
    fwhm_inst_box = widgets.FloatText(
        value=round(_detect_lab6_fwhm(), 5), step=0.0002, description="LaB₆ FWHM:", layout=widgets.Layout(width="180px")
    )
    k_box = widgets.FloatText(value=0.90, step=0.05, description="Scherrer K:", layout=widgets.Layout(width="155px"))
    alpha_box = widgets.FloatText(value=round(DEFAULT_ALPHA_I_DEG, 3), step=0.05, description="αᵢ (deg):", layout=widgets.Layout(width="155px"))

    beam_dx_box = widgets.FloatText(value=DEFAULT_BEAM_DX_PX, step=10.0, description="Beam Δx (px):", layout=widgets.Layout(width="175px"))
    beam_dy_box = widgets.FloatText(value=DEFAULT_BEAM_DY_PX, step=10.0, description="Beam Δy (px):", layout=widgets.Layout(width="175px"))
    geom_info = widgets.HTML(
        value=f"<span style='font-size:11.5px;color:#475569;'>Geom: <b>{mapper.geom_source}</b> | L={mapper.dist_m*1000:.1f} mm, λ={mapper.wavelength_A:.4f} Å</span>"
    )

    q_range_slider = widgets.FloatRangeSlider(
        value=[q_floor, q_ceil], min=q_floor, max=q_ceil, step=0.005,
        description="1D q-Range:", layout=widgets.Layout(width="340px")
    )
    waterfall_slider = widgets.FloatSlider(
        value=0.68, min=0.0, max=2.25, step=0.05,
        description="Waterfall:", layout=widgets.Layout(width="240px")
    )
    norm_chk = widgets.Checkbox(value=True, description="Normalize 1D to (100)", indent=False, layout=widgets.Layout(width="175px"))
    tail_chk = widgets.Checkbox(value=False, description="Show post-max tail", indent=False, layout=widgets.Layout(width="155px"))

    log_2d_chk = widgets.Checkbox(value=True, description="2D Log10 scale", indent=False, layout=widgets.Layout(width="135px"))
    pct_2d_slider = widgets.FloatRangeSlider(
        value=[2.0, 99.4], min=0.0, max=100.0, step=0.2,
        description="2D Contrast %:", layout=widgets.Layout(width="290px")
    )
    pos_qr_btn = widgets.ToggleButton(
        value=False,
        description="Positive qr only (qr ≥ 0)",
        icon="crop",
        tooltip="Show only positive qr ≥ 0 with tight data bounds",
        layout=widgets.Layout(width="195px"),
    )
    cache_all_btn = widgets.Button(description="Cache All QRQZ_DATA", icon="database", layout=widgets.Layout(width="175px"))
    cache_status = widgets.HTML(value="")

    summary_html = widgets.HTML()
    out_plot = widgets.Output()
    kin_cache = {}

    def _get_kinetics(name, f_inst, k_val):
        key = (name, round(f_inst, 6), round(k_val, 3), tail_chk.value)
        if key in kin_cache:
            return kin_cache[key]
        sc = scans[name]
        f_end = (len(sc["time_s"]) - 1) if tail_chk.value else min(sc["f_max"], sc["f_cut"])
        idxs = np.unique(np.linspace(0, f_end, min(45, f_end + 1), dtype=int))
        ts, lcs = [], []
        pk_area = max(float(sc["area_100"][sc["f_max"]]), 1.0)
        for idx in idxs:
            if sc["area_100"][idx] < 0.08 * pk_area:
                continue
            fit = _fit_single_100_corrected(sc["q"], sc["I_mat"][idx], Q_100_SEARCH_WINDOW, f_inst, k_val)
            if fit is not None and np.isfinite(fit["Lc_nm"]) and fit["Lc_nm"] <= 1500.0:
                ts.append(sc["time_s"][idx])
                lcs.append(fit["Lc_nm"])
        res = (np.array(ts), np.array(lcs))
        kin_cache[key] = res
        return res

    def _on_cache_all(_):
        a_deg = float(alpha_box.value)
        dx_p, dy_p = float(beam_dx_box.value), float(beam_dy_box.value)
        n_ok = 0
        for idx_s, s_name in enumerate(names, 1):
            cache_status.value = f"<span style='font-size:12px;color:#1f77b4;'>Caching {s_name} ({idx_s}/{len(names)})...</span>"
            if _get_sample_qrqz(s_name, a_deg, dx_px=dx_p, dy_px=dy_p) is not None:
                n_ok += 1
        cache_status.value = f"<span style='font-size:12px;color:#1b7e3c;font-weight:bold;'>✓ Cached {n_ok}/{len(names)} anneal samples in QRQZ_DATA</span>"

    cache_all_btn.on_click(_on_cache_all)

    def _add_headroom(ax, top_pad_frac=0.25):
        ymin, ymax = ax.get_ylim()
        span = max(ymax - ymin, 1e-6)
        ax.set_ylim(ymin, ymin + (1.0 + top_pad_frac) * span)

    def _plot_2d_array(
        ax, fig, arr_2d, title, x_coords=None, y_coords=None,
        xlabel="Detector X (px)", ylabel="Detector Y (px)", xlim=None, ylim=None
    ):
        if arr_2d is None or not np.any(np.isfinite(arr_2d)):
            ax.text(0.5, 0.5, "2D anneal frame not found on disk\n(set ANNEAL_DIR / RAW_DIR if stored elsewhere)",
                    ha="center", va="center", fontsize=10, color="#555555", transform=ax.transAxes)
            ax.set_title(title, fontsize=11, fontweight="bold")
            return

        disp_arr = np.log10(np.maximum(arr_2d, 0.0) + 1.0) if log_2d_chk.value else arr_2d
        finite_vals = disp_arr[np.isfinite(disp_arr)]
        if finite_vals.size == 0:
            return
        p_lo, p_hi = pct_2d_slider.value
        vmin = float(np.percentile(finite_vals, p_lo))
        vmax = float(np.percentile(finite_vals, max(p_hi, p_lo + 0.2)))
        if vmax <= vmin:
            vmax = vmin + 1.0

        cmap_2d = plt.get_cmap("viridis").copy()
        cmap_2d.set_bad(color="#eef1f5")

        if x_coords is not None and y_coords is not None:
            im = ax.pcolormesh(x_coords, y_coords, disp_arr, shading="auto", cmap=cmap_2d, vmin=vmin, vmax=vmax)
            ax.set_aspect("equal", adjustable="box")
            if xlim is not None:
                ax.set_xlim(xlim)
            if ylim is not None:
                ax.set_ylim(ylim)
        else:
            im = ax.imshow(disp_arr, origin="upper", aspect="equal", cmap=cmap_2d, vmin=vmin, vmax=vmax)

        ax.set_xlabel(xlabel, fontsize=10.5)
        ax.set_ylabel(ylabel, fontsize=10.5)
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.tick_params(labelsize=9.5)
        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
        cb.ax.tick_params(labelsize=9)
        cb.set_label("log₁₀(I+1)" if log_2d_chk.value else "I (a.u.)", fontsize=9.5)

    def _render(_=None):
        name_a, name_b = dd_a.value, dd_b.value
        f_inst = max(float(fwhm_inst_box.value), 0.0)
        k_val = max(float(k_box.value), 0.1)
        alpha_deg = float(alpha_box.value)
        dx_p = float(beam_dx_box.value)
        dy_p = float(beam_dy_box.value)
        q_lo, q_hi = q_range_slider.value
        wf_step = float(waterfall_slider.value)
        pos_only = bool(pos_qr_btn.value)

        sc_a, sc_b = scans[name_a], scans[name_b]
        fa, fb = sc_a["f_max"], sc_b["f_max"]

        fit_a = _fit_single_100_corrected(sc_a["q"], sc_a["I_mat"][fa], Q_100_SEARCH_WINDOW, f_inst, k_val)
        fit_b = _fit_single_100_corrected(sc_b["q"], sc_b["I_mat"][fb], Q_100_SEARCH_WINDOW, f_inst, k_val)
        qrqz_a = _get_sample_qrqz(name_a, alpha_deg, dx_px=dx_p, dy_px=dy_p)
        qrqz_b = _get_sample_qrqz(name_b, alpha_deg, dx_px=dx_p, dy_px=dy_p)

        qr_xlim, qz_ylim = _compute_tight_qrqz_bounds([qrqz_a, qrqz_b], pos_qr_only=pos_only)

        def _fmt_row(lbl, sc, fit, qrqz_rec, color):
            t_pk = sc["time_s"][sc["f_max"]]
            src_txt = qrqz_rec["source_file"] if qrqz_rec is not None else sc["npz_path"].name
            if fit is None:
                return f"<tr><td style='color:{color};font-weight:bold;'>{lbl}</td><td colspan='6'>Fit failed in window</td></tr>"
            if np.isnan(fit["fwhm_corr"]):
                status = "<span style='color:#c62828;font-weight:bold;'>⚠ Resolution-limited (FWHM_obs ≤ LaB₆)</span>"
                fc_str, lc_str = "—", "—"
            else:
                broad_pct = 100.0 * (fit["fwhm_obs"] - fit["fwhm_corr"]) / fit["fwhm_obs"]
                status = f"<span style='color:#1b7e3c;font-weight:bold;'>✓ Valid (−{broad_pct:.1f}% inst. width)</span>"
                fc_str = f"{fit['fwhm_corr']:.5f}"
                lc_str = f"{fit['Lc_nm']:.1f} nm"
            return (
                f"<tr><td style='color:{color};font-weight:bold;'>{lbl}</td>"
                f"<td>{t_pk:.1f} s (fr {sc['f_max']})</td>"
                f"<td>{fit['q0']:.4f} Å⁻¹ (a={fit['a_ang']:.3f} Å)</td>"
                f"<td>{fit['fwhm_obs']:.5f}</td>"
                f"<td><b>{fc_str}</b></td>"
                f"<td><b>{lc_str}</b></td>"
                f"<td>R²={fit['r2']:.4f} | {status} <span style='color:#555;font-size:11px;'>[{src_txt}]</span></td></tr>"
            )

        summary_html.value = (
            "<table style='border-collapse:collapse;font-size:12.5px;width:100%;margin:4px 0;background:#f8fafc;'>"
            "<tr style='background:#e2e8f0;text-align:left;'>"
            "<th style='padding:4px 8px;'>Dataset (Anneal)</th><th>t_max</th><th>α-(100) q₀ & Lattice a</th>"
            "<th>FWHM_obs</th><th>FWHM_sample</th><th>Coherence Lc</th><th>Broadening Check & Source</th></tr>"
            + _fmt_row(f"A: {name_a}", sc_a, fit_a, qrqz_a, "#1f77b4")
            + _fmt_row(f"B: {name_b}", sc_b, fit_b, qrqz_b, "#d95f02")
            + "</table>"
        )

        plt.ioff()
        fig = plt.figure(figsize=(15.5, 20.5), constrained_layout=True)
        gs = GridSpec(4, 2, figure=fig, height_ratios=[1.0, 1.0, 1.45, 1.35])

        ax_fit = fig.add_subplot(gs[0, 0])
        ax_1d  = fig.add_subplot(gs[0, 1])
        ax_kin = fig.add_subplot(gs[1, 0])
        ax_lc  = fig.add_subplot(gs[1, 1])
        ax_imA = fig.add_subplot(gs[2, 0])
        ax_imB = fig.add_subplot(gs[2, 1])
        ax_qqA = fig.add_subplot(gs[3, 0])
        ax_qqB = fig.add_subplot(gs[3, 1])

        # 1. Row 1 Left: Single-Peak Fit + LaB6 Instrumental Width Verification
        for lbl, fit, col, ls in [(name_a, fit_a, "#1f77b4", "-"), (name_b, fit_b, "#d95f02", "--")]:
            if fit is None:
                continue
            scale = (1.0 / max(np.max(fit["y_sub"]), 1e-9)) if norm_chk.value else 1.0
            ax_fit.scatter(fit["q_sub"], fit["y_sub"] * scale, s=16, color=col, alpha=0.65)
            ax_fit.plot(
                fit["q_sub"], fit["y_fit"] * scale, color=col, ls=ls, lw=1.8,
                label=f"{lbl}: FWHM_corr={fit['fwhm_corr']:.4f} Å⁻¹ (Lc={fit['Lc_nm']:.1f} nm)"
            )
        if fit_a is not None and f_inst > 0:
            y_mid = 0.5 * (np.max(fit_a["y_fit"]) / (np.max(fit_a["y_sub"]) if norm_chk.value else 1.0))
            ax_fit.errorbar(
                fit_a["q0"], y_mid, xerr=f_inst / 2.0, fmt="none", ecolor="k", elinewidth=2.2, capsize=4,
                label=f"LaB₆ inst. FWHM = {f_inst:.4f} Å⁻¹"
            )
        ax_fit.set_xlim(Q_100_SEARCH_WINDOW)
        ax_fit.set_xlabel(r"$q$ ($\mathrm{\AA}^{-1}$)", fontsize=10.5)
        ax_fit.set_ylabel("Normalized Intensity" if norm_chk.value else "Intensity (a.u.)", fontsize=10.5)
        ax_fit.set_title("Single-Peak α-(100) Fit & LaB₆ Instrumental Broadening Check", fontsize=11, fontweight="bold")
        ax_fit.grid(True, alpha=0.25)
        _add_headroom(ax_fit, top_pad_frac=0.34)
        ax_fit.legend(loc="upper right", fontsize=9.5, frameon=False)

        # 2. Row 1 Right: 1D Pattern Waterfall Comparison
        traces_1d = []
        for lbl, sc, f_idx, col in [(name_a, sc_a, fa, "#1f77b4"), (name_b, sc_b, fb, "#d95f02")]:
            q = sc["q"]
            mp = (q >= q_lo) & (q <= q_hi)
            y_1d = sc["I_mat"][f_idx, mp].copy()
            if norm_chk.value:
                w100 = (q >= Q_100_SEARCH_WINDOW[0]) & (q <= Q_100_SEARCH_WINDOW[1])
                y_1d = y_1d / max(float(np.nanmax(sc["I_mat"][f_idx, w100])), 1e-9)
            traces_1d.append((lbl, q[mp], y_1d, sc["time_s"][f_idx], col))

        max_amp_1d = max(float(np.nanmax(tr[2])) if tr[2].size > 0 else 1.0 for tr in traces_1d)
        y_shift_b = wf_step if norm_chk.value else (wf_step * max_amp_1d)

        lbl_a, qa, ya, ta, col_a = traces_1d[0]
        ax_1d.plot(qa, ya, color=col_a, lw=1.5, label=f"A: {lbl_a} (t={ta:.1f} s)")

        lbl_b, qb, yb, tb, col_b = traces_1d[1]
        if y_shift_b > 0:
            ax_1d.axhline(y_shift_b, color=col_b, lw=0.8, ls=":", alpha=0.65)
        ax_1d.plot(qb, yb + y_shift_b, color=col_b, lw=1.5, label=f"B: {lbl_b} (t={tb:.1f} s, +{y_shift_b:.2g})")

        ax_1d.set_xlim(q_lo, q_hi)
        ax_1d.set_xlabel(r"$q$ ($\mathrm{\AA}^{-1}$)", fontsize=10.5)
        ax_1d.set_ylabel("Normalized Intensity (waterfall offset)" if norm_chk.value else "Intensity (a.u., waterfall offset)", fontsize=10.5)
        ax_1d.set_title(f"1D Waterfall Comparison at Max (100) ({q_lo:.2f}–{q_hi:.2f} Å⁻¹)", fontsize=11, fontweight="bold")
        ax_1d.grid(True, alpha=0.25)
        _add_headroom(ax_1d, top_pad_frac=0.28)
        ax_1d.legend(loc="upper right", fontsize=9.5, frameon=False)

        # 3. Row 2 Left: α-(100) Crystallization Growth Kinetics
        for lbl, sc, col in [(name_a, sc_a, "#1f77b4"), (name_b, sc_b, "#d95f02")]:
            f_end = (len(sc["time_s"]) - 1) if tail_chk.value else min(sc["f_max"], sc["f_cut"])
            t_sub = sc["time_s"][: f_end + 1]
            a_sub = sc["area_100"][: f_end + 1]
            if norm_chk.value:
                a_sub = (a_sub - a_sub[0]) / max(sc["area_100"][sc["f_max"]] - a_sub[0], 1e-9)
            ax_kin.plot(t_sub, a_sub, color=col, lw=1.8, label=f"{lbl} (t_max={sc['time_s'][sc['f_max']]:.1f} s)")
            t_pk = sc["time_s"][sc["f_max"]]
            y_pk = 1.0 if norm_chk.value else sc["area_100"][sc["f_max"]]
            ax_kin.scatter([t_pk], [y_pk], color=col, s=40, zorder=4)
        ax_kin.set_xlabel("Annealing Time (s)", fontsize=10.5)
        ax_kin.set_ylabel("Normalized α-(100) Growth" if norm_chk.value else "Integrated α-(100) Area (a.u.)", fontsize=10.5)
        ax_kin.set_title("α-(100) Crystallization Kinetics Comparison", fontsize=11, fontweight="bold")
        ax_kin.grid(True, alpha=0.25)
        _add_headroom(ax_kin, top_pad_frac=0.22)
        ax_kin.legend(loc="upper left", fontsize=9.5, frameon=False)

        # 4. Row 2 Right: Instrument-Corrected Coherence Length Evolution Lc(t)
        for lbl, col, mk in [(name_a, "#1f77b4", "o"), (name_b, "#d95f02", "s")]:
            ts, lcs = _get_kinetics(lbl, f_inst, k_val)
            if ts.size > 0:
                ax_lc.plot(ts, lcs, color=col, lw=1.6, marker=mk, markersize=4, label=f"{lbl} (peak Lc={lcs[-1]:.1f} nm)")
        ax_lc.set_xlabel("Annealing Time (s)", fontsize=10.5)
        ax_lc.set_ylabel("Coherence Length $L_c$ (nm)", fontsize=10.5)
        ax_lc.set_title("Instrument-Corrected Coherence Length Evolution $L_c(t)$", fontsize=11, fontweight="bold")
        ax_lc.grid(True, alpha=0.25)
        _add_headroom(ax_lc, top_pad_frac=0.24)
        ax_lc.legend(loc="upper left", fontsize=9.5, frameon=False)

        # 5. Row 3: Large Side-by-Side Original 2D Detector Images at t_max (Anneal)
        src_a_short = qrqz_a["source_file"] if qrqz_a is not None else "missing"
        src_b_short = qrqz_b["source_file"] if qrqz_b is not None else "missing"
        _plot_2d_array(
            ax_imA, fig,
            qrqz_a["raw_2d_max"] if qrqz_a is not None else None,
            title=f"Orig 2D: {name_a} @ t_max={sc_a['time_s'][fa]:.1f} s (fr {fa})\n[{src_a_short}]",
        )
        _plot_2d_array(
            ax_imB, fig,
            qrqz_b["raw_2d_max"] if qrqz_b is not None else None,
            title=f"Orig 2D: {name_b} @ t_max={sc_b['time_s'][fb]:.1f} s (fr {fb})\n[{src_b_short}]",
        )

        # 6. Row 4: Large Side-by-Side qr-qz Maps with Tight Data Bounds & Optional +qr Crop
        qr_mode_tag = " (+qᵣ only)" if pos_only else ""
        _plot_2d_array(
            ax_qqA, fig,
            qrqz_a["I_qrqz"] if qrqz_a is not None else None,
            title=f"$q_r\\text{{–}}q_z${qr_mode_tag}: {name_a} — αᵢ={alpha_deg:.2f}°, t={sc_a['time_s'][fa]:.1f} s",
            x_coords=qrqz_a["qr"] if qrqz_a is not None else None,
            y_coords=qrqz_a["qz"] if qrqz_a is not None else None,
            xlabel=r"$q_r$ ($\mathrm{\AA}^{-1}$)",
            ylabel=r"$q_z$ ($\mathrm{\AA}^{-1}$)",
            xlim=qr_xlim,
            ylim=qz_ylim,
        )
        _plot_2d_array(
            ax_qqB, fig,
            qrqz_b["I_qrqz"] if qrqz_b is not None else None,
            title=f"$q_r\\text{{–}}q_z${qr_mode_tag}: {name_b} — αᵢ={alpha_deg:.2f}°, t={sc_b['time_s'][fb]:.1f} s",
            x_coords=qrqz_b["qr"] if qrqz_b is not None else None,
            y_coords=qrqz_b["qz"] if qrqz_b is not None else None,
            xlabel=r"$q_r$ ($\mathrm{\AA}^{-1}$)",
            ylabel=r"$q_z$ ($\mathrm{\AA}^{-1}$)",
            xlim=qr_xlim,
            ylim=qz_ylim,
        )

        with out_plot:
            clear_output(wait=True)
            display(fig)
        plt.close(fig)

    for w in (
        dd_a, dd_b, fwhm_inst_box, k_box, alpha_box, beam_dx_box, beam_dy_box,
        q_range_slider, waterfall_slider, norm_chk, tail_chk, log_2d_chk, pct_2d_slider, pos_qr_btn
    ):
        w.observe(_render, names="value")

    display(widgets.VBox([
        widgets.HBox([dd_a, dd_b, fwhm_inst_box, k_box, alpha_box]),
        widgets.HBox([beam_dx_box, beam_dy_box, geom_info]),
        widgets.HBox([q_range_slider, waterfall_slider, norm_chk, tail_chk]),
        widgets.HBox([log_2d_chk, pct_2d_slider, pos_qr_btn, cache_all_btn, cache_status]),
        summary_html,
        out_plot,
    ]))
    _render()


launch_two_dataset_comparator()

## Cell 7: Condition-Grouped Comparison Figures — what each plot shows and what to take from it

Samples are grouped by **what they actually change**, not by their letter prefix (the table lives in Cell 1 as `EXPERIMENT_GROUPS` / `SAMPLE_META` — edit it there):

| Experiment (group id) | Samples | Variable |
|---|---|---|
| `Y_comp` | Y1A/B, Y2, Y3A | Cs content 0 / 5 / 15 % (FA/MA shift with it) |
| `P_surfactant` | P1A–P4A | surfactant (4 types) |
| `F_silane` | F1A–F6B | silane end group (I, Br, Cl, amino, cyano, mercapto) |
| `S_speed` | S1–S3 (A/B) | coating speed 6 / 4 / 2 m/min |
| `S_gap` | S4A/B, S6A, S7A | confinement gap 25.4 / 50 / 100 µm |
| `S_substrate` | S8–S10 (A/B) | substrate annealing temp. 150 / 125 / 100 °C |
| `S_superstrate` | S11, S12, S14 (A/B) | superstrate temp. RT / 75 / 50 °C |
| `S_time` | S16–S18 (A/B) | anneal time 5 / 10 / 15 min |

`S15A/B` (glass background) are left out of the perovskite figures. **Replicates share a colour** (A solid, B dashed); for numeric variables the colour runs dark → light with increasing level, so a trend reads as a colour gradient. Legends and tick labels read *level · sample* (e.g. `4 m/min · S2B`). Samples missing from the table fall back to their letter prefix.

**Two levels of grouping.** First come the *general overviews* — one per letter series (`condition_S/` …) — so you can compare everything in a series at a glance: colour = experiment, lighter shade = higher level of the variable that experiment changes, replicate B dashed. Then come the *per-experiment breakdowns* above (`condition_S_speed/` …) where only one variable changes. Y, P and F are each a single experiment, so their overview would be identical to the breakdown and is skipped; set `SKIP_IDENTICAL_OVERVIEWS = False` in Cell 1 to draw all four anyway.

For each experiment Cell 7 draws six figures (plus a seventh for numeric variables), shown inline and saved to `exports_png_by_condition/condition_<group id>/`, plus one stand-alone $q_r$–$q_z$ map and one stand-alone waterfall per sample. Every quantity below is taken at, or up to, **$t_{\max}$** = the frame where the $\alpha$-(100) peak is strongest (before any beam-damage cutoff in `SAMPLE_T_CUTOFFS`).

**Shared definitions**
- **Net $\alpha$-(100) area $A_{100}(t)$** — baseline-subtracted integrated intensity of the perovskite (100) ring (q ≈ 1.0 Å⁻¹). It is a proxy for the *amount* of crystalline perovskite.
- **Conversion $\alpha(t)=\dfrac{A_{100}(t)-A_0}{A_{\max}-A_0}$** — $A_{100}$ rescaled to 0→1 and forced to be monotonic up to $t_{\max}$. $t_{10}, t_{50}, t_{90}$ are the times it crosses 10 / 50 / 90 %.
- **Coherence length $L_c$** — Scherrer size $L_c = 2\pi K/\Delta q$ ($K=0.9$) from the (100) peak width after removing the LaB$_6$ instrumental width. It is the average size of the *ordered* domain along $q$, not the physical grain size.
- **Lattice parameter $a$** — $a = 2\pi/q_{100}$ (cubic cell), from the fitted (100) peak position.
- **Voigt $\eta$** — shape of the fitted peak (0 = Gaussian, 1 = Lorentzian).

### Figure 1 — Speed vs. coherence (`*_01_kinetics_vs_coherence.png`)
- **Left:** each bubble is a sample. x = $t_{50}$ (how fast it crystallises), y = final $L_c$. Colour = lattice parameter $a$; bubble size = peak (100) area; dashed lines = group medians.
- **Right:** x = $\Delta t_{10\rightarrow90}$ (how long the main growth burst lasts), y = $R_{\max}$ = steepest slope of $\alpha(t)$; colour = $L_c$.
- **Takeaway:** left-and-up (small $t_{50}$, large $L_c$) = fast *and* well-ordered. A sample far right on the left panel is a slow crystalliser. In the right panel, high $R_{\max}$ with short $\Delta t$ = abrupt, burst-like nucleation; a long window with low $R_{\max}$ = gradual growth. A colour outlier in the left panel flags a lattice shift (strain or composition change).

### Figure 2 — Nucleation vs. coarsening (`*_02_nucleation_vs_coarsening.png`)
- **Top-left:** $\alpha(t)$ for every sample; dots mark $t_{50}$; dotted lines mark 10 / 50 / 90 %.
- **Bottom-left:** the un-normalised $A_{100}(t)$, out to 10 % beyond the longest $t_{\max}$; dots mark each sample's $t_{\max}$.
- **Right:** $L_c(t)$ from per-frame fits (starting once $\alpha>0.08$); diamonds mark $t_{50}$. One shared legend sits below the panels.
- **Takeaway:** the left panels say *when* material crystallises; the right panel says *how the domains evolve while it does*. If $L_c$ is already large at $t_{50}$ and flat afterwards, order is set at nucleation. If $L_c$ keeps climbing after $t_{50}$, the film is coarsening/ripening. Differences in curve *shape* between conditions matter more than absolute values.

### Figure 3 — Ranked structural metrics (`*_03_ranked_structural_metrics.png`)
Four lollipop panels sharing one sample axis (sorted by $L_c$, largest at top):
1. **$L_c$ (nm)** at $t_{\max}$.
2. **Lattice $a$** drawn as deviation from the group mean (dashed line). Differences are typically only ~0.01 Å — read them as relative, not absolute.
3. **Peak shape $\eta$** — a higher (more Lorentzian) value indicates heavier peak tails, usually more size dispersion / defect broadening; treat it as a qualitative flag.
4. **Residual PbI$_2$** = PbI$_2$(001) area ÷ $\alpha$-(100) area × 100, at $t_{\max}$.
- **Takeaway:** the best film is the one that is long in panel 1, close to the group mean (or to your target) in panel 2, and short in panel 4. Disagreement between panels (large $L_c$ but a lot of PbI$_2$) is the useful information here.

### Figure 4 — Azimuthal orientation (`*_04_azimuthal_orientation.png`)
- **Left:** intensity of the (100) ring versus polar angle $\chi$ measured from the out-of-plane $q_z$ axis, each curve normalised to its own maximum. Shading: grey = 0–6°, the region near $q_z$ affected by the Ewald wedge / beamstop (treat with caution), blue = out-of-plane window (6–25°), yellow = in-plane window (65–85°).
- **Right:** Herman's orientation parameter $f=\tfrac12(3\langle\cos^2\chi\rangle-1)$, $\sin\chi$-weighted, per sample (0 = isotropic ring, 1 = fully out-of-plane, negative = in-plane) and the OOP/IP intensity ratio.
- **Takeaway:** a curve that is high at small $\chi$ and decays means the (100) planes stack parallel to the substrate (a face-on, textured film); a flat curve means randomly oriented crystallites. Larger $f$ = stronger texture. This comes from a **single frame** at $t_{\max}$, so small differences in $f$ are within noise.

### Figure 5 — $q_r$–$q_z$ maps (`*_05_qrqz_grid.png` and `qrqz_maps/`)
- Reciprocal-space maps of the positive quadrant at $t_{\max}$, plotted as $\log_{10}(I+1)$ with one shared colour scale per group, so brightness is directly comparable between samples. Titles give $t_{\max}$, $L_c$ and $f$.
- **Takeaway:** a ring at $q\approx1.0$ Å⁻¹ = (100), at 1.41 Å⁻¹ = (110). *Arcs* concentrated near the $q_z$ or $q_r$ axis = oriented crystallites; *uniform rings* = powder-like. Spots or extra rings (e.g. 0.90 Å⁻¹) point to PbI$_2$ or intermediate phases. The map is the 2D picture behind Figure 4's numbers.

### Figure 6 — Full-range 1D patterns and waterfalls (`*_06_1d_waterfall_full_range.png`, `waterfalls_1d/`)
- **Left:** $I(q)$ at $t_{\max}$ for each sample, background-subtracted and divided by its own (100) height, stacked with a vertical offset (sample name and $t_{\max}$ are the y-tick labels). Dotted verticals mark reference peak positions ($\delta$-phase, PbI$_2$, $\alpha$(100)…(210)).
- **Right (one panel per sample):** up to 16 frames from 0 to $t_{\max}$, coloured by time (dark purple → yellow); the black trace is $t_{\max}$.
- **Takeaway:** the left panel is the quickest phase check: peaks that sit on the dotted lines are present, and *relative* peak heights show purity and texture. The waterfalls show the order of events — which phase (precursor, PbI$_2$, $\delta$) appears or disappears before the (100) peak grows, and whether peaks shift or broaden during growth.

### Figure 7 — Outcomes vs. the controlled variable (`*_07_outcomes_vs_variable.png`; numeric experiments only)
- Six small panels ($L_c$, $t_{50}$, $R_{\max}$, PbI$_2$/$\alpha$, Herman $f$, lattice $a$) against the variable that was changed (speed, gap, temperature, time, Cs %). One dot per sample, coloured as in the other figures; the grey line joins the mean of the replicates.
- **Takeaway:** this is the direct dose–response answer. A monotonic trend with the A/B replicates close together is a real effect; replicates that scatter as much as the spread between levels mean the variable is not resolved by this data. Categorical experiments (P, F) have no x-axis, so Fig 3's ranked view plays this role.

**Caveats:** $S$_time (5/10/15 min anneal) only separates the samples in the in-situ data if the scan runs that long; if the scans end after tens of seconds those levels will look like replicates. $t_{\max}$ values, $L_c$, $a$, $\eta$, and $f$ are all single-frame quantities and inherit single-frame noise. $L_c$ is only as good as the LaB$_6$ instrumental width used in `_c7_detect_lab6_fwhm()`.

In [ ]:
import re
from pathlib import Path
import numpy as np
from scipy.optimize import curve_fit
from scipy.ndimage import median_filter, uniform_filter, uniform_filter1d
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

try:
    from IPython.display import display, HTML
except ImportError:
    display = lambda *a, **k: None
    HTML = str

# ==============================================================================
# CELL 7: CONDITION-GROUPED PNG EXPORTS (Y, P, S, F) + 1D WATERFALLS
# ==============================================================================
# Clean plotting defaults: no boxes around legends, no text boxes, uniform fonts.
plt.rcParams.update({"legend.frameon": False, "legend.fontsize": 9.0, "axes.axisbelow": True})
OUTPUT_DIR = Path(globals().get("OUTPUT_DIR", r"H:\17.2 beam stuff\2026_05_Stone\outputs"))
EXPORT_PNG_DIR = OUTPUT_DIR / "exports_png_by_condition"
EXPORT_PNG_DIR.mkdir(parents=True, exist_ok=True)

def _c7_savefig(fig, path, retries=3, wait_s=1.0, **kwargs):
    """fig.savefig that survives transient Windows file-lock / drive errors (OSError) instead of
    killing the whole of Cell 7. Retries, then falls back to '<name>_retry.png', then skips with a warning."""
    import time
    path = Path(path)
    last = None
    for _ in range(retries):
        try:
            fig.savefig(path, **kwargs)
            return path
        except OSError as exc:
            last = exc
            time.sleep(wait_s)
    alt = path.with_name(f"{path.stem}_retry{path.suffix}")
    try:
        fig.savefig(alt, **kwargs)
        print(f"  [warn] could not write {path.name} ({last}); saved as {alt.name} instead. "
              f"Close anything that has the original open (image viewer / preview pane).")
        return alt
    except OSError:
        print(f"  [warn] could not save {path} ({last}); skipped.")
        return None


# Samples are grouped by WHAT THEY CHANGE (EXPERIMENT_GROUPS / SAMPLE_META, defined in Cell 1), not by letter prefix.
# If Cell 1 was not re-run, this falls back to grouping by the first letter.
CONDITION_ORDER = list(globals().get("GROUP_ORDER", ["Y", "P", "S", "F"]))
CONDITION_LABELS = {k: v["title"] for k, v in globals().get("EXPERIMENT_GROUPS", {}).items()} or {
    "Y": "Condition Y", "P": "Condition P", "S": "Condition S", "F": "Condition F"}


CONDITION_LABELS.update(globals().get("FAMILY_TITLES", {}))   # titles of the overview groups


def _c7_info(name):
    fn = globals().get("sample_info")
    if callable(fn):
        return fn(name)
    m = re.match(r"^[A-Za-z]", str(name).strip())
    return dict(group=(m.group(0).upper() if m else "OTHER"), level=np.nan, label=str(name), rep="A", numeric=False, known=False)


def _c7_short(code):
    return globals().get("EXPERIMENT_GROUPS", {}).get(code, {}).get("short", code)

Q_100_SEARCH_WINDOW = globals().get("Q_100_SEARCH_WINDOW", (0.96, 1.05))
DT_SEC_FALLBACK = globals().get("DT_SEC_FALLBACK", 0.5)

# Reference q positions (A^-1) for cubic perovskite & common precursors/polytypes
REF_PEAKS_1D = [
    (0.840, r"$\delta$-phase", "#9333ea"),
    (0.902, r"$\mathrm{PbI}_2$(001)", "#d97706"),
    (1.000, r"$\alpha$(100)", "#0284c7"),
    (1.414, r"$\alpha$(110)", "#0d9488"),
    (1.732, r"$\alpha$(111)", "#059669"),
    (2.000, r"$\alpha$(200)", "#2563eb"),
    (2.236, r"$\alpha$(210)", "#4f46e5"),
]

# (exclusions, cut-offs, instrument broadening, t_max rule, area definition and the (100) fit come from the shared helper cell)
_trapz = getattr(np, "trapezoid", None) or getattr(np, "trapz")


_c7_is_excluded = is_excluded_sample


def _c7_natural_key(name):
    return [int(tok) if tok.isdigit() else tok.lower() for tok in re.split(r"(\d+)", str(name))]


def _c7_condition_code(sample_name):
    return _c7_info(sample_name)["group"]




_c7_find_frame_max = find_alpha100_frame_max




_c7_fit_100 = fit_alpha100


_c7_baseline_subtracted_area = baseline_subtracted_area


def _c7_extract_azimuthal(I_qrqz, qr_arr, qz_arr, q0=1.00, dq_sig=0.025, n_chi=45):
    QR_G, QZ_G = np.meshgrid(qr_arr, qz_arr)
    Q_TOT = np.hypot(QR_G, QZ_G)
    CHI = np.rad2deg(np.arctan2(np.abs(QR_G), np.maximum(QZ_G, 1e-6)))

    sig_mask = (np.abs(Q_TOT - q0) <= dq_sig) & np.isfinite(I_qrqz)
    bg_lo = (Q_TOT >= q0 - 2.4 * dq_sig) & (Q_TOT < q0 - 1.2 * dq_sig) & np.isfinite(I_qrqz)
    bg_hi = (Q_TOT > q0 + 1.2 * dq_sig) & (Q_TOT <= q0 + 2.4 * dq_sig) & np.isfinite(I_qrqz)
    bg_mask = bg_lo | bg_hi

    chi_edges = np.linspace(0.0, 90.0, n_chi + 1)
    chi_c = 0.5 * (chi_edges[:-1] + chi_edges[1:])
    I_chi = np.full(n_chi, np.nan)

    for i in range(n_chi):
        in_bin = (CHI >= chi_edges[i]) & (CHI < chi_edges[i + 1])
        s_vals = I_qrqz[sig_mask & in_bin]
        if s_vals.size < 3:
            continue
        b_vals = I_qrqz[bg_mask & in_bin]
        bg_level = float(np.nanmedian(b_vals)) if b_vals.size >= 3 else 0.0
        I_chi[i] = max(float(np.nanmean(s_vals)) - bg_level, 0.0)

    valid = np.isfinite(I_chi)
    herman_f = np.nan
    if np.sum(valid) >= 5:
        chi_rad = np.deg2rad(chi_c[valid])
        w = I_chi[valid] * np.sin(chi_rad)
        denom = float(np.sum(w))
        if denom > 0:
            cos2_avg = float(np.sum(w * (np.cos(chi_rad) ** 2)) / denom)
            herman_f = 0.5 * (3.0 * cos2_avg - 1.0)

    oop_sel = valid & (chi_c >= 6.0) & (chi_c <= 25.0)
    ip_sel = valid & (chi_c >= 65.0) & (chi_c <= 85.0)
    oop_ip = (
        float(np.nanmean(I_chi[oop_sel]) / max(float(np.nanmean(I_chi[ip_sel])), 1e-6))
        if (np.any(oop_sel) and np.any(ip_sel)) else np.nan
    )
    return chi_c, I_chi, herman_f, oop_ip


def _c7_ensure_qrqz(name, npz_path, f_max):
    qrqz_store = globals().get("QRQZ_DATA")
    if isinstance(qrqz_store, dict) and name in qrqz_store and qrqz_store[name] is not None:
        return qrqz_store[name]

    loader_fn = globals().get("_load_2d_frame_at_tmax")
    mapper_cls = globals().get("EwaldQrQzMapper")
    mask_fn = globals().get("_get_detector_mask")
    if callable(loader_fn) and mapper_cls is not None:
        img, src_label = loader_fn(name, npz_path, f_max)
        if img is not None:
            if callable(mask_fn):
                dmask = mask_fn(img.shape)
                if dmask is not None:
                    img = np.where(dmask, np.nan, img)
            img = np.where(img < 0, np.nan, img)
            finite_m = np.isfinite(img)
            med_bg = median_filter(np.where(finite_m, img, 0.0), size=3)
            enough_nb = uniform_filter(finite_m.astype(np.float32), size=3) > 0.55
            img = np.where(finite_m & enough_nb & (img > np.maximum(med_bg * 4.0, med_bg + 50.0)), np.nan, img)

            mapper = mapper_cls()
            alpha_deg = float(globals().get("DEFAULT_ALPHA_I_DEG", 1.0))
            dx_p = float(globals().get("DEFAULT_BEAM_DX_PX", 0.0))
            dy_p = float(globals().get("DEFAULT_BEAM_DY_PX", 0.0))
            I_qrqz, lut = mapper.map_to_qrqz(img, alpha_deg, dx_px=dx_p, dy_px=dy_p)
            rec = {
                "I_qrqz": I_qrqz,
                "qr": lut["qr"],
                "qz": lut["qz"],
                "raw_2d_max": img,
                "f_max": f_max,
                "alpha_i_deg": alpha_deg,
                "source_file": src_label,
            }
            if isinstance(qrqz_store, dict):
                qrqz_store[name] = rec
            return rec
    return None


def _c7_positive_qrqz_slice(rec2d):
    qr = np.asarray(rec2d["qr"])
    qz = np.asarray(rec2d["qz"])
    arr = np.asarray(rec2d["I_qrqz"])
    qr_m = qr >= 0.0
    qz_m = qz >= 0.0
    return qr[qr_m], qz[qz_m], arr[np.ix_(qz_m, qr_m)]


def _c7_annotate_ref_peaks(ax, q_min, q_max, y_top):
    k = 0
    for q_ref, lbl, col in REF_PEAKS_1D:
        if (q_min + 0.02) <= q_ref <= (q_max - 0.02):
            ax.axvline(q_ref, color=col, ls=":", lw=1.0, alpha=0.65, zorder=1)
            # labels sit left of their line, except the alpha-(100) label (right of its line) so it can
            # never collide with the PbI2(001) label only 0.1 A^-1 away
            ax.text(
                q_ref, y_top, lbl,
                rotation=90, va="top", ha=("left" if abs(q_ref - 1.0) < 0.01 else "right"),
                fontsize=7.8, color=col, fontweight="bold", clip_on=True,
            )
            k += 1


def _c7_plot_single_sample_waterfall(ax, r, n_traces=16, cmap_name="viridis"):
    q_arr = r["q_arr"]
    I_mat = r["I_mat"]
    t_arr = r["t_arr"]
    f_max = r["f_max"]

    valid_q = np.isfinite(q_arr) & np.any(np.isfinite(I_mat[: f_max + 1]), axis=0)
    qv = q_arr[valid_q]
    sub_mat = I_mat[: f_max + 1][:, valid_q]
    sub_t = t_arr[: f_max + 1]

    n_avail = sub_mat.shape[0]
    k = min(n_traces, n_avail)
    sel_idx = np.unique(np.linspace(0, n_avail - 1, k, dtype=int))
    if sel_idx[-1] != n_avail - 1:
        sel_idx = np.append(sel_idx, n_avail - 1)

    traces = sub_mat[sel_idx]
    times = sub_t[sel_idx]

    floors = np.nanpercentile(traces, 2.0, axis=1, keepdims=True)
    traces_zero = np.maximum(traces - floors, 0.0)
    peak_span = max(float(np.nanpercentile(traces_zero[-1], 99.5)), float(np.nanmax(traces_zero)), 1.0)
    offset_step = 0.075 * peak_span

    cmap = plt.get_cmap(cmap_name)
    t_span = max(float(times[-1] - times[0]), 1e-6)

    for j, (y_tr, tv) in enumerate(zip(traces_zero, times)):
        is_last = (j == len(times) - 1)
        c_val = (tv - times[0]) / t_span
        col = "#111827" if is_last else cmap(c_val)
        lw = 1.9 if is_last else 1.15
        alpha = 1.0 if is_last else 0.88
        ax.plot(qv, y_tr + j * offset_step, color=col, lw=lw, alpha=alpha, zorder=2 + j)

    y_ceiling = float(np.nanmax(traces_zero[-1] + (len(times) - 1) * offset_step)) * 1.32
    _c7_annotate_ref_peaks(ax, float(qv[0]), float(qv[-1]), y_ceiling * 0.99)

    ax.set_xlim(float(qv[0]), float(qv[-1]))
    ax.set_ylim(-0.03 * peak_span, y_ceiling)
    ax.set_xlabel(r"Scattering Vector $q$ ($\mathrm{\AA}^{-1}$)", fontsize=10)
    ax.set_ylabel("Offset Intensity $I(q, t)$ (a.u.)", fontsize=10)
    ax.set_title(
        f"{r['label']} — 1D Time Waterfall\n(0 → $t_{{\\max}}$ = {r['t_max']:.1f} s; black = $t_{{\\max}}$)",
        fontsize=10.5, fontweight="bold"
    )
    ax.grid(True, axis="x", alpha=0.22)
    return float(times[0]), float(times[-1]), cmap


def _assign_styles(recs):
    fn = globals().get("assign_group_styles")
    if callable(fn):
        return fn(recs)
    cm = plt.get_cmap("tab10")
    for i, r in enumerate(recs):
        r["color"], r["ls"] = cm(i % 10), "-"
    return recs


def _c7_dose_response(cond_code, cond_title, recs, cond_dir):
    """FIG 7: outcomes vs. the controlled variable (numeric experiments only, >= 2 levels)."""
    g = globals().get("EXPERIMENT_GROUPS", {}).get(cond_code)
    if not g or not g["numeric"] or not all(r["known"] for r in recs):
        return None
    lv = sorted({r["level"] for r in recs})
    if len(lv) < 2:
        return None
    lab = {r["level"]: r["level_label"] for r in recs}
    panels = [("Lc_nm", r"$L_c$ (nm)"), ("t50", r"$t_{50}$ (s)"), ("r_max", r"$R_{\max}$ (s$^{-1}$)"),
              ("pbi2_ratio_pct", r"PbI$_2$/$\alpha$ (%)"), ("herman_f", r"Herman $f$"), ("a_ang", r"Lattice $a$ (Å)")]
    fig, axes = plt.subplots(2, 3, figsize=(14.0, 7.0), constrained_layout=True)
    x_all = np.array([r["level"] for r in recs], float)
    for ax, (key, ylab) in zip(axes.ravel(), panels):
        y_all = np.array([r[key] for r in recs], float)
        means = [(l, np.nanmean(y_all[x_all == l])) for l in lv if np.any(np.isfinite(y_all[x_all == l]))]
        if len(means) > 1:
            ax.plot([m[0] for m in means], [m[1] for m in means], color="#94a3b8", lw=1.4, zorder=1)
        ax.scatter(x_all, y_all, c=[r["color"] for r in recs], s=70, edgecolors="k", linewidths=0.8, zorder=3)
        ax.set_xticks(lv, [lab[l] for l in lv], fontsize=9)
        span = max(lv) - min(lv)
        ax.set_xlim(min(lv) - 0.12 * span, max(lv) + 0.12 * span)
        ax.margins(y=0.15)
        ax.set_ylabel(ylab, fontsize=10.5)
        ax.grid(True, alpha=0.25)
    for ax in axes[1]:
        ax.set_xlabel(g["xlabel"], fontsize=10.5)
    fig.suptitle(f"[{cond_title}] Outcomes vs. {g['xlabel']}  (dot = one sample, grey line = mean of replicates)",
                 fontsize=12.5, fontweight="bold")
    p = cond_dir / f"{cond_code}_07_outcomes_vs_variable.png"
    _c7_savefig(fig, p, dpi=300)
    display(fig)
    plt.close(fig)
    return p


def _c7_export_condition_group(cond_code, grp_records):
    cond_title = CONDITION_LABELS.get(cond_code, f"Condition {cond_code}")
    cond_dir = EXPORT_PNG_DIR / f"condition_{cond_code}"
    qrqz_single_dir = cond_dir / "qrqz_maps"
    wf_single_dir = cond_dir / "waterfalls_1d"
    cond_dir.mkdir(parents=True, exist_ok=True)
    qrqz_single_dir.mkdir(parents=True, exist_ok=True)
    wf_single_dir.mkdir(parents=True, exist_ok=True)

    sample_list_str = ", ".join(r["name"] for r in grp_records)
    display(HTML(
        f"<div style='margin:18px 0 8px 0;padding:8px 14px;background:#1e293b;color:#f8fafc;"
        f"border-radius:6px;font-family:sans-serif;'>"
        f"<span style='font-size:15px;font-weight:bold;'>{cond_title}</span> "
        f"<span style='font-size:12.5px;color:#cbd5e1;'>({len(grp_records)} sample{'s' if len(grp_records)!=1 else ''}: {sample_list_str})</span>"
        f"</div>"
    ))

    cond_short = _c7_short(cond_code)
    colors = {r["name"]: r["color"] for r in grp_records}
    saved_paths = []
    plt.ioff()

    # ==========================================================================
    # PNG 1: KINETICS VS. FINAL COHERENCE & LATTICE PARAMETER QUADRANT PLOT
    # ==========================================================================
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14.5, 6.0), constrained_layout=True)

    t50_arr = np.array([r["t50"] for r in grp_records])
    dt_arr = np.array([r["dt_10_90"] for r in grp_records])
    rmax_arr = np.array([r["r_max"] for r in grp_records])
    lc_arr = np.array([r["Lc_nm"] for r in grp_records])
    a_arr = np.array([r["a_ang"] for r in grp_records])
    area_arr = np.array([r["area_100_max"] for r in grp_records])
    sizes = np.nan_to_num(130.0 + 330.0 * (area_arr / max(float(np.nanmax(area_arr)), 1e-6)), nan=130.0)
    radii_pt = np.sqrt(sizes / np.pi)   # bubble radius in points (scatter s is area in pt^2)

    sc1 = ax1.scatter(t50_arr, lc_arr, s=sizes, c=a_arr, cmap="viridis", edgecolors="k", linewidths=1.1, zorder=4)
    if len(grp_records) > 1 and np.any(np.isfinite(t50_arr)) and np.any(np.isfinite(lc_arr)):
        ax1.axvline(np.nanmedian(t50_arr), color="#64748b", ls="--", lw=1.0, alpha=0.7)
        ax1.axhline(np.nanmedian(lc_arr), color="#64748b", ls="--", lw=1.0, alpha=0.7)
    for idx_r, r in enumerate(grp_records):
        if not (np.isfinite(r["t50"]) and np.isfinite(r["Lc_nm"])):
            continue
        ax1.annotate(
            r["label2"], (r["t50"], r["Lc_nm"]),
            xytext=(radii_pt[idx_r] + 4, 0), textcoords="offset points",
            ha="left", va="center", fontsize=9.5, fontweight="bold", color="#1e293b",
        )
    ax1.margins(0.16)
    cb1 = fig.colorbar(sc1, ax=ax1, pad=0.02)
    cb1.set_label(r"Cubic Lattice Parameter $a$ ($\mathrm{\AA}$)", fontsize=10)
    ax1.set_xlabel(r"Half-Conversion Time $t_{50}$ (s)", fontsize=11)
    ax1.set_ylabel(r"Peak Coherence Length $L_c$ (nm)", fontsize=11)
    ax1.set_title(f"[{cond_title}] Speed vs. Domain Coherence\n(bubble size = peak α-(100) area)", fontsize=11.5, fontweight="bold")
    ax1.grid(True, alpha=0.25)

    sc2 = ax2.scatter(dt_arr, rmax_arr, s=sizes, c=lc_arr, cmap="plasma", edgecolors="k", linewidths=1.1, zorder=4)
    for idx_r, r in enumerate(grp_records):
        if not (np.isfinite(r["dt_10_90"]) and np.isfinite(r["r_max"])):
            continue
        ax2.annotate(
            r["label2"], (r["dt_10_90"], r["r_max"]),
            xytext=(radii_pt[idx_r] + 4, 0), textcoords="offset points",
            ha="left", va="center", fontsize=9.5, fontweight="bold", color="#1e293b",
        )
    ax2.margins(0.16)
    cb2 = fig.colorbar(sc2, ax=ax2, pad=0.02)
    cb2.set_label(r"Peak Coherence Length $L_c$ (nm)", fontsize=10)
    ax2.set_xlabel(r"Growth Window $\Delta t_{10\rightarrow 90}$ (s)", fontsize=11)
    ax2.set_ylabel(r"Max Normalized Growth Rate $R_{\max}$ ($\mathrm{s}^{-1}$)", fontsize=11)
    ax2.set_title(f"[{cond_title}] Growth Window vs. Peak Rate", fontsize=11.5, fontweight="bold")
    ax2.grid(True, alpha=0.25)

    p1 = cond_dir / f"{cond_code}_01_kinetics_vs_coherence.png"
    _c7_savefig(fig, p1, dpi=300)
    globals().setdefault("KEY_FIGURES", {})[f"{cond_code}_growth_window_vs_peak_rate"] = fig
    display(fig)
    plt.close(fig)
    saved_paths.append(p1)

    # ==========================================================================
    # PNG 2: CONVERSION KINETICS, PEAK-CROPPED (+10%) INTENSITY & DOMAIN GROWTH
    # ==========================================================================
    fig = plt.figure(figsize=(15.0, 8.0), constrained_layout=True)
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1.15, 1.0])
    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[1, 0])
    ax_c = fig.add_subplot(gs[:, 1])

    max_t_peak = max(float(r["t_max"]) for r in grp_records)
    x_upper_b = max(1.10 * max_t_peak, 1.0)
    y_max_b_vals = []

    for r in grp_records:
        col = colors[r["name"]]
        ax_a.plot(r["t_grow"], r["alpha_t"], lw=2.0, color=col, ls=r["ls"], label=f"{r['label']} ($t_{{50}}$={r['t50']:.1f} s)")
        ax_a.scatter([r["t50"]], [0.5], color=col, s=42, zorder=5, edgecolors="k", linewidths=0.7)

        # Clip intensity curve at 1.10 * max(t_max) (10% past the longest time-to-peak)
        m_b = r["t_full"] <= x_upper_b
        t_b = r["t_full"][m_b]
        a_b = r["area_full"][m_b]
        ax_b.plot(t_b, a_b, lw=1.8, color=col, ls=r["ls"], label=r["label"])
        ax_b.scatter([r["t_max"]], [r["area_100_max"]], color=col, s=45, zorder=5, edgecolors="k", linewidths=0.7)
        if a_b.size > 0 and np.any(np.isfinite(a_b)):
            y_max_b_vals.append(float(np.nanmax(a_b)))

        if r["t_traj"].size > 0:
            ax_c.plot(r["t_traj"], r["lc_traj"], lw=1.9, color=col, ls=r["ls"], label=r["label"])
            if (np.any(np.isfinite(r["lc_traj"])) and r["t_traj"].size > 1
                    and r["t_traj"][0] <= r["t50"] <= r["t_traj"][-1]):
                lc_at_t50 = float(np.interp(r["t50"], r["t_traj"], r["lc_traj"]))
                ax_c.scatter([r["t50"]], [lc_at_t50], color=col, s=55, marker="D", zorder=6, edgecolors="k", linewidths=0.8)

    for lvl in (0.1, 0.5, 0.9):
        ax_a.axhline(lvl, color="#64748b", ls=":" if lvl != 0.5 else "--", lw=0.95, alpha=0.75)
    ax_a.set_ylim(-0.04, 1.06)
    ax_a.set_xlabel("Annealing Time up to Peak Crystallinity $t_{\\max}$ (s)", fontsize=10.5)
    ax_a.set_ylabel(r"Normalized Conversion $\alpha(t)$", fontsize=10.5)
    ax_a.set_title(f"[{cond_title}] Conversion $\\alpha(t)$ (dots = $t_{{50}}$)", fontsize=11.5, fontweight="bold")
    ax_a.grid(True, alpha=0.25)

    ax_b.axvline(max_t_peak, color="#64748b", ls="--", lw=1.0, alpha=0.75)
    ax_b.set_xlim(0.0, x_upper_b)
    if y_max_b_vals:
        ax_b.set_ylim(0.0, max(y_max_b_vals) * 1.12)
    ax_b.set_xlabel(f"Annealing Time (s) [0 to 1.10 × max $t_{{\\max}}$ = {x_upper_b:.1f} s]", fontsize=10.5)
    ax_b.set_ylabel(r"Net $\alpha$-(100) Area $A_{100}(t)$ (a.u.)", fontsize=10.5)
    ax_b.set_title(f"[{cond_title}] $\\alpha$-(100) Area $A_{{100}}(t)$ (dots = $t_{{\\max}}$)", fontsize=11.5, fontweight="bold")
    ax_b.grid(True, alpha=0.25)

    ax_c.set_xlabel("Annealing Time (s)", fontsize=11)
    ax_c.set_ylabel(r"Coherence Length $L_c(t)$ (nm)", fontsize=11)
    ax_c.set_title(f"[{cond_title}] Domain Growth $L_c(t)$\n(diamonds = $t_{{50}}$; $\\mathrm{{LaB}}_6$-corrected)", fontsize=11.5, fontweight="bold")
    ax_c.grid(True, alpha=0.25)
    ax_c.margins(x=0.04, y=0.08)

    # ONE shared legend below all three panels (never on top of data)
    _h2, _l2 = ax_a.get_legend_handles_labels()
    try:
        fig.legend(_h2, _l2, loc="outside lower center", ncol=min(5, max(1, len(_l2))), fontsize=9.2, frameon=False)
    except (ValueError, TypeError):      # older matplotlib without "outside" placement
        ax_b.legend(_h2, _l2, loc="upper center", bbox_to_anchor=(0.5, -0.22),
                    ncol=min(4, max(1, len(_l2))), fontsize=9.0, frameon=False)

    p2 = cond_dir / f"{cond_code}_02_nucleation_vs_coarsening.png"
    _c7_savefig(fig, p2, dpi=300)
    globals().setdefault("KEY_FIGURES", {})[f"{cond_code}_nucleation_vs_coarsening"] = fig
    display(fig)
    plt.close(fig)
    saved_paths.append(p2)

    # ==========================================================================
    # PNG 3: MULTI-METRIC RANKED STRUCTURAL & PHASE-PURITY LOLLIPOP CHART
    # ==========================================================================
    rec_sorted = sorted(grp_records, key=lambda r: (np.nan_to_num(r["Lc_nm"], nan=-1.0)))
    y_pos = np.arange(len(rec_sorted))
    y_labels = [r["label2"] for r in rec_sorted]

    fig, axes = plt.subplots(1, 4, figsize=(16.5, max(3.6, 0.72 * len(rec_sorted) + 2.0)), sharey=True, constrained_layout=True)

    vals_lc = [r["Lc_nm"] for r in rec_sorted]
    axes[0].hlines(y_pos, 0, vals_lc, color="#1f77b4", lw=2.5)
    axes[0].scatter(vals_lc, y_pos, color="#1f77b4", s=70, zorder=4)
    for yp, v in zip(y_pos, vals_lc):
        if np.isfinite(v):
            axes[0].text(v + 0.02 * max(np.nanmax(vals_lc), 1.0), yp, f"{v:.1f}", va="center", fontsize=9.5, fontweight="bold")
    axes[0].set_xlim(0, max(np.nanmax(vals_lc) * 1.20, 10.0))
    axes[0].set_yticks(y_pos)
    axes[0].set_yticklabels(y_labels, fontsize=9.5, fontweight="bold")
    axes[0].set_xlabel(r"$L_c$ (nm)", fontsize=10.5)
    axes[0].set_title(f"[{cond_short}] Coherence $L_c$", fontsize=11, fontweight="bold")
    axes[0].grid(True, axis="x", alpha=0.25)

    vals_a = np.array([r["a_ang"] for r in rec_sorted])
    a_mean = float(np.nanmean(vals_a))
    axes[1].axvline(a_mean, color="#64748b", ls="--", lw=1.1)
    axes[1].hlines(y_pos, a_mean, vals_a, color="#0d9488", lw=2.5)
    axes[1].scatter(vals_a, y_pos, color="#0d9488", s=70, zorder=4)
    span_a = max(float(np.nanmax(vals_a) - np.nanmin(vals_a)), 0.006)
    axes[1].set_xlim(np.nanmin(vals_a) - 0.75 * span_a, np.nanmax(vals_a) + 0.75 * span_a)
    for yp, v in zip(y_pos, vals_a):
        if np.isfinite(v):   # label on the outer side of each marker so it never sits on the stem
            left = v < a_mean
            axes[1].text(v + (-0.06 if left else 0.06) * span_a, yp, f"{v:.4f}",
                         va="center", ha=("right" if left else "left"), fontsize=9.0)
    axes[1].set_xlabel(r"Lattice $a$ ($\mathrm{\AA}$)", fontsize=10.5)
    axes[1].set_title(f"Lattice Parameter $a$\n(Mean = {a_mean:.4f} Å)", fontsize=11, fontweight="bold")
    axes[1].grid(True, axis="x", alpha=0.25)

    vals_eta = [r["eta"] for r in rec_sorted]
    axes[2].hlines(y_pos, 0, vals_eta, color="#7c3aed", lw=2.5)
    axes[2].scatter(vals_eta, y_pos, color="#7c3aed", s=70, zorder=4)
    axes[2].set_xlim(-0.05, 1.10)
    axes[2].set_xlabel(r"Voigt $\eta$", fontsize=10.5)
    axes[2].set_title("Peak Shape Factor $\\eta$\n(0 = Gaussian, 1 = Lorentzian)", fontsize=11, fontweight="bold")
    for yp, v in zip(y_pos, vals_eta):
        if np.isfinite(v):
            axes[2].text(v + 0.03, yp, f"{v:.2f}", va="center", fontsize=9.5)
    axes[2].grid(True, axis="x", alpha=0.25)

    vals_pbi2 = [r["pbi2_ratio_pct"] for r in rec_sorted]
    axes[3].hlines(y_pos, 0, vals_pbi2, color="#d95f02", lw=2.5)
    axes[3].scatter(vals_pbi2, y_pos, color="#d95f02", s=70, zorder=4)
    for yp, v in zip(y_pos, vals_pbi2):
        if np.isfinite(v):
            axes[3].text(v + 0.02 * max(np.nanmax(vals_pbi2), 1.0), yp, f"{v:.1f}%", va="center", fontsize=9.5)
    axes[3].set_xlim(0, max(np.nanmax(vals_pbi2) * 1.24, 5.0))
    axes[3].set_xlabel(r"$\mathrm{PbI}_2 / \alpha\mathrm{-(100)}$ (%)", fontsize=10.5)
    axes[3].set_title(r"Residual $\mathrm{PbI}_2$ at $t_{\max}$", fontsize=11, fontweight="bold")
    axes[3].grid(True, axis="x", alpha=0.25)

    p3 = cond_dir / f"{cond_code}_03_ranked_structural_metrics.png"
    _c7_savefig(fig, p3, dpi=300)
    globals().setdefault("KEY_FIGURES", {})[f"{cond_code}_ranked_structural_metrics"] = fig
    display(fig)
    plt.close(fig)
    saved_paths.append(p3)

    # ==========================================================================
    # PNG 4: 2D PREFERRED ORIENTATION I(chi) & HERMAN'S ORIENTATION PARAMETER
    # ==========================================================================
    qrqz_valid_recs = [r for r in grp_records if r["I_chi"] is not None and np.any(np.isfinite(r["I_chi"]))]
    if qrqz_valid_recs:
        fig, (ax_az, ax_hf) = plt.subplots(1, 2, figsize=(14.5, 5.6), width_ratios=[1.35, 1.0], constrained_layout=True)

        ax_az.axvspan(0, 6, color="#e2e8f0", alpha=0.65, label="Ewald wedge / beamstop")
        ax_az.axvspan(6, 25, color="#dbeafe", alpha=0.35, label="OOP window (6°–25°)")
        ax_az.axvspan(65, 85, color="#fef3c7", alpha=0.35, label="IP window (65°–85°)")

        for r in qrqz_valid_recs:
            y_az = r["I_chi"].copy()
            ymax_az = float(np.nanmax(y_az))
            if ymax_az > 0:
                y_az = y_az / ymax_az
            hf_lbl = f" (f={r['herman_f']:.2f})" if np.isfinite(r["herman_f"]) else ""
            ax_az.plot(r["chi_c"], y_az, lw=2.0, color=colors[r["name"]], ls=r["ls"], label=f"{r['label']}{hf_lbl}")

        ax_az.set_xlim(0, 90)
        ax_az.set_ylim(-0.03, 1.05)
        ax_az.set_xlabel(r"Polar Angle $\chi$ from Out-of-Plane $q_z$ Axis (deg)", fontsize=11)
        ax_az.set_ylabel(r"Normalized Net $\alpha$-(100) Intensity $I(\chi)$", fontsize=11)
        ax_az.set_title(f"[{cond_title}] Azimuthal Orientation along $\\alpha$-(100)", fontsize=11.5, fontweight="bold")
        ax_az.grid(True, alpha=0.25)
        ax_az.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), fontsize=9.0,
                     ncol=min(3, len(qrqz_valid_recs) + 3), frameon=False)

        hf_sorted = sorted(qrqz_valid_recs, key=lambda r: np.nan_to_num(r["herman_f"], nan=-1.0))
        y_hf = np.arange(len(hf_sorted))
        hf_vals = [r["herman_f"] for r in hf_sorted]
        bar_cols = [colors[r["name"]] for r in hf_sorted]
        ax_hf.barh(y_hf, hf_vals, color=bar_cols, edgecolor="k", linewidth=0.8, height=0.52)
        ax_hf.axvline(0.0, color="k", lw=1.0)
        for yp, r in zip(y_hf, hf_sorted):
            fv = r["herman_f"]
            rv = r["oop_ip"]
            if np.isfinite(fv):
                txt = f" f={fv:.2f}" + (f" (OOP/IP={rv:.1f})" if np.isfinite(rv) else "")
                ax_hf.text(max(fv, 0.0) + 0.015, yp, txt, va="center", fontsize=9.5, fontweight="bold")
        ax_hf.set_yticks(y_hf)
        ax_hf.set_yticklabels([r["label2"] for r in hf_sorted], fontsize=9.5, fontweight="bold")
        ax_hf.set_xlim(min(-0.2, float(np.nanmin(hf_vals)) - 0.1), min(1.15, max(0.45, float(np.nanmax(hf_vals)) + 0.34)))
        ax_hf.set_xlabel(r"Herman's Orientation Parameter $f$ ($0$=isotropic, $1$=out-of-plane)", fontsize=10.5)
        ax_hf.set_title(f"[{cond_short}] Out-of-Plane Texture Strength", fontsize=11.5, fontweight="bold")
        ax_hf.grid(True, axis="x", alpha=0.25)

        p4 = cond_dir / f"{cond_code}_04_azimuthal_orientation.png"
        _c7_savefig(fig, p4, dpi=300)
        display(fig)
        plt.close(fig)
        saved_paths.append(p4)

        # ======================================================================
        # PNG 5: POSITIVE-QUADRANT (qr >= 0, qz >= 0) 2D MAPS FOR THIS CONDITION
        # ======================================================================
        all_logs = []
        qr_maxs, qz_maxs = [], []
        pos_slices = {}
        for r in qrqz_valid_recs:
            qr_p, qz_p, arr_p = _c7_positive_qrqz_slice(r["qrqz_rec"])
            pos_slices[r["name"]] = (qr_p, qz_p, arr_p)
            disp = np.log10(np.maximum(arr_p, 0.0) + 1.0)
            fin = np.isfinite(disp)
            if np.any(fin):
                all_logs.append(disp[fin])
                cols_v = np.where(np.any(fin, axis=0))[0]
                rows_v = np.where(np.any(fin, axis=1))[0]
                qr_maxs.append(float(qr_p[cols_v[-1]]))
                qz_maxs.append(float(qz_p[rows_v[-1]]))

        cat_vals = np.concatenate(all_logs)
        vmin_shared = float(np.percentile(cat_vals, 2.0))
        vmax_shared = float(np.percentile(cat_vals, 99.4))
        xlim_shared = (0.0, max(qr_maxs) + 0.03)
        ylim_shared = (0.0, max(qz_maxs) + 0.03)

        cmap_2d = plt.get_cmap("viridis").copy()
        cmap_2d.set_bad(color="#eef1f5")

        n_maps = len(qrqz_valid_recs)
        ncols = min(3, n_maps)
        nrows = int(np.ceil(n_maps / ncols))
        fig, axes_grid = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 5.0 * nrows), constrained_layout=True)
        axes_flat = np.atleast_1d(axes_grid).ravel()

        for idx_m, r in enumerate(qrqz_valid_recs):
            qr_p, qz_p, arr_p = pos_slices[r["name"]]
            disp = np.log10(np.maximum(arr_p, 0.0) + 1.0)
            ax_m = axes_flat[idx_m]
            im = ax_m.pcolormesh(qr_p, qz_p, disp, shading="auto", cmap=cmap_2d, vmin=vmin_shared, vmax=vmax_shared)
            ax_m.set_aspect("equal", adjustable="box")
            ax_m.set_xlim(xlim_shared)
            ax_m.set_ylim(ylim_shared)
            ax_m.set_xlabel(r"$q_r$ ($\mathrm{\AA}^{-1}$)", fontsize=10)
            ax_m.set_ylabel(r"$q_z$ ($\mathrm{\AA}^{-1}$)", fontsize=10)
            ax_m.set_title(
                f"{r['label']}\n$t$={r['t_max']:.1f} s · $L_c$={r['Lc_nm']:.1f} nm · $f$={r['herman_f']:.2f}",
                fontsize=10, fontweight="bold"
            )

            fig_s, ax_s = plt.subplots(figsize=(6.0, 5.6), constrained_layout=True)
            im_s = ax_s.pcolormesh(qr_p, qz_p, disp, shading="auto", cmap=cmap_2d, vmin=vmin_shared, vmax=vmax_shared)
            ax_s.set_aspect("equal", adjustable="box")
            ax_s.set_xlim(xlim_shared)
            ax_s.set_ylim(ylim_shared)
            ax_s.set_xlabel(r"$q_r$ ($\mathrm{\AA}^{-1}$)", fontsize=11)
            ax_s.set_ylabel(r"$q_z$ ($\mathrm{\AA}^{-1}$)", fontsize=11)
            ax_s.set_title(f"{r['label']}  ($q_r$–$q_z$ at $t_{{\\max}}$)\n$t$={r['t_max']:.1f} s · $L_c$={r['Lc_nm']:.1f} nm", fontsize=11, fontweight="bold")
            cb_s = fig_s.colorbar(im_s, ax=ax_s, fraction=0.046, pad=0.03)
            cb_s.set_label(r"$\log_{10}(I + 1)$", fontsize=10)
            _c7_savefig(fig_s, qrqz_single_dir / f"{r['name']}_qrqz.png", dpi=300)
            plt.close(fig_s)

        for idx_empty in range(n_maps, len(axes_flat)):
            axes_flat[idx_empty].axis("off")

        cb_g = fig.colorbar(im, ax=axes_flat[:n_maps], fraction=0.025, pad=0.02)
        cb_g.set_label(r"$\log_{10}(I + 1)$ (Group-Shared Contrast)", fontsize=10.5)

        p5 = cond_dir / f"{cond_code}_05_qrqz_grid.png"
        _c7_savefig(fig, p5, dpi=300)
        display(fig)
        plt.close(fig)
        saved_paths.append(p5)

    # ==========================================================================
    # PNG 6: FULL-RANGE 1D WATERFALLS (PEAK COMPARISON + PER-SAMPLE EVOLUTION)
    # ==========================================================================
    n_samp = len(grp_records)
    if n_samp == 1:
        fig, (ax_pk, ax_evo) = plt.subplots(1, 2, figsize=(15.0, 5.6), constrained_layout=True)
        evo_axes = [ax_evo]
    else:
        n_evo_cols = min(2, n_samp)
        n_evo_rows = int(np.ceil(n_samp / n_evo_cols))
        fig = plt.figure(figsize=(15.5, max(5.8, 4.3 * n_evo_rows)), constrained_layout=True)
        gs6 = GridSpec(n_evo_rows, 1 + n_evo_cols, figure=fig, width_ratios=[1.15] + [1.0] * n_evo_cols)
        ax_pk = fig.add_subplot(gs6[:, 0])
        evo_axes = [fig.add_subplot(gs6[idx_s // n_evo_cols, 1 + (idx_s % n_evo_cols)]) for idx_s in range(n_samp)]

    # Left panel: normalized full-range 1D spectrum at t_max for each sample in condition
    q_mins_all, q_maxs_all = [], []
    offset_unit = 1.15
    for idx_s, r in enumerate(grp_records):
        q_arr = r["q_arr"]
        y_pk = r["I_mat"][r["f_max"]]
        valid_q = np.isfinite(q_arr) & np.isfinite(y_pk)
        qv = q_arr[valid_q]
        yv = y_pk[valid_q]
        q_mins_all.append(float(qv[0]))
        q_maxs_all.append(float(qv[-1]))

        y_base = float(np.nanpercentile(yv, 2.0))
        y_net = np.maximum(yv - y_base, 0.0)
        m100 = (qv >= Q_100_SEARCH_WINDOW[0]) & (qv <= Q_100_SEARCH_WINDOW[1])
        norm_scale = float(np.nanmax(y_net[m100])) if np.any(m100) else float(np.nanmax(y_net))
        y_norm = y_net / max(norm_scale, 1e-6)

        y_off = idx_s * offset_unit
        col = colors[r["name"]]
        ax_pk.plot(qv, y_norm + y_off, lw=1.7, color=col, ls=r["ls"])
        ax_pk.axhline(y_off, color="#cbd5e1", lw=0.7, ls="-", zorder=0)

    q_lo_g = min(q_mins_all)
    q_hi_g = max(q_maxs_all)
    y_top_pk = ((n_samp - 1) * offset_unit + 1.05) / 0.76     # top ~24 % reserved for reference-peak labels
    _c7_annotate_ref_peaks(ax_pk, q_lo_g, q_hi_g, y_top_pk * 0.99)
    ax_pk.set_yticks([i * offset_unit for i in range(n_samp)])
    ax_pk.set_yticklabels([f"{r['label']}\n$t_{{\\max}}$={r['t_max']:.1f} s" for r in grp_records], fontsize=8.8)
    for _tl, _r in zip(ax_pk.get_yticklabels(), grp_records):
        _tl.set_color(colors[_r["name"]])
        _tl.set_fontweight("bold")
    ax_pk.set_xlim(q_lo_g, q_hi_g)
    ax_pk.set_ylim(-0.08, y_top_pk)
    ax_pk.set_xlabel(r"Scattering Vector $q$ ($\mathrm{\AA}^{-1}$)", fontsize=10.5)
    ax_pk.set_ylabel(r"$I(q, t_{\max}) / I_{100}$ (offset per sample)", fontsize=10.5)
    ax_pk.set_title(f"[{cond_title}] Full-Range 1D Pattern at $t_{{\\max}}$", fontsize=11.5, fontweight="bold")
    ax_pk.grid(True, axis="x", alpha=0.22)

    # Right panel(s): per-sample time-evolution 1D waterfalls + standalone PNG exports
    for ax_e, r in zip(evo_axes, grp_records):
        t0_w, t1_w, cmap_w = _c7_plot_single_sample_waterfall(ax_e, r, n_traces=16)
        sm_w = ScalarMappable(norm=Normalize(vmin=t0_w, vmax=t1_w), cmap=cmap_w)
        sm_w.set_array([])
        cb_w = fig.colorbar(sm_w, ax=ax_e, fraction=0.036, pad=0.02)
        cb_w.set_label("Time (s)", fontsize=8.8)

        fig_w, ax_w = plt.subplots(figsize=(7.6, 5.4), constrained_layout=True)
        _c7_plot_single_sample_waterfall(ax_w, r, n_traces=18)
        cb_ws = fig_w.colorbar(sm_w, ax=ax_w, fraction=0.038, pad=0.02)
        cb_ws.set_label("Annealing Time (s)", fontsize=10)
        _c7_savefig(fig_w, wf_single_dir / f"{r['name']}_1d_waterfall.png", dpi=300)
        plt.close(fig_w)

    p6 = cond_dir / f"{cond_code}_06_1d_waterfall_full_range.png"
    _c7_savefig(fig, p6, dpi=300)
    display(fig)
    plt.close(fig)
    saved_paths.append(p6)

    p7 = _c7_dose_response(cond_code, cond_title, grp_records, cond_dir)
    if p7 is not None:
        saved_paths.append(p7)

    return saved_paths, len(qrqz_valid_recs), len(grp_records)


def generate_cell7_grouped_pngs():
    fwhm_inst = get_instrument()      # dict: H_G, H_L, H_tot, eta (LaB6 calibration)
    npz_files = [
        p for p in sorted(OUTPUT_DIR.rglob("*.npz"), key=lambda p: _c7_natural_key(p.stem))
        if not _c7_is_excluded(p)
    ]
    assert npz_files, f"No valid anneal .npz files found under {OUTPUT_DIR}."

    all_records = []
    for npz_p in npz_files:
        name = npz_p.stem.replace("_reduced", "").strip()
        if _c7_is_excluded(name):
            continue
        with np.load(npz_p, allow_pickle=False) as d:
            keys = set(d.files)
            q_key = next((k for k in ("q", "q_A_inv", "q_grid", "radial") if k in keys), None)
            I_key = next((k for k in ("I_matrix", "I_1d", "I_qt", "I_q", "I", "intensities", "intensity") if k in keys), None)
            t_key = next((k for k in ("time_s", "t_sec", "times", "time", "t") if k in keys), None)
            if q_key is None or I_key is None:
                continue
            q_arr = np.asarray(d[q_key], dtype=np.float64)
            I_mat = np.asarray(d[I_key], dtype=np.float64)
            if I_mat.ndim != 2 or I_mat.shape[0] < 10:
                continue
            t_arr = (
                np.asarray(d[t_key], dtype=np.float64)
                if t_key is not None
                else resolve_time_axis(name, I_mat.shape[0], np.float64)
            )

        t_cut = _CUTOFF_LOOKUP.get(name.lower(), None)
        f_cut = int(np.where(t_arr <= t_cut)[0][-1]) if (t_cut is not None and np.any(t_arr <= t_cut)) else (len(t_arr) - 1)
        f_max = int(np.clip(_c7_find_frame_max(q_arr, I_mat, t_arr=t_arr, t_cutoff=t_cut), 0, f_cut))

        area_100 = _c7_baseline_subtracted_area(q_arr, I_mat, Q_100_SEARCH_WINDOW)
        area_pbi2 = _c7_baseline_subtracted_area(q_arr, I_mat, Q_PBI2_WINDOW)
        area_100_sm = uniform_filter1d(area_100, size=min(3, len(area_100)), mode="nearest")

        t_grow = t_arr[: f_max + 1]
        a_grow = area_100_sm[: f_max + 1]
        a0 = float(np.min(a_grow[: max(2, min(4, len(a_grow)))]))
        amax = max(float(np.max(a_grow)), a0 + 1e-6)
        alpha_raw = np.clip((a_grow - a0) / (amax - a0), 0.0, 1.0)
        alpha_t = np.maximum.accumulate(alpha_raw)

        def _t_at_frac(frac):
            idx = np.where(alpha_t >= frac)[0]
            if len(idx) == 0:
                return float(t_grow[-1])
            i = int(idx[0])
            if i == 0:
                return float(t_grow[0])
            dv = max(alpha_t[i] - alpha_t[i - 1], 1e-9)
            return float(t_grow[i - 1] + (frac - alpha_t[i - 1]) / dv * (t_grow[i] - t_grow[i - 1]))

        t10, t50, t90 = _t_at_frac(0.10), _t_at_frac(0.50), _t_at_frac(0.90)
        dt_10_90 = max(t90 - t10, 0.0)
        r_max = float(np.nanmax(np.gradient(alpha_t, np.maximum(t_grow, 1e-6)))) if len(t_grow) > 2 else np.nan

        fit_pk = _c7_fit_100(q_arr, I_mat[f_max], Q_100_SEARCH_WINDOW, fwhm_inst, SCHERRER_K)
        if fit_pk is None:
            continue

        idxs = np.unique(np.linspace(0, f_max, min(36, f_max + 1), dtype=int))
        t_traj, alpha_traj, lc_traj = [], [], []
        for idx in idxs:
            if alpha_t[idx] < 0.08:
                continue
            ft = _c7_fit_100(q_arr, I_mat[idx], Q_100_SEARCH_WINDOW, fwhm_inst, SCHERRER_K)
            if ft is not None and np.isfinite(ft["Lc_nm"]) and ft["Lc_nm"] <= 1500.0:
                t_traj.append(float(t_arr[idx]))
                alpha_traj.append(float(alpha_t[idx]))
                lc_traj.append(float(ft["Lc_nm"]))

        pbi2_ratio_pct = 100.0 * float(area_pbi2[f_max]) / max(float(area_100[f_max]), 1e-6)

        qrqz_rec = _c7_ensure_qrqz(name, npz_p, f_max)
        chi_c, I_chi, herman_f, oop_ip = (None, None, np.nan, np.nan)
        if qrqz_rec is not None and qrqz_rec.get("I_qrqz") is not None:
            chi_c, I_chi, herman_f, oop_ip = _c7_extract_azimuthal(
                qrqz_rec["I_qrqz"], qrqz_rec["qr"], qrqz_rec["qz"], q0=fit_pk["q0"]
            )

        info = _c7_info(name)
        all_records.append({
            "name": name,
            "cond": info["group"],
            "known": info["known"], "level": info["level"], "numeric": info["numeric"], "rep": info["rep"],
            "level_label": info["label"],
            "label": (f"{info['label']} · {name}" if info["known"] else name),      # legends
            "label2": (f"{info['label']}\n{name}" if info["known"] else name),     # tick labels / bubble labels
            "q_arr": q_arr,
            "I_mat": I_mat,
            "t_arr": t_arr,
            "f_max": f_max,
            "t_max": float(t_arr[f_max]),
            "t10": t10,
            "t50": t50,
            "t90": t90,
            "dt_10_90": dt_10_90,
            "r_max": r_max,
            "area_100_max": float(area_100[f_max]),
            "t_full": t_arr,
            "area_full": area_100,
            "q0": fit_pk["q0"],
            "a_ang": fit_pk["a_ang"],
            "fwhm_corr": fit_pk["fwhm_corr"],
            "eta": fit_pk["eta"],
            "Lc_nm": fit_pk["Lc_nm"],
            "pbi2_ratio_pct": pbi2_ratio_pct,
            "t_grow": t_grow,
            "alpha_t": alpha_t,
            "t_traj": np.array(t_traj),
            "alpha_traj": np.array(alpha_traj),
            "lc_traj": np.array(lc_traj),
            "qrqz_rec": qrqz_rec,
            "chi_c": chi_c,
            "I_chi": I_chi,
            "herman_f": herman_f,
            "oop_ip": oop_ip,
        })

    assert all_records, "No valid fitted anneal samples available for Cell 7 export."
    # S1 (the reference film) is also the baseline level of the gap / substrate-T / superstrate-T experiments
    records_ext = add_baseline_copies(all_records, key="cond") if callable(globals().get("add_baseline_copies")) else all_records
    globals()["CELL7_RECORDS"] = records_ext      # consumed by Cells 7B / 7C (copies flagged baseline_copy=True)

    grouped = {}
    for rec in records_ext:
        grouped.setdefault(rec["cond"], []).append(rec)

    for _code, _recs in grouped.items():
        _assign_styles(_recs)

    ordered_codes = [c for c in CONDITION_ORDER if c in grouped] + sorted(c for c in grouped if c not in CONDITION_ORDER)

    # general (letter-series) overviews first, then the per-experiment breakdown
    _ov = globals().get("build_overview_groups")
    overview = _ov(all_records, key="cond") if callable(_ov) else {}
    for _fam, _recs in overview.items():
        grouped[_fam] = _recs
    ordered_codes = list(overview) + ordered_codes

    summary_lines = []
    for code in ordered_codes:
        grp_recs = grouped[code]
        saved_paths, n_2d, n_wf = _c7_export_condition_group(code, grp_recs)
        s_names = ", ".join(r["name"] for r in grp_recs)
        summary_lines.append(
            f"  • Condition {code} ({len(grp_recs)} samples: {s_names}) -> {len(saved_paths)} comparison PNGs + {n_2d} (qr,qz) maps + {n_wf} 1D waterfalls in exports_png_by_condition/condition_{code}/"
        )

    missing_codes = [c for c in CONDITION_ORDER if c not in grouped]
    print(f"✓ Condition-grouped PNG exports saved under:\n  {EXPORT_PNG_DIR}")
    for line in summary_lines:
        print(line)
    if missing_codes:
        print(f"  (Note: No anneal .npz scans found for experiment group: {', '.join(missing_codes)})")

    if DROPPED_NOT_IN_TABLE:
        print("Skipped (data file not on the sample table):", sorted(DROPPED_NOT_IN_TABLE))


generate_cell7_grouped_pngs()


# ======================================================================================
# Cell 7 data export  (the numbers behind the Cell 7 graphs; 7B and 7C write their own CSVs)
# ======================================================================================
def export_cell7_data(records=None):
    import numpy as _np, pandas as _pd
    recs = records if records is not None else globals().get("CELL7_RECORDS")
    assert recs, "Run Cell 7 first (CELL7_RECORDS is empty)."
    out_dir = Path(EXPORT_PNG_DIR) / "cell7_data"
    ts_dir = out_dir / "per_sample_timeseries"
    ts_dir.mkdir(parents=True, exist_ok=True)

    # 1) one row per sample: everything shown on the Cell 7 graphs
    rows = []
    for r in recs:
        rows.append({
            "sample": r["name"], "experiment": r["cond"], "level": r["level"], "level_label": r["level_label"],
            "replicate": r["rep"],
            "t10_s": r["t10"], "t50_s": r["t50"], "t90_s": r["t90"], "dt_10_90_s": r["dt_10_90"],
            "R_max_per_s": r["r_max"], "t_max_s": r["t_max"], "frame_t_max": r["f_max"],
            "A100_at_t_max": r["area_100_max"], "Lc_nm": r["Lc_nm"], "lattice_a_A": r["a_ang"],
            "q0_invA": r["q0"], "fwhm_corr_invA": r["fwhm_corr"], "eta": r["eta"],
            "PbI2_over_a100_pct": r["pbi2_ratio_pct"], "herman_f": r.get("herman_f", _np.nan),
            "n_frames": len(r["t_arr"]), "baseline_copy": bool(r.get("baseline_copy", False)),
        })
    df = _pd.DataFrame(rows).sort_values(["experiment", "level", "sample"])
    df.to_csv(out_dir / "cell7_sample_results.csv", index=False)

    # 2) per-sample time series: A100(t) on every frame; alpha(t) and Lc(t) on the fitted growth frames
    for r in [x for x in recs if not x.get("baseline_copy")]:
        cols = {"t_s": _np.asarray(r["t_full"], float), "A100_area": _np.asarray(r["area_full"], float)}
        d1 = _pd.DataFrame(cols)
        try:
            d2 = _pd.DataFrame({"t_traj_s": _np.asarray(r["t_traj"], float),
                                "alpha": _np.asarray(r["alpha_traj"], float),
                                "Lc_nm": _np.asarray(r["lc_traj"], float)})
        except Exception:
            d2 = _pd.DataFrame()
        _pd.concat([d1, d2], axis=1).to_csv(ts_dir / f"{r['name']}_timeseries.csv", index=False)

    # 3) 1D pattern at t_max, long format (sample, q, I)
    parts = []
    for r in [x for x in recs if not x.get("baseline_copy")]:
        q = _np.asarray(r["q_arr"], float); I = _np.asarray(r["I_mat"][r["f_max"]], float)
        parts.append(_pd.DataFrame({"sample": r["name"], "q_invA": q, "I_at_t_max": I}))
    _pd.concat(parts, ignore_index=True).to_csv(out_dir / "cell7_pattern_at_tmax.csv", index=False)

    # 4) azimuthal profile along alpha-(100), long format (sample, chi, I)
    parts = []
    for r in [x for x in recs if not x.get("baseline_copy")]:
        if r.get("chi_c") is not None and r.get("I_chi") is not None:
            parts.append(_pd.DataFrame({"sample": r["name"], "chi_deg": _np.asarray(r["chi_c"], float),
                                        "I_chi": _np.asarray(r["I_chi"], float)}))
    if parts:
        _pd.concat(parts, ignore_index=True).to_csv(out_dir / "cell7_azimuthal_profiles.csv", index=False)

    print(f"✓ Cell 7 data saved under: {out_dir}")
    print("   cell7_sample_results.csv · cell7_pattern_at_tmax.csv · cell7_azimuthal_profiles.csv · per_sample_timeseries/")
    return df


export_cell7_data()


## Cell 7B: Process (heating / velocity) vs. outcome figures
**Run order:** Cell 5C (heating + motion) → Cell 6B → Cell 7 → this cell. It needs `HEATING_DATA` / `HEATING_SUMMARY` from 5C and `CELL7_RECORDS` from Cell 7. Samples with no heating folder are shown in the outcome plots but left out of the process panels (and listed). Figures use the same experiment groups, colours and labels as Cell 7. Fig 8a/8b are drawn for the general series overview (S) first, then for each experiment.

**Kinetics view — anchored at $t_{10}$ / $t_{50}$ / $t_{90}$** (when the transition happens)
- **Fig 8a (per experiment):** $\alpha(t)$, substrate temperature $T(t)$, and $\alpha$ vs. $T$. Diamonds mark the temperature at $t_{50}$. If the $\alpha$-vs-$T$ curves collapse together, temperature explains the timing; if they stay apart, something else (composition, thickness, shear) sets the kinetics. $t_{50}$ is simply the most noise-robust point on the curve; $t_{10}$ is closer to nucleation onset and $t_{90}$ to the end of growth.
- **Fig 9a / 10a:** process variables (heating rate, plateau temp., temp. at $t_{10}$/$t_{50}$/$t_{90}$, cruise velocity, max acceleration) vs. $t_{10}$, $t_{50}$, $t_{90}$, $\Delta t_{10\rightarrow90}$, $R_{\max}$ — scatter grid / Spearman heatmap.

**Peak view — anchored at $t_{\max}$** (the strongest (100) frame)
- **Fig 8b (per experiment):** net $\alpha$-(100) area, $T(t)$, and $L_c$ vs. temperature; diamonds mark the temperature and domain size at $t_{\max}$.
- **Fig 9b / 10b:** process variables vs. $t_{\max}$, peak $A_{100}$, $L_c$, lattice $a$, $\eta$, PbI$_2$/$\alpha$, Herman $f$.

**Fig 11 — setpoint check (numeric experiments):** measured heating rate, plateau temperature and cruise velocity against the *nominal* level. The dashed line is "machine did what the table says" (plateau T vs. substrate setting; velocity vs. coating speed). Use it to catch a wrong setting or an offset before interpreting any trend.

Fig 9/10 pool every experiment, so they mix unlike variables; read them as a screen for which process quantities matter, and use Fig 7 (Cell 7) and Fig 8 for the within-experiment comparison. Everything plotted is saved as `process_vs_outcome_table.csv`.

**Read with care:** (1) *Temperature at an anchor time is partly an echo of that time* — the film heats as time passes, so a later $t_{50}$ or $t_{\max}$ automatically sits at a higher $T$. Heating rate and plateau temperature are the independent variables. (2) With few samples, treat ρ as a trend indicator; |ρ| below ~0.6 with n < 10 is indistinguishable from noise. (3) Thermocouples sit on the substrate holder, so the film lags the reading by about a second on fast ramps. (4) $t_{\max}$ depends on the beam-damage cutoffs in `SAMPLE_T_CUTOFFS`.


In [ ]:
# ==============================================================================
# CELL 7B: PROCESS (HEATING / VELOCITY) vs. OUTCOME FIGURES
# Run AFTER Cell 5C (HEATING_DATA, HEATING_SUMMARY) and Cell 7 (CELL7_RECORDS).
#   Fig 8a (per experiment): kinetics anchored at t50 - alpha(t), T(t), alpha vs T
#   Fig 8b (per experiment): state anchored at t_max  - A100(t), T(t), Lc vs T
#   Fig 9a/10a (all samples): process vs KINETICS outcomes (t10, t50, t90, dt, R_max)  [scatter / heatmap]
#   Fig 9b/10b (all samples): process vs PEAK outcomes (t_max, A100, Lc, a, eta, PbI2, f)
#   Fig 11 (per numeric experiment): measured heating rate / plateau T / velocity vs the nominal setting
# "Experiment" = the group of samples that change ONE variable (EXPERIMENT_GROUPS in Cell 1).
# Samples without heating data are skipped (and listed).  Saved to
# exports_png_by_condition/process_vs_outcome/ ; table saved as process_vs_outcome_table.csv
# ==============================================================================
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.stats import spearmanr

plt.rcParams.update({"legend.frameon": False})

_C7B_DIR = Path(globals().get("EXPORT_PNG_DIR", Path(globals().get("OUTPUT_DIR", ".")) / "exports_png_by_condition")) / "process_vs_outcome"
_C7B_DIR.mkdir(parents=True, exist_ok=True)
_C7B_SAVE = globals().get("_c7_savefig") or (lambda fig, path, **kw: fig.savefig(path, **kw))
_C7B_OFFSET = float(globals().get("ANNEAL_T0_OFFSET_S", 0.0))   # fallback only


def _c7b_off(h):
    """Seconds to add to GIWAXS time_s to get the heating clock (per sample, from the h5-UTC alignment in Cell 5C)."""
    try:
        v = float((h or {}).get("gi_off_s", np.nan))
    except (TypeError, ValueError):
        v = np.nan
    return v if np.isfinite(v) else _C7B_OFFSET

# (column in table, axis label).  Two views of the same samples:
#   KINETICS = timing of the crystallisation transition (anchored at t10 / t50 / t90)
#   PEAK     = film state at the strongest (100) frame t_max (anchored at t_max)
_C7B_PROC_COMMON = [
    ("rate", r"Heating rate (°C/s)"),
    ("T_plateau", r"Plateau temp. (°C)"),
]
_C7B_PROC_TAIL = [
    ("vel", r"Cruise velocity (m/min)"),
    ("acc", r"Max. accel. (m/min$^2$)"),
]
_C7B_VIEWS = {
    "kinetics": dict(
        title="Kinetics view: timing of crystallisation",
        process=_C7B_PROC_COMMON + [("T_t10", r"Temp. at $t_{10}$ (°C)"), ("T_t50", r"Temp. at $t_{50}$ (°C)"),
                                    ("T_t90", r"Temp. at $t_{90}$ (°C)")] + _C7B_PROC_TAIL,
        outcome=[("t10", r"$t_{10}$ (s)"), ("t50", r"$t_{50}$ (s)"), ("t90", r"$t_{90}$ (s)"),
                 ("dt", r"$\Delta t_{10\rightarrow90}$ (s)"), ("r_max", r"$R_{\max}$ (s$^{-1}$)")],
        tag="a",
    ),
    "peak": dict(
        title="Peak view: film state at the maximum (100) peak, $t_{\\max}$",
        process=_C7B_PROC_COMMON + [("T_tmax", r"Temp. at $t_{\max}$ (°C)")] + _C7B_PROC_TAIL,
        outcome=[("t_max", r"$t_{\max}$ (s)"), ("A100", r"Peak $A_{100}$ (a.u.)"), ("Lc", r"$L_c$ (nm)"),
                 ("a", r"Lattice $a$ (Å)"), ("eta", r"Voigt $\eta$"), ("pbi2", r"PbI$_2$/$\alpha$ (%)"),
                 ("herman", r"Herman $f$")],
        tag="b",
    ),
}


def _c7b_lookup(store, name):
    if name in store:
        return store[name]
    low = {str(k).lower(): v for k, v in store.items()}
    return low.get(str(name).lower())


def _c7b_temp_curve(hrec):
    """(t_since_anneal_start, smoothed T) for one heating record."""
    an = hrec.get("analysis") or {}
    tw, Ts = an.get("tw"), an.get("Ts")
    if tw is not None and Ts is not None and len(tw) > 3:
        tw, Ts = np.asarray(tw, float), np.asarray(Ts, float)
    else:
        tw, Ts = np.asarray(hrec["t"], float), np.asarray(hrec["T_mean"], float)
    ok = np.isfinite(tw) & np.isfinite(Ts)
    return tw[ok], Ts[ok]


def _c7b_build_table(recs, heat_data, heat_summary):
    rows, missing = [], []
    summ = heat_summary.set_index(heat_summary["sample_id"].astype(str).str.lower()) if len(heat_summary) else None
    for r in recs:
        h = _c7b_lookup(heat_data, r["name"])
        if h is None or summ is None or r["name"].lower() not in summ.index:
            missing.append(r["name"])
            h, s = None, {}
            tw, Ts = np.array([]), np.array([])
        else:
            s = summ.loc[r["name"].lower()]
            tw, Ts = _c7b_temp_curve(h)
        _o = _c7b_off(h)
        _T_at = lambda t, _o=_o: float(np.interp(t + _o, tw, Ts)) if (tw.size > 3 and np.isfinite(t)) else np.nan
        T_t10, T_t50, T_t90, T_tmax = _T_at(r["t10"]), _T_at(r["t50"]), _T_at(r["t90"]), _T_at(r["t_max"])
        vel = s.get("velocity_cruise_m_min", np.nan)
        if not np.isfinite(vel):
            vel = s.get("velocity_mean_m_min", np.nan)
        rows.append(dict(
            sample=r["name"], cond=r["cond"], level=r.get("level", np.nan), level_label=r.get("level_label", ""),
            rate=s.get("avg_rate_C_per_s", np.nan), T_plateau=s.get("T_plateau_C", np.nan),
            T_t10=T_t10, T_t50=T_t50, T_t90=T_t90, T_tmax=T_tmax,
            vel=vel, acc=s.get("accel_max_m_min2", np.nan),
            t10=r["t10"], t50=r["t50"], t90=r["t90"], dt=r["dt_10_90"], r_max=r["r_max"],
            t_max=r["t_max"], A100=r["area_100_max"], Lc=r["Lc_nm"], a=r["a_ang"], eta=r["eta"],
            pbi2=r["pbi2_ratio_pct"], herman=r["herman_f"], baseline_copy=bool(r.get("baseline_copy", False)),
        ))
    return pd.DataFrame(rows), missing


# ------------------------------------------------------------------------------
# FIG 7: temperature-aligned kinetics per condition
# ------------------------------------------------------------------------------
def _c7b_fig7(cond_code, recs, heat_data):
    recs = [r for r in recs if _c7b_lookup(heat_data, r["name"]) is not None]
    if not recs:
        return None
    title = globals().get("CONDITION_LABELS", {}).get(cond_code, f"Condition {cond_code}")
    cmap = plt.get_cmap("tab10")
    fig = plt.figure(figsize=(14.5, 7.4), constrained_layout=True)
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1.15, 1.0])
    ax_a = fig.add_subplot(gs[0, 0])
    ax_t = fig.add_subplot(gs[1, 0], sharex=ax_a)
    ax_x = fig.add_subplot(gs[:, 1])
    x_hi = 1.10 * max(float(r["t_max"]) for r in recs)

    for i, r in enumerate(recs):
        col, ls = r.get("color", cmap(i % 10)), r.get("ls", "-")
        h = _c7b_lookup(heat_data, r["name"])
        tw, Ts = _c7b_temp_curve(h)
        tg, al = r["t_grow"], r["alpha_t"]
        _o = _c7b_off(h)
        T_of_t = lambda t, _o=_o: np.interp(np.asarray(t, float) + _o, tw, Ts)
        have_T = tw.size > 3
        T50 = float(T_of_t(r["t50"])) if (have_T and np.isfinite(r["t50"])) else np.nan
        lab = f"{r.get('label', r['name'])} ($T_{{50}}$={T50:.0f} °C)" if np.isfinite(T50) else r.get("label", r["name"])

        ax_a.plot(tg, al, color=col, ls=ls, lw=2.0, label=lab)
        ax_a.scatter([r["t50"]], [0.5], color=col, s=42, zorder=5, edgecolors="k", linewidths=0.7)
        if have_T:
            m = (tw - _o) <= x_hi
            ax_t.plot((tw - _o)[m], Ts[m], color=col, ls=ls, lw=1.8)
            if np.isfinite(T50):
                ax_t.scatter([r["t50"]], [T50], color=col, s=55, marker="D", zorder=6, edgecolors="k", linewidths=0.8)
                ax_x.plot(T_of_t(tg), al, color=col, ls=ls, lw=2.0)
                ax_x.scatter([T50], [0.5], color=col, s=55, marker="D", zorder=6, edgecolors="k", linewidths=0.8)

    for lvl in (0.1, 0.5, 0.9):
        ax_a.axhline(lvl, color="#64748b", ls="--" if lvl == 0.5 else ":", lw=0.9, alpha=0.7)
        ax_x.axhline(lvl, color="#64748b", ls="--" if lvl == 0.5 else ":", lw=0.9, alpha=0.7)
    ax_a.set_xlim(0, x_hi)
    ax_a.set_ylim(-0.04, 1.06)
    ax_a.set_ylabel(r"Conversion $\alpha(t)$")
    ax_a.set_title(f"[{title}] Conversion $\\alpha(t)$ (dots = $t_{{50}}$)", fontsize=11.5, fontweight="bold")
    plt.setp(ax_a.get_xticklabels(), visible=False)
    ax_t.set_xlabel("Annealing time (s)")
    ax_t.set_ylabel("Substrate temp. (°C)")
    ax_t.set_title("Temperature (diamonds = temp. at $t_{50}$)", fontsize=11.5, fontweight="bold")
    ax_x.set_xlabel("Substrate temperature (°C)")
    ax_x.set_ylabel(r"Conversion $\alpha$")
    ax_x.set_ylim(-0.04, 1.06)
    ax_x.set_title(f"[{title}] Conversion vs. Temperature\n(curves that overlap = temperature explains timing)", fontsize=11.5, fontweight="bold")
    for ax in (ax_a, ax_t, ax_x):
        ax.grid(True, alpha=0.25)
    h_, l_ = ax_a.get_legend_handles_labels()
    try:
        fig.legend(h_, l_, loc="outside lower center", ncol=min(5, len(l_)), fontsize=9.2, frameon=False)
    except (ValueError, TypeError):
        ax_t.legend(h_, l_, loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=min(4, len(l_)), fontsize=9, frameon=False)
    p = _C7B_DIR / f"{cond_code}_08a_temperature_aligned_kinetics.png"
    _C7B_SAVE(fig, p, dpi=300)
    try:
        display(fig)
    except NameError:
        plt.show()
    plt.close(fig)
    return p


# ------------------------------------------------------------------------------
# FIG 7B: the same idea anchored at the maximum (100) peak t_max
#   left-top: net alpha-(100) area A100(t);  left-bottom: T(t);  right: L_c vs temperature
#   diamonds = t_max  (temperature at which the film reaches its strongest (100) signal)
# ------------------------------------------------------------------------------
def _c7b_fig7b(cond_code, recs, heat_data):
    recs = [r for r in recs if _c7b_lookup(heat_data, r["name"]) is not None]
    if not recs:
        return None
    title = globals().get("CONDITION_LABELS", {}).get(cond_code, f"Condition {cond_code}")
    cmap = plt.get_cmap("tab10")
    fig = plt.figure(figsize=(14.5, 7.4), constrained_layout=True)
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1.15, 1.0])
    ax_a = fig.add_subplot(gs[0, 0])
    ax_t = fig.add_subplot(gs[1, 0], sharex=ax_a)
    ax_x = fig.add_subplot(gs[:, 1])
    x_hi = 1.10 * max(float(r["t_max"]) for r in recs)

    for i, r in enumerate(recs):
        col, ls = r.get("color", cmap(i % 10)), r.get("ls", "-")
        _hr = _c7b_lookup(heat_data, r["name"])
        tw, Ts = _c7b_temp_curve(_hr)
        have_T = tw.size > 3
        _o = _c7b_off(_hr)
        T_of_t = lambda t, _o=_o: np.interp(np.asarray(t, float) + _o, tw, Ts)
        Tm = float(T_of_t(r["t_max"])) if have_T else np.nan
        lab = f"{r.get('label', r['name'])} ($T_{{\\max}}$={Tm:.0f} °C)" if np.isfinite(Tm) else r.get("label", r["name"])

        m = r["t_full"] <= x_hi
        ax_a.plot(r["t_full"][m], r["area_full"][m], color=col, ls=ls, lw=1.9, label=lab)
        ax_a.scatter([r["t_max"]], [r["area_100_max"]], color=col, s=48, zorder=5, edgecolors="k", linewidths=0.7)
        if have_T:
            mt = (tw - _o) <= x_hi
            ax_t.plot((tw - _o)[mt], Ts[mt], color=col, ls=ls, lw=1.8)
            if np.isfinite(Tm):
                ax_t.scatter([r["t_max"]], [Tm], color=col, s=55, marker="D", zorder=6, edgecolors="k", linewidths=0.8)
            if r["t_traj"].size > 1:
                ax_x.plot(T_of_t(r["t_traj"]), r["lc_traj"], color=col, ls=ls, lw=1.9)
                if np.isfinite(Tm) and np.any(np.isfinite(r["lc_traj"])):
                    ax_x.scatter([Tm], [r["Lc_nm"]], color=col, s=55, marker="D", zorder=6, edgecolors="k", linewidths=0.8)

    ax_a.set_xlim(0, x_hi)
    ax_a.set_ylabel(r"Net $\alpha$-(100) area $A_{100}$ (a.u.)")
    ax_a.set_title(f"[{title}] $\\alpha$-(100) Area (dots = $t_{{\\max}}$)", fontsize=11.5, fontweight="bold")
    plt.setp(ax_a.get_xticklabels(), visible=False)
    ax_t.set_xlabel("Annealing time (s)")
    ax_t.set_ylabel("Substrate temp. (°C)")
    ax_t.set_title("Temperature (diamonds = temp. at $t_{\\max}$)", fontsize=11.5, fontweight="bold")
    ax_x.set_xlabel("Substrate temperature (°C)")
    ax_x.set_ylabel(r"Coherence length $L_c$ (nm)")
    ax_x.set_title(f"[{title}] Domain Size vs. Temperature\n(diamonds = state at $t_{{\\max}}$)", fontsize=11.5, fontweight="bold")
    for ax in (ax_a, ax_t, ax_x):
        ax.grid(True, alpha=0.25)
    h_, l_ = ax_a.get_legend_handles_labels()
    try:
        fig.legend(h_, l_, loc="outside lower center", ncol=min(5, len(l_)), fontsize=9.2, frameon=False)
    except (ValueError, TypeError):
        ax_t.legend(h_, l_, loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=min(4, len(l_)), fontsize=9, frameon=False)
    p = _C7B_DIR / f"{cond_code}_08b_temperature_at_peak.png"
    _C7B_SAVE(fig, p, dpi=300)
    try:
        display(fig)
    except NameError:
        plt.show()
    plt.close(fig)
    return p


# ------------------------------------------------------------------------------
# FIG 8 / 9: process vs outcome (all samples, coloured by condition)
# ------------------------------------------------------------------------------
def _c7b_rho(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 4 or np.ptp(x[m]) == 0 or np.ptp(y[m]) == 0:
        return np.nan, int(m.sum())
    return float(spearmanr(x[m], y[m])[0]), int(m.sum())


def _c7b_fig89(df, view):
    V = _C7B_VIEWS[view]
    _OUT = V["outcome"]
    tag = V["tag"]
    proc = [(c, l) for c, l in V["process"] if c in df and np.isfinite(df[c]).sum() >= 4 and np.ptp(df[c].dropna()) > 0]
    dropped = [l for c, l in V["process"] if c not in [p[0] for p in proc]]
    if dropped:
        print("  (process variables with no usable spread, skipped: " + ", ".join(dropped) + ")")
    if not proc:
        print("  no process variable has enough spread for correlation panels.")
        return []
    conds = [c for c in globals().get("CONDITION_ORDER", []) if c in set(df["cond"])] + sorted(set(df["cond"]) - set(globals().get("CONDITION_ORDER", [])))
    ccol = {c: plt.get_cmap("tab10")(i % 10) for i, c in enumerate(conds)}
    nr, nc = len(_OUT), len(proc)
    fig, axes = plt.subplots(nr, nc, figsize=(3.3 * nc + 1.0, 2.7 * nr + 0.8), constrained_layout=True, squeeze=False)
    rho_mat = np.full((nr, nc), np.nan)
    n_mat = np.zeros((nr, nc), int)
    for i, (oc, ol) in enumerate(_OUT):
        for j, (pc, pl) in enumerate(proc):
            ax = axes[i, j]
            x, y = df[pc].to_numpy(float), df[oc].to_numpy(float)
            ax.scatter(x, y, c=[ccol[c] for c in df["cond"]], s=38, edgecolors="k", linewidths=0.6, zorder=3)
            rho, n = _c7b_rho(x, y)
            rho_mat[i, j], n_mat[i, j] = rho, n
            ax.set_title(f"ρ={rho:+.2f}  (n={n})" if np.isfinite(rho) else f"n={n} (too few)", fontsize=9)
            ax.margins(0.12)
            ax.grid(True, alpha=0.25)
            if i == nr - 1:
                ax.set_xlabel(pl, fontsize=9)
            if j == 0:
                ax.set_ylabel(ol, fontsize=9)
            ax.tick_params(labelsize=8)
    handles = [plt.Line2D([], [], marker="o", ls="", markerfacecolor=ccol[c], markeredgecolor="k", label=globals().get("EXPERIMENT_GROUPS", {}).get(c, {}).get("short", f"Condition {c}")) for c in conds]
    try:
        fig.legend(handles=handles, loc="outside lower center", ncol=min(8, len(handles)), fontsize=9.5, frameon=False)
    except (ValueError, TypeError):
        fig.legend(handles=handles, loc="lower center", ncol=min(8, len(handles)), fontsize=9.5, frameon=False)
    fig.suptitle(f"{V['title']}\nprocess variables vs. outcomes (one point per sample; ρ = Spearman rank correlation)", fontsize=12, fontweight="bold")
    p8 = _C7B_DIR / f"09{tag}_{view}_process_vs_outcome_scatter.png"
    _C7B_SAVE(fig, p8, dpi=300)
    try:
        display(fig)
    except NameError:
        plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(1.7 * nc + 2.6, 0.85 * nr + 2.2), constrained_layout=True)
    im = ax.imshow(rho_mat, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
    for i in range(nr):
        for j in range(nc):
            txt = f"{rho_mat[i, j]:+.2f}\nn={n_mat[i, j]}" if np.isfinite(rho_mat[i, j]) else f"–\nn={n_mat[i, j]}"
            ax.text(j, i, txt, ha="center", va="center", fontsize=9, color="k")
    ax.set_xticks(range(nc), [l for _, l in proc], rotation=25, ha="right", fontsize=9)
    ax.set_yticks(range(nr), [l for _, l in _OUT], fontsize=10)
    ax.set_title(f"{V['title']}\nSpearman ρ: process (columns) vs. outcome (rows)", fontsize=11.5, fontweight="bold")
    cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
    cb.set_label("Spearman ρ", fontsize=10)
    p9 = _C7B_DIR / f"10{tag}_{view}_correlation_heatmap.png"
    _C7B_SAVE(fig, p9, dpi=300)
    globals().setdefault("KEY_FIGURES", {})[f"spearman{tag}_{view}"] = fig
    try:
        display(fig)
    except NameError:
        plt.show()
    plt.close(fig)
    return [p8, p9]


# ------------------------------------------------------------------------------
# FIG 11: did the machine do what the table says?  measured process vs nominal level
# ------------------------------------------------------------------------------
def _c7b_fig11(df):
    saved = []
    egs = globals().get("EXPERIMENT_GROUPS", {})
    panels = [("rate", "Heating rate (°C/s)"), ("T_plateau", "Plateau temp. (°C)"), ("vel", "Cruise velocity (m/min)")]
    for code in [c for c in globals().get("CONDITION_ORDER", []) if c in set(df["cond"])]:
        g = egs.get(code)
        d = df[df["cond"] == code]
        if not g or not g["numeric"] or d["level"].nunique() < 2:
            continue
        cols = [(c, l) for c, l in panels if np.isfinite(d[c]).sum() >= 2]
        if not cols:
            continue
        fig, axes = plt.subplots(1, len(cols), figsize=(4.6 * len(cols) + 0.4, 4.2), constrained_layout=True, squeeze=False)
        lv = sorted(d["level"].unique())
        labs = d.drop_duplicates("level").set_index("level")["level_label"].to_dict()
        cmap_ = plt.get_cmap("viridis")
        for ax, (c, yl) in zip(axes[0], cols):
            colr = [cmap_(0.08 + 0.80 * lv.index(l) / max(len(lv) - 1, 1)) for l in d["level"]]
            ax.scatter(d["level"], d[c], c=colr, s=70, edgecolors="k", linewidths=0.8, zorder=3)
            if g["check"] == c:      # this measured quantity should equal the nominal setting
                lo, hi = min(lv), max(lv)
                ax.plot([lo, hi], [lo, hi], color="#64748b", ls="--", lw=1.1, zorder=1, label="nominal setting")
                ax.legend(loc="upper left", fontsize=9, frameon=False)
            ax.set_xticks(lv, [labs[l] for l in lv], fontsize=9)
            ax.margins(0.15)
            ax.set_xlabel(g["xlabel"], fontsize=10)
            ax.set_ylabel(yl, fontsize=10)
            ax.grid(True, alpha=0.25)
        fig.suptitle(f"[{g['title']}] measured process values vs. the nominal setting", fontsize=12, fontweight="bold")
        p = _C7B_DIR / f"{code}_11_setpoint_check.png"
        _C7B_SAVE(fig, p, dpi=300)
        try:
            display(fig)
        except NameError:
            plt.show()
        plt.close(fig)
        saved.append(p)
    return saved


def generate_cell7b():
    recs = globals().get("CELL7_RECORDS")
    heat = globals().get("HEATING_DATA")
    summ = globals().get("HEATING_SUMMARY")
    if not recs:
        raise RuntimeError("CELL7_RECORDS not found - run Cell 7 first.")
    if not heat or summ is None or len(summ) == 0:
        print("HEATING_DATA / HEATING_SUMMARY not found (run Cell 5C first) - process-dependent figures will be empty.")
        heat, summ = {}, pd.DataFrame(columns=["sample_id"])
    plt.ioff()
    saved = []
    by_cond = {}
    for r in recs:
        by_cond.setdefault(r["cond"], []).append(r)
    order = [c for c in globals().get("CONDITION_ORDER", []) if c in by_cond] + sorted(c for c in by_cond if c not in globals().get("CONDITION_ORDER", []))
    _ov = globals().get("build_overview_groups")       # general (letter-series) overviews first
    overview = _ov(recs, key="cond") if callable(_ov) else {}
    for _fam, _rs in overview.items():
        by_cond[_fam] = _rs
    order = list(overview) + order
    for c in order:
        for fn in (_c7b_fig7, _c7b_fig7b):
            p = fn(c, by_cond[c], heat)
            if p:
                saved.append(p)
    df, missing = _c7b_build_table(recs, heat, summ)
    if missing:
        print(f"No matching heating data for: {', '.join(missing)} (shown in outcome plots, left out of process panels)")
    df.to_csv(_C7B_DIR / "process_vs_outcome_table.csv", index=False)
    saved += _c7b_fig11(df)
    df_pool = df[~df["baseline_copy"]].reset_index(drop=True)      # pooled correlations: each sample counted once
    if df_pool[["rate", "vel"]].notna().any(axis=1).sum() >= 4:
        for view in ("kinetics", "peak"):
            saved += _c7b_fig89(df_pool, view)
    else:
        print("Fewer than 4 samples with heating/velocity data - skipping the pooled correlation figures.")
    print(f"✓ Cell 7B saved {len(saved)} figure(s) + table under: {_C7B_DIR}")
    return saved


generate_cell7b()


## Cell 7C: Are the differences real? — replicate noise, summary table, Avrami kinetics, Arrhenius
**Run after** Cell 7 (needs `CELL7_RECORDS`); the temperature parts also use Cell 5C. Outputs go to `exports_png_by_condition/differences/` together with three CSV tables.

- **Fig 12 — difference vs. replicate noise.** For each experiment and metric: (spread of the level means) ÷ (mean A/B replicate range). Below 1 = the "difference" is smaller than the scatter between identical samples; above ~2–4 = likely real. Experiments with no replicate pair (e.g. P, single samples) show "–". With only two replicates the noise estimate is itself rough, so use the colours as a guide.
- **Fig 13 — everything on one page.** Every level of every experiment as % change from a reference level (`REFERENCE_LEVEL_C7C`: Cs 0 %, 6 m/min, 150 °C, 10 min; otherwise the experiment mean), colour scaled within each metric. Actual values are in `level_summary_table.csv`.
- **Fig 14 — Avrami (JMAK) fit,** $\alpha(t)=1-\exp[-(kt)^n]$, per sample. $k$ is a rate constant (a better "speed" number than $t_{50}$); $n$ describes the shape of the transformation. **If conversion is nearly instantaneous compared with the frame spacing, $k$ and $n$ cannot be resolved** — those fits (fewer than 4 frames between 10 and 90 %) are drawn hollow and counted in a printed note. In that case rely on $t_{50}$, end-state metrics and Fig 12/13.
- **Fig 15 — Arrhenius** for the substrate-temperature experiment: $\ln k$ vs. $1000/T$, using (left) the measured temperature at $t_{50}$ and (right) the measured plateau temperature. The slope gives an apparent activation energy. *The film is still heating while it converts, so $k$ mixes ramp rate and intrinsic kinetics:* a negative slope on the left panel means conversion is set by the ramp rather than by isothermal kinetics, and the number should not be read as a true $E_a$. Only three temperatures are available, so treat any $E_a$ as a rough estimate.

In [ ]:
# ==============================================================================
# CELL 7C: ARE THE DIFFERENCES REAL?  replicate noise, normalised summary, Avrami kinetics, Arrhenius
# Run AFTER Cell 7 (CELL7_RECORDS) and, for the temperature parts, Cell 5C (HEATING_DATA / HEATING_SUMMARY).
#   Fig 12 : between-level difference / replicate noise, per experiment x metric          (which differences are real?)
#   Fig 13 : every level of every experiment as % change vs. a reference level             (everything on one page)
#   Fig 14 : Avrami (JMAK) fit of alpha(t) per sample -> rate constant k and exponent n   (per experiment)
#   Fig 15 : Arrhenius plot ln k vs 1/T (substrate-temperature experiment) -> apparent Ea
# Saved to exports_png_by_condition/differences/  (+ replicate_agreement.csv, level_summary_table.csv, avrami_fits.csv)
# ==============================================================================
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm, TwoSlopeNorm
from matplotlib.patches import Patch
from scipy.optimize import curve_fit

plt.rcParams.update({"legend.frameon": False})

# Reference level for the "% change vs. reference" table (Fig 13): 6 m/min for the speed series (the reference run),
# Cs-free for composition; substrate 150 °C and anneal time 10 min follow the P/F run conditions.  None -> compare with the mean of all levels in that experiment.
REFERENCE_LEVEL_C7C = {"Y_comp": 0.0, "S_speed": 6.0, "S_gap": 25.4, "S_substrate": 150.0, "S_superstrate": 50.0, "S_time": 10.0}
MIN_FRAMES_IN_TRANSITION = 4        # frames with 10 % < alpha < 90 % needed before k and n are considered resolved
GAS_CONSTANT_KJ = 8.314462618e-3    # kJ / (mol K)

_C7C_DIR = Path(globals().get("EXPORT_PNG_DIR", Path(globals().get("OUTPUT_DIR", ".")) / "exports_png_by_condition")) / "differences"
_C7C_DIR.mkdir(parents=True, exist_ok=True)
_C7C_SAVE = globals().get("_c7_savefig") or (lambda fig, path, **kw: fig.savefig(path, **kw))
_C7C_OFFSET = float(globals().get("ANNEAL_T0_OFFSET_S", 0.0))
_EG = globals().get("EXPERIMENT_GROUPS", {})

# (record/table key, short label, long label)
_C7C_METRICS = [
    ("Lc", r"$L_c$", r"$L_c$ (nm)"), ("t50", r"$t_{50}$", r"$t_{50}$ (s)"),
    ("k", r"$k$", r"Avrami $k$ (s$^{-1}$)"), ("n", r"$n$", r"Avrami $n$"),
    ("r_max", r"$R_{\max}$", r"$R_{\max}$ (s$^{-1}$)"), ("pbi2", r"PbI$_2$ %", r"PbI$_2$/$\alpha$ (%)"),
    ("herman", r"$f$", r"Herman $f$"), ("a", r"$a$", r"Lattice $a$ (Å)"), ("eta", r"$\eta$", r"Voigt $\eta$"),
]


def _c7c_show(fig):
    try:
        display(fig)
    except NameError:
        plt.show()


# ------------------------------------------------------------------------------
# AVRAMI (JMAK):  alpha(t) = 1 - exp(-(k t)^n)
# ------------------------------------------------------------------------------
def _avrami_model(t, k, n):
    return 1.0 - np.exp(-np.power(np.clip(k * t, 0.0, None), n))


def _c7c_fit_avrami(r):
    t = np.asarray(r["t_grow"], float)
    a = np.asarray(r["alpha_t"], float)
    ok = np.isfinite(t) & np.isfinite(a)
    t, a = t[ok], a[ok]
    out = dict(k=np.nan, k_se=np.nan, n=np.nan, n_se=np.nan, r2=np.nan, t_half=np.nan, n_win=0, dt=np.nan, resolved=False)
    if t.size < 5:
        return out
    tt = t - t[0]
    out["dt"] = float(np.median(np.diff(t)))
    out["n_win"] = int(np.sum((a > 0.10) & (a < 0.90)))
    t50 = max(float(r["t50"]) - t[0], out["dt"])
    try:
        n0 = 1.5
        popt, pcov = curve_fit(_avrami_model, tt, a, p0=[np.log(2) ** (1 / n0) / t50, n0],
                               bounds=([1e-3, 0.2], [100.0, 10.0]), maxfev=20000)
    except (RuntimeError, ValueError):
        return out
    se = np.sqrt(np.diag(pcov)) if np.all(np.isfinite(pcov)) else [np.nan, np.nan]
    res = a - _avrami_model(tt, *popt)
    out.update(k=float(popt[0]), n=float(popt[1]), k_se=float(se[0]), n_se=float(se[1]),
               r2=float(1 - np.sum(res ** 2) / max(np.sum((a - a.mean()) ** 2), 1e-12)),
               t_half=float(np.log(2) ** (1 / popt[1]) / popt[0]))
    out["resolved"] = out["n_win"] >= MIN_FRAMES_IN_TRANSITION
    return out


# ------------------------------------------------------------------------------
# TABLE
# ------------------------------------------------------------------------------
def _c7c_lookup(store, name):
    if not store:
        return None
    if name in store:
        return store[name]
    return {str(k).lower(): v for k, v in store.items()}.get(str(name).lower())


def _c7c_T_at(hrec, t):
    if hrec is None or not np.isfinite(t):
        return np.nan
    an = hrec.get("analysis") or {}
    tw, Ts = an.get("tw"), an.get("Ts")
    if tw is None or Ts is None or len(tw) < 4:
        tw, Ts = hrec.get("t"), hrec.get("T_mean")
    if tw is None:
        return np.nan
    tw, Ts = np.asarray(tw, float), np.asarray(Ts, float)
    m = np.isfinite(tw) & np.isfinite(Ts)
    return float(np.interp(t + _C7C_OFFSET, tw[m], Ts[m])) if m.sum() > 3 else np.nan


def _c7c_build(recs):
    heat = globals().get("HEATING_DATA") or {}
    summ = globals().get("HEATING_SUMMARY")
    srow = (summ.set_index(summ["sample_id"].astype(str).str.lower()) if (summ is not None and len(summ)) else None)
    rows, fits = [], {}
    for r in recs:
        if not r.get("known"):
            continue
        f = _c7c_fit_avrami(r)
        fits[r["name"]] = f
        h = _c7c_lookup(heat, r["name"])
        plateau = np.nan
        if srow is not None and r["name"].lower() in srow.index:
            plateau = float(srow.loc[r["name"].lower()].get("T_plateau_C", np.nan))
        rows.append(dict(
            group=r["cond"], sample=r["name"], level=r["level"], level_label=r["level_label"], rep=r["rep"],
            Lc=r["Lc_nm"], t50=r["t50"], k=f["k"], k_se=f["k_se"], n=f["n"], n_se=f["n_se"], r_max=r["r_max"],
            pbi2=r["pbi2_ratio_pct"], herman=r["herman_f"], a=r["a_ang"], eta=r["eta"],
            avrami_r2=f["r2"], frames_10_90=f["n_win"], frame_dt_s=f["dt"], avrami_resolved=f["resolved"],
            T_t50=_c7c_T_at(h, r["t50"]), T_plateau=plateau, baseline_copy=bool(r.get("baseline_copy", False)),
        ))
    return pd.DataFrame(rows), fits


# ------------------------------------------------------------------------------
# FIG 12: between-level difference vs replicate noise
# ------------------------------------------------------------------------------
def _c7c_replicate_table(df):
    rows = []
    for g, d in df.groupby("group", sort=False):
        levels = sorted(d["level"].unique())
        if len(levels) < 2:
            continue
        for key, lab, _ in _C7C_METRICS:
            means, ranges = [], []
            for lv in levels:
                v = d.loc[d["level"] == lv, key].to_numpy(float)
                v = v[np.isfinite(v)]
                if v.size:
                    means.append(v.mean())
                if v.size >= 2:
                    ranges.append(v.max() - v.min())
            between = (max(means) - min(means)) if len(means) >= 2 else np.nan
            noise = float(np.mean(ranges)) if ranges else np.nan
            ratio = (between / noise) if (np.isfinite(between) and np.isfinite(noise) and noise > 0) else np.nan
            rows.append(dict(group=g, metric=key, between_level_range=between, mean_replicate_range=noise,
                             ratio=ratio, n_levels=len(levels), n_levels_with_replicates=len(ranges)))
    return pd.DataFrame(rows)


def _c7c_fig12(rep):
    groups = [g for g in globals().get("GROUP_ORDER", []) if g in set(rep["group"])]
    if not groups:
        return None
    keys = [m[0] for m in _C7C_METRICS]
    M = np.full((len(groups), len(keys)), np.nan)
    for i, g in enumerate(groups):
        for j, k in enumerate(keys):
            v = rep[(rep["group"] == g) & (rep["metric"] == k)]["ratio"]
            if len(v):
                M[i, j] = v.iloc[0]
    cmap = ListedColormap(["#e5e7eb", "#fde68a", "#86efac", "#22c55e"])
    norm = BoundaryNorm([0, 1, 2, 4, 1e9], cmap.N)
    fig, ax = plt.subplots(figsize=(1.15 * len(keys) + 3.2, 0.62 * len(groups) + 2.6), constrained_layout=True)
    ax.imshow(np.where(np.isfinite(M), M, np.nan), cmap=cmap, norm=norm, aspect="auto")
    for i in range(len(groups)):
        for j in range(len(keys)):
            ax.text(j, i, "–" if not np.isfinite(M[i, j]) else (f"{M[i, j]:.1f}" if M[i, j] < 99 else ">99"),
                    ha="center", va="center", fontsize=10, color="#111827")
    ax.set_xticks(range(len(keys)), [m[1] for m in _C7C_METRICS], fontsize=11)
    ax.set_yticks(range(len(groups)), [_EG.get(g, {}).get("title", g) for g in groups], fontsize=10)
    ax.set_xticks(np.arange(-.5, len(keys)), minor=True)
    ax.set_yticks(np.arange(-.5, len(groups)), minor=True)
    ax.grid(which="minor", color="white", lw=2)
    ax.tick_params(which="minor", length=0)
    ax.set_title("Is the difference between levels bigger than the A/B replicate scatter?\n"
                 "(ratio = spread of level means ÷ mean replicate range;  “–” = no replicate pair to judge by)",
                 fontsize=11.5, fontweight="bold")
    handles = [Patch(facecolor=c, label=l) for c, l in zip(["#e5e7eb", "#fde68a", "#86efac", "#22c55e"],
                                                           ["< 1  within replicate noise", "1–2  marginal", "2–4  likely real", "> 4  clear"])]
    try:
        fig.legend(handles=handles, loc="outside lower center", ncol=4, fontsize=9.5, frameon=False)
    except (ValueError, TypeError):
        ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=4, fontsize=9, frameon=False)
    p = _C7C_DIR / "12_difference_vs_replicate_noise.png"
    _C7C_SAVE(fig, p, dpi=300)
    globals().setdefault("KEY_FIGURES", {})["difference_vs_replicate_noise"] = fig
    _c7c_show(fig)
    plt.close(fig)
    return p


# ------------------------------------------------------------------------------
# FIG 13: every level vs a reference, one page
# ------------------------------------------------------------------------------
def _c7c_fig13(df):
    keys = [m[0] for m in _C7C_METRICS]
    rows, labels, groups_ = [], [], []
    for g in [g for g in globals().get("GROUP_ORDER", []) if g in set(df["group"])]:
        d = df[df["group"] == g]
        levels = sorted(d["level"].unique())
        means = {lv: d[d["level"] == lv][keys].mean(numeric_only=True) for lv in levels}
        ref_lv = REFERENCE_LEVEL_C7C.get(g)
        ref = means[ref_lv] if ref_lv in means else pd.concat(list(means.values()), axis=1).mean(axis=1)
        for lv in levels:
            pct = (means[lv] - ref) / ref.abs().replace(0, np.nan) * 100.0
            rows.append((pct.to_numpy(float), means[lv], lv == ref_lv))
            nloc = int((d["level"] == lv).sum())
            labels.append(f"{_EG.get(g, {}).get('short', g)} · {d[d['level'] == lv]['level_label'].iloc[0]} (n={nloc})")
            groups_.append(g)
    if not rows:
        return None, None
    P = np.array([r[0] for r in rows])
    # table of actual means
    tab = pd.DataFrame([r[1] for r in rows], columns=keys)
    tab.insert(0, "row", labels)
    tab.insert(0, "group", groups_)
    tab.to_csv(_C7C_DIR / "level_summary_table.csv", index=False)

    colmax = np.nanmax(np.abs(P), axis=0)
    colmax[~np.isfinite(colmax) | (colmax == 0)] = 1.0
    Z = P / colmax                                           # colour = change relative to the largest change in that column
    fig, ax = plt.subplots(figsize=(1.15 * len(keys) + 4.5, 0.33 * len(labels) + 2.4), constrained_layout=True)
    im = ax.imshow(Z, cmap="coolwarm", norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1), aspect="auto")
    for i, (pct, _, is_ref) in enumerate(rows):
        for j in range(len(keys)):
            if is_ref:
                txt = "ref"
            elif np.isfinite(pct[j]):
                txt = "0%" if abs(pct[j]) < 0.05 else (f"{pct[j]:+.0f}%" if abs(pct[j]) >= 1 else f"{pct[j]:+.1f}%")
            else:
                txt = "–"
            ax.text(j, i, txt, ha="center", va="center", fontsize=8.6, fontweight=("bold" if is_ref else "normal"), color="#111827")
    ax.set_xticks(range(len(keys)), [m[1] for m in _C7C_METRICS], fontsize=11)
    ax.set_yticks(range(len(labels)), labels, fontsize=8.8)
    ax.xaxis.tick_top()
    bounds = [i for i in range(1, len(groups_)) if groups_[i] != groups_[i - 1]]
    for b in bounds:
        ax.axhline(b - 0.5, color="k", lw=1.4)
    ax.set_title("Change relative to the reference level of each experiment (colour scaled within each column)",
                 fontsize=11.5, fontweight="bold", pad=34)
    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cb.set_label("relative to largest change in column", fontsize=9.5)
    p = _C7C_DIR / "13_change_vs_reference_summary.png"
    _C7C_SAVE(fig, p, dpi=300)
    globals().setdefault("KEY_FIGURES", {})["change_vs_reference_summary"] = fig
    _c7c_show(fig)
    plt.close(fig)
    return p, tab


# ------------------------------------------------------------------------------
# FIG 14: Avrami fits per experiment
# ------------------------------------------------------------------------------
def _c7c_fig14(code, recs, fits):
    g = _EG.get(code, {})
    recs = [r for r in recs if r.get("known") and r["name"] in fits]
    if len(recs) < 2:
        return None
    lv = sorted({r["level"] for r in recs})
    lab = {r["level"]: r["level_label"] for r in recs}
    fig, (ax_a, ax_k, ax_n) = plt.subplots(1, 3, figsize=(16.0, 4.9), constrained_layout=True,
                                           gridspec_kw={"width_ratios": [1.5, 1.0, 1.0]})
    for r in recs:
        f = fits[r["name"]]
        t = np.asarray(r["t_grow"], float)
        ax_a.plot(t, r["alpha_t"], ls="none", marker=".", ms=4, color=r["color"], alpha=0.55)
        if np.isfinite(f["k"]):
            tt = np.linspace(0, t[-1] - t[0], 300)
            ax_a.plot(tt + t[0], _avrami_model(tt, f["k"], f["n"]), color=r["color"], ls=r["ls"], lw=1.7, label=r["label"])
    ax_a.set_xlabel("Annealing time (s)")
    ax_a.set_ylabel(r"Conversion $\alpha(t)$")
    ax_a.set_title("Data (dots) and Avrami fit (lines)", fontsize=11, fontweight="bold")
    ax_a.grid(True, alpha=0.25)
    ax_a.set_ylim(-0.04, 1.06)
    ax_a.legend(loc="lower right", fontsize=8, ncol=2)

    xs = {l: i for i, l in enumerate(lv)} if not g.get("numeric", False) else {l: l for l in lv}
    for ax, key, err, yl in ((ax_k, "k", "k_se", r"Rate constant $k$ (s$^{-1}$)"), (ax_n, "n", "n_se", r"Exponent $n$")):
        for r in recs:
            f = fits[r["name"]]
            if not np.isfinite(f[key]):
                continue
            ax.errorbar(xs[r["level"]], f[key], yerr=(f[err] if np.isfinite(f[err]) else None), fmt="o", ms=8,
                        color=r["color"], mec="k", mfc=(r["color"] if f["resolved"] else "white"), mew=1.2, ecolor="#64748b",
                        elinewidth=1.0, capsize=3, zorder=3)
        ax.set_xticks([xs[l] for l in lv], [lab[l] for l in lv], fontsize=9, rotation=(0 if g.get("numeric") else 20),
                      ha=("center" if g.get("numeric") else "right"))
        ax.margins(0.15)
        ax.set_ylabel(yl)
        ax.set_xlabel(g.get("xlabel", ""))
        ax.grid(True, alpha=0.25)
    ax_k.set_title("Rate constant (error bar = fit SE)", fontsize=11, fontweight="bold")
    ax_n.set_title("Exponent $n$ (error bar = fit SE)", fontsize=11, fontweight="bold")
    n_unres = sum(not fits[r["name"]]["resolved"] for r in recs)
    fig.suptitle(f"[{g.get('title', code)}] Avrami (JMAK) analysis of the conversion curve"
                 + (f"   —   hollow markers = fewer than {MIN_FRAMES_IN_TRANSITION} frames between 10 and 90 % (k, n poorly resolved)"
                    if n_unres else ""), fontsize=11.5, fontweight="bold")
    p = _C7C_DIR / f"{code}_14_avrami.png"
    _C7C_SAVE(fig, p, dpi=300)
    _c7c_show(fig)
    plt.close(fig)
    return p


# ------------------------------------------------------------------------------
# FIG 15: Arrhenius (experiments whose variable is a substrate temperature)
# ------------------------------------------------------------------------------
def _c7c_fig15(df):
    saved = []
    for code, g in _EG.items():
        if g.get("check") != "T_plateau":
            continue
        d = df[(df["group"] == code) & np.isfinite(df["k"]) & (df["k"] > 0)]
        if len(d) < 3 or d["level"].nunique() < 2:
            continue
        fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.9), constrained_layout=True)
        notes = []
        for ax, (col, xl) in zip(axes, [("T_t50", "temperature at $t_{50}$ (measured)"), ("T_plateau", "plateau temperature (measured)")]):
            T = d[col].to_numpy(float)
            src = "measured"
            if not np.isfinite(T).all():
                T = d["level"].to_numpy(float)          # fall back to nominal setting
                src = "nominal"
            x = 1000.0 / (T + 273.15)
            y = np.log(d["k"].to_numpy(float))
            ax.scatter(x, y, c=[c for c in [
                {r["name"]: r["color"] for r in (globals().get("CELL7_RECORDS") or [])}.get(s, "#64748b") for s in d["sample"]]],
                s=75, edgecolors="k", linewidths=0.9, zorder=3)
            ok = np.isfinite(x) & np.isfinite(y)
            title = f"1000/T from {xl}" + ("" if src == "measured" else " — NOT AVAILABLE, nominal used")
            if ok.sum() >= 3 and np.ptp(x[ok]) > 0:
                (slope, icpt), cov = np.polyfit(x[ok], y[ok], 1, cov=True)
                se = float(np.sqrt(cov[0, 0]))
                yhat = slope * x[ok] + icpt
                r2 = 1 - np.sum((y[ok] - yhat) ** 2) / max(np.sum((y[ok] - y[ok].mean()) ** 2), 1e-12)
                xx = np.linspace(x[ok].min(), x[ok].max(), 50)
                ax.plot(xx, slope * xx + icpt, color="#64748b", lw=1.4, ls="--", zorder=1)
                Ea, Ea_se = -slope * GAS_CONSTANT_KJ * 1000.0, se * GAS_CONSTANT_KJ * 1000.0
                title += f"\napparent $E_a$ = {Ea:.0f} ± {Ea_se:.0f} kJ/mol   ($R^2$ = {r2:.2f}, n = {ok.sum()})"
                if Ea < 0:
                    title += "\nnegative: conversion is set by the heating ramp, not by isothermal kinetics"
                notes.append(Ea)
            ax.set_xlabel("1000 / T  (K$^{-1}$)")
            ax.set_ylabel(r"$\ln k$  ($k$ in s$^{-1}$)")
            ax.set_title(title, fontsize=10.5, fontweight="bold")
            ax.grid(True, alpha=0.25)
            ax.margins(0.12)
        fig.suptitle(f"[{g['title']}] Arrhenius plot of the Avrami rate constant "
                     "(the film is still heating while it converts, so k mixes ramp rate and intrinsic kinetics)",
                     fontsize=11, fontweight="bold")
        p = _C7C_DIR / f"{code}_15_arrhenius.png"
        _C7C_SAVE(fig, p, dpi=300)
        _c7c_show(fig)
        plt.close(fig)
        saved.append(p)
    return saved


def generate_cell7c():
    recs = globals().get("CELL7_RECORDS")
    if not recs:
        raise RuntimeError("CELL7_RECORDS not found - run Cell 7 first.")
    plt.ioff()
    df, fits = _c7c_build(recs)
    if df.empty:
        print("No samples with experiment metadata (Cell 1 SAMPLE_META) - nothing to compare.")
        return []
    df.to_csv(_C7C_DIR / "avrami_fits.csv", index=False)
    saved = []

    rep = _c7c_replicate_table(df)
    rep.to_csv(_C7C_DIR / "replicate_agreement.csv", index=False)
    p = _c7c_fig12(rep)
    if p:
        saved.append(p)
    p, _ = _c7c_fig13(df)
    if p:
        saved.append(p)
    by = {}
    for r in recs:
        by.setdefault(r["cond"], []).append(r)
    for code in [c for c in globals().get("GROUP_ORDER", []) if c in by]:
        p = _c7c_fig14(code, by[code], fits)
        if p:
            saved.append(p)
    saved += _c7c_fig15(df)

    n_unres = int((~df["avrami_resolved"]).sum())
    if n_unres:
        print(f"Note: {n_unres} of {len(df)} samples have fewer than {MIN_FRAMES_IN_TRANSITION} frames between 10 and 90 % conversion "
              f"(median frame spacing {np.nanmedian(df['frame_dt_s']):.2f} s, median 10-90 % window "
              f"{np.nanmedian(df['frames_10_90']):.0f} frames): conversion is nearly instantaneous at this time resolution, so their "
              f"Avrami k and n are poorly determined (shown hollow). t50-based and end-state metrics are still valid.")
    print(f"✓ Cell 7C saved {len(saved)} figure(s) + 3 tables under: {_C7C_DIR}")
    return saved


generate_cell7c()


## Cell 7D: Temperature at the maximum (100) peak
Run after Cells 5C, 7 and 7B. Bar chart of the substrate temperature at each sample's `t_max` (plateau marked) and `t_max` vs `T(t_max)`.

In [ ]:
# ==============================================================================
# CELL 7D: TEMPERATURE AT THE MAXIMUM (100) PEAK, directly
# Run AFTER Cells 5C, 7 and 7B (needs CELL7_RECORDS, HEATING_DATA, HEATING_SUMMARY).
#   Figure 1: bar chart of the substrate temperature at t_max for every sample (setpoint / plateau marked)
#   Figure 2: t_max vs the temperature reached at t_max (shows whether the peak appears on the ramp or at the plateau)
# T(t_max) = thermocouple temperature (mean of the 3 holder thermocouples) interpolated at the sample's t_max,
# with the GIWAXS clock aligned to the heating clock through the UTC start time of the anneal .h5 (Cell 5C).  Error bars = +-half a frame in t_max.
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

_D_DIR = (globals().get("_C7B_DIR") or (OUTPUT_DIR / "exports_png_by_condition" / "cell7b_process")) / "temperature_at_peak"
_D_DIR.mkdir(parents=True, exist_ok=True)
_D_DPI = 300


def _d_save(fig, name):
    p = _D_DIR / name
    saver = globals().get("_c7_savefig") or (lambda f, path, **kw: f.savefig(path, **kw))
    saver(fig, p, dpi=_D_DPI, bbox_inches="tight", facecolor="white")
    return p


_recs_all = [r for r in globals().get("CELL7_RECORDS", []) if not r.get("baseline_copy", False)]
_heat, _summ = globals().get("HEATING_DATA"), globals().get("HEATING_SUMMARY")
if not _recs_all or not _heat or _summ is None or len(_summ) == 0:
    raise RuntimeError("Run Cells 5C, 7 and 7B first (CELL7_RECORDS / HEATING_DATA / HEATING_SUMMARY are needed).")

_df, _missing = _c7b_build_table(_recs_all, _heat, _summ)
_df = _df[np.isfinite(_df["T_tmax"])].copy()
if _missing:
    print("No heating data for (left out):", ", ".join(_missing))

# +-half-frame range of T at t_max
_by_name = {r["name"]: r for r in _recs_all}
_lo, _hi, _dtf = [], [], []
for _, row in _df.iterrows():
    r = _by_name[row["sample"]]
    h = _c7b_lookup(_heat, row["sample"])
    tw, Ts = _c7b_temp_curve(h)
    _off = _c7b_off(h)
    tg = np.asarray(r.get("t_grow", []), float)
    dtf = float(np.median(np.diff(tg))) if tg.size > 1 else 0.5
    _dtf.append(dtf)
    _lo.append(float(np.interp(row["t_max"] - 0.5 * dtf + _off, tw, Ts)))
    _hi.append(float(np.interp(row["t_max"] + 0.5 * dtf + _off, tw, Ts)))
_df["T_lo"], _df["T_hi"] = _lo, _hi
_df["frac_of_plateau_pct"] = 100.0 * _df["T_tmax"] / _df["T_plateau"]
_df["label"] = [_by_name[s].get("label", s) for s in _df["sample"]]
_df["color"] = [_by_name[s].get("color", "#4b5563") for s in _df["sample"]]

# order: experiment order, then as in the records
_cond_order = [c for c in globals().get("CONDITION_ORDER", []) if c in set(_df["cond"])] + \
              sorted(c for c in set(_df["cond"]) if c not in globals().get("CONDITION_ORDER", []))
_df["_o"] = [(_cond_order.index(c), i) for i, c in enumerate(_df["cond"])]
_df = _df.sort_values("_o").drop(columns="_o").reset_index(drop=True)
_df[["sample", "cond", "label", "t_max", "T_tmax", "T_lo", "T_hi", "T_plateau", "frac_of_plateau_pct", "rate"]].to_csv(
    _D_DIR / "temperature_at_tmax_table.csv", index=False)

# ---------------- Figure 1: bars ----------------
n = len(_df)
fig1, ax = plt.subplots(figsize=(10.8, max(4.0, 0.36 * n + 1.8)), constrained_layout=True)
ypos = np.arange(n)[::-1]
xerr = np.vstack([np.clip(_df["T_tmax"] - _df["T_lo"], 0, None), np.clip(_df["T_hi"] - _df["T_tmax"], 0, None)])
ax.barh(ypos, _df["T_tmax"], color=_df["color"], height=0.64, xerr=xerr,
        error_kw={"ecolor": "#111111", "elinewidth": 1.2, "capsize": 3})
ax.scatter(_df["T_plateau"], ypos, marker="|", s=260, color="#dc2626", zorder=5, label="plateau temperature")
xmax = float(np.nanmax(np.r_[_df["T_plateau"].values, _df["T_hi"].values])) * 1.04
for y, t, tm, fr in zip(ypos, _df["T_tmax"], _df["t_max"], _df["frac_of_plateau_pct"]):
    ax.text(xmax, y, f"{t:.0f} °C · t_max = {tm:.1f} s" + (f" · {fr:.0f}% of plateau" if np.isfinite(fr) else ""),
            va="center", ha="left", fontsize=8.2, fontweight="bold")      # labels sit in a column right of the plateau markers
ax.set_yticks(ypos)
ax.set_yticklabels([f"{l} · {s}" for l, s in zip(_df["label"], _df["sample"])], fontsize=9)
ax.set_xlim(0, xmax * 1.62)
ax.set_xlabel(r"Substrate temperature at $t_{\max}$ (°C, bars = ±½ frame in $t_{\max}$)", fontsize=11)
ax.set_title(r"Summary: Temperature at Max $\alpha$-(100) Peak ($T(t_{\max})$) Across All Samples", fontsize=12, fontweight="bold")
ax.grid(True, axis="x", alpha=0.25)
ax.legend(loc="upper right", frameon=False, fontsize=9)
_d_save(fig1, "Summary_temperature_at_tmax.png")
globals().setdefault("KEY_FIGURES", {})["Summary_temperature_at_tmax"] = fig1
plt.show()

# ---------------- Figure 2: t_max vs T(t_max) ----------------
fig2, ax2 = plt.subplots(figsize=(7.4, 5.6), constrained_layout=True)
for cond in _cond_order:
    d = _df[_df["cond"] == cond]
    if d.empty:
        continue
    ax2.errorbar(d["t_max"], d["T_tmax"], yerr=np.vstack([d["T_tmax"] - d["T_lo"], d["T_hi"] - d["T_tmax"]]).clip(0),
                 fmt="o", ms=7, mec="k", mew=0.6, color=d["color"].iloc[0], capsize=2.5, lw=1.0,
                 label=globals().get("CONDITION_LABELS", {}).get(cond, cond))
ax2.set_xlabel(r"Time to max $\alpha$-(100) peak, $t_{\max}$ (s)", fontsize=11)
ax2.set_ylabel(r"Substrate temperature at $t_{\max}$ (°C)", fontsize=11)
ax2.set_title(r"When the Max $\alpha$-(100) Peak Appears: $t_{\max}$ vs. $T(t_{\max})$", fontsize=12, fontweight="bold")
ax2.grid(True, alpha=0.25)
ax2.legend(fontsize=8, frameon=False, ncol=2)
_d_save(fig2, "tmax_vs_temperature_at_tmax.png")
globals().setdefault("KEY_FIGURES", {})["tmax_vs_temperature_at_tmax"] = fig2
plt.show()

print(f"✓ Cell 7D: {n} sample(s) -> {_D_DIR}")
print(_df[["sample", "t_max", "T_tmax", "T_plateau", "frac_of_plateau_pct"]].round(1).to_string(index=False))


## Cell 8: Export the key figures for presentation (high resolution)
Run after Cells 6, 7, 7B and 7C. Saves only the handful of figures worth showing, at `KEY_FIG_DPI` (PNG, plus vector PDF) into `outputs/key_figures/`.

In [ ]:
# ==============================================================================
# CELL 8: EXPORT THE KEY FIGURES (high resolution)
# ==============================================================================
KEY_FIG_DPI = 600                     # resolution of the PNGs
KEY_FIG_FORMATS = ("png", "pdf")      # "pdf" = vector, stays sharp at any size; remove it if you only want PNGs
KEY_FIG_PICK = (                      # a figure is exported if any of these appears in its name; None = export everything registered
    "Summary_time_to_peak_tmax",      #   how fast the (100) peak appears
    "nucleation_vs_coarsening",       #   conversion alpha(t), area, L_c(t) per group (the "Conversion alpha(t)" panel)
    "growth_window_vs_peak_rate",     #   kinetics: growth window vs peak rate (one per experiment group)
    "ranked_structural_metrics",      #   grouped comparison (one per experiment group)
    "spearman",                       #   Spearman correlations: process vs outcome
    "difference_vs_replicate_noise",  #   are differences bigger than replicate scatter?
    "change_vs_reference_summary",    #   change relative to the reference level
    "temperature_at_tmax",            #   substrate temperature at t_max (Cell 7D)
    "Summary_coherence_length_Lc",    #   caveat figure (resolution-limited L_c)
)

_figs = globals().get("KEY_FIGURES", {})
if not _figs:
    print("No figures registered yet: run Cells 6, 7, 7B and 7C in this session first, then rerun this cell.")
else:
    import re as _re
    _kdir = OUTPUT_DIR / "key_figures"
    _kdir.mkdir(parents=True, exist_ok=True)
    _saver = globals().get("_c7_savefig") or (lambda fig, path, **kw: fig.savefig(path, **kw))
    _n = 0
    for _name, _fig in _figs.items():
        if KEY_FIG_PICK is not None and not any(_k in _name for _k in KEY_FIG_PICK):
            continue
        _stem = _re.sub(r"[^A-Za-z0-9_.-]+", "_", _name).strip("_")
        for _ext in KEY_FIG_FORMATS:
            _p = _kdir / f"{_stem}.{_ext}"
            try:
                _saver(_fig, _p, dpi=KEY_FIG_DPI, bbox_inches="tight", facecolor="white")
            except Exception as _e:
                print(f"  ! could not save {_p.name}: {_e}")
        _n += 1
        print(f"  saved {_stem}  ({KEY_FIG_DPI} dpi {'/'.join(KEY_FIG_FORMATS)})")
    print(f"\n{_n} figure(s) -> {_kdir}")
